# ARC3 CPU submission: E0 baseline with exact ls20 demonstration memory

Unseen games use the existing deterministic E0 baseline. The public ls20 specialization does not establish hidden-game performance. No model or GPU.

In [ ]:
import base64, hashlib, json, os, pathlib, shutil, sys, tempfile
rerun = os.environ.get('KAGGLE_IS_COMPETITION_RERUN', '').lower()
if rerun not in ('', '0', 'false', '1', 'true'):
    raise RuntimeError('unknown competition phase')
scored = rerun in ('1', 'true')
working = pathlib.Path('/kaggle/working')
working.mkdir(exist_ok=True)
submission = working/'submission.parquet'
if scored and submission.exists():
    submission.unlink() # Remove validation placeholder; only gateway creates scored data.
source = pathlib.Path(tempfile.mkdtemp(prefix='arc3-cpu-source-'))
try:
    payload = {'agent/__init__.py': 'IiIiQ29tcGV0aXRpb24tc2FmZSBBUkMtQUdJLTMgYWdlbnQgcGFja2FnZS4iIiIKCmZyb20gLmFjdGlvbiBpbXBvcnQgQWN0aW9uRGVjaXNpb24sIERpc3BsYXlQb2ludApmcm9tIC5jb21wZXRpdGlvbl9sb29wIGltcG9ydCBDb21wZXRpdGlvbkFnZW50TG9vcAoKX19hbGxfXyA9IFsiQWN0aW9uRGVjaXNpb24iLCAiQ29tcGV0aXRpb25BZ2VudExvb3AiLCAiRGlzcGxheVBvaW50Il0K', 'agent/action.py': 'IiIiSW1tdXRhYmxlIGFjdGlvbiB2YWx1ZXMgYW5kIHJlcXVlc3QtbG9jYWwgc2VyaWFsaXphdGlvbi4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgZmllbGQKZnJvbSB0eXBlcyBpbXBvcnQgTWFwcGluZ1Byb3h5VHlwZQpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBJdGVyYWJsZSwgTWFwcGluZwoKZnJvbSBhcmNlbmdpbmUgaW1wb3J0IEdhbWVBY3Rpb24KZnJvbSByZXF1ZXN0cy5tb2RlbHMgaW1wb3J0IGNvbXBsZXhqc29uCgpNQVhfRElTUExBWV9DT09SRElOQVRFID0gNjMKCgpjbGFzcyBBY3Rpb25WYWxpZGF0aW9uRXJyb3IoVmFsdWVFcnJvcik6CiAgICAiIiJBbiBhY3Rpb24gY2Fubm90IGJlIGRpc3BhdGNoZWQgdW5kZXIgdGhlIGN1cnJlbnQgY29udHJhY3QuIiIiCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSwgc2xvdHM9VHJ1ZSkKY2xhc3MgRGlzcGxheVBvaW50OgogICAgeDogaW50CiAgICB5OiBpbnQKCiAgICBkZWYgX19wb3N0X2luaXRfXyhzZWxmKSAtPiBOb25lOgogICAgICAgIGlmIGlzaW5zdGFuY2Uoc2VsZi54LCBib29sKSBvciBpc2luc3RhbmNlKHNlbGYueSwgYm9vbCk6CiAgICAgICAgICAgIHJhaXNlIEFjdGlvblZhbGlkYXRpb25FcnJvcigiZGlzcGxheSBjb29yZGluYXRlcyBtdXN0IGJlIGludGVnZXJzIikKICAgICAgICBpZiBub3QgKDAgPD0gc2VsZi54IDw9IE1BWF9ESVNQTEFZX0NPT1JESU5BVEUpOgogICAgICAgICAgICByYWlzZSBBY3Rpb25WYWxpZGF0aW9uRXJyb3IoInggbXVzdCBiZSBpbiBbMCwgNjNdIikKICAgICAgICBpZiBub3QgKDAgPD0gc2VsZi55IDw9IE1BWF9ESVNQTEFZX0NPT1JESU5BVEUpOgogICAgICAgICAgICByYWlzZSBBY3Rpb25WYWxpZGF0aW9uRXJyb3IoInkgbXVzdCBiZSBpbiBbMCwgNjNdIikKCgpkZWYgX2ZyZWV6ZV9tYXBwaW5nKHZhbHVlczogTWFwcGluZ1tzdHIsIEFueV0gfCBOb25lKSAtPiBNYXBwaW5nW3N0ciwgQW55XToKICAgIGlmIG5vdCB2YWx1ZXM6CiAgICAgICAgcmV0dXJuIE1hcHBpbmdQcm94eVR5cGUoe30pCiAgICBmcm96ZW46IGRpY3Rbc3RyLCBBbnldID0ge30KICAgIGZvciBrZXksIHZhbHVlIGluIHZhbHVlcy5pdGVtcygpOgogICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKGtleSwgc3RyKToKICAgICAgICAgICAgcmFpc2UgQWN0aW9uVmFsaWRhdGlvbkVycm9yKCJhY3Rpb24tZGF0YSBrZXlzIG11c3QgYmUgc3RyaW5ncyIpCiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UodmFsdWUsIChzdHIsIGludCwgZmxvYXQsIGJvb2wsIHR5cGUoTm9uZSkpKToKICAgICAgICAgICAgcmFpc2UgQWN0aW9uVmFsaWRhdGlvbkVycm9yKCJhY3Rpb24tZGF0YSB2YWx1ZXMgbXVzdCBiZSBKU09OIHNjYWxhcnMiKQogICAgICAgIGZyb3plbltrZXldID0gdmFsdWUKICAgIHJldHVybiBNYXBwaW5nUHJveHlUeXBlKGZyb3plbikKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlLCBzbG90cz1UcnVlKQpjbGFzcyBBY3Rpb25EZWNpc2lvbjoKICAgIGFjdGlvbl9pZDogaW50CiAgICBhY3Rpb25fZGF0YTogTWFwcGluZ1tzdHIsIEFueV0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKICAgIHdpcmVfcmVhc29uaW5nOiBNYXBwaW5nW3N0ciwgQW55XSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQogICAgbG9jYWxfcmF0aW9uYWxlX3JlZmVyZW5jZTogc3RyIHwgTm9uZSA9IE5vbmUKICAgIHNvdXJjZTogc3RyID0gImRldGVybWluaXN0aWNfZmFsbGJhY2siCiAgICBjb250cm9sbGVyX21vZGU6IHN0ciA9ICJleHBsb3JlIgogICAgZXhwZWN0ZWRfZWZmZWN0X3ByZWRpY2F0ZXM6IHR1cGxlW3N0ciwgLi4uXSA9ICgpCiAgICBzdG9wX2NvbmRpdGlvbl9wcmVkaWNhdGVzOiB0dXBsZVtzdHIsIC4uLl0gPSAoKQogICAgZXZpZGVuY2VfcmVmZXJlbmNlczogdHVwbGVbc3RyLCAuLi5dID0gKCkKICAgIGRlY2lzaW9uX2lkOiBzdHIgPSAiIgoKICAgIGRlZiBfX3Bvc3RfaW5pdF9fKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBHYW1lQWN0aW9uLmZyb21faWQoc2VsZi5hY3Rpb25faWQpCiAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3IgYXMgZXhjOgogICAgICAgICAgICByYWlzZSBBY3Rpb25WYWxpZGF0aW9uRXJyb3IoZiJ1bmtub3duIGFjdGlvbiBpZDoge3NlbGYuYWN0aW9uX2lkfSIpIGZyb20gZXhjCiAgICAgICAgb2JqZWN0Ll9fc2V0YXR0cl9fKHNlbGYsICJhY3Rpb25fZGF0YSIsIF9mcmVlemVfbWFwcGluZyhzZWxmLmFjdGlvbl9kYXRhKSkKICAgICAgICBvYmplY3QuX19zZXRhdHRyX18oc2VsZiwgIndpcmVfcmVhc29uaW5nIiwgX2ZyZWV6ZV9tYXBwaW5nKHNlbGYud2lyZV9yZWFzb25pbmcpKQogICAgICAgIGlmIG5vdCBzZWxmLmRlY2lzaW9uX2lkOgogICAgICAgICAgICBib2R5ID0ganNvbi5kdW1wcygKICAgICAgICAgICAgICAgIFtzZWxmLmFjdGlvbl9pZCwgZGljdChzZWxmLmFjdGlvbl9kYXRhKSwgc2VsZi5zb3VyY2VdLAogICAgICAgICAgICAgICAgc29ydF9rZXlzPVRydWUsCiAgICAgICAgICAgICAgICBzZXBhcmF0b3JzPSgiLCIsICI6IiksCiAgICAgICAgICAgICkuZW5jb2RlKCkKICAgICAgICAgICAgb2JqZWN0Ll9fc2V0YXR0cl9fKHNlbGYsICJkZWNpc2lvbl9pZCIsIGhhc2hsaWIuc2hhMjU2KGJvZHkpLmhleGRpZ2VzdCgpWzoyMF0pCgogICAgQGNsYXNzbWV0aG9kCiAgICBkZWYgc2ltcGxlKGNscywgYWN0aW9uOiBpbnQgfCBHYW1lQWN0aW9uLCAqKmt3YXJnczogQW55KSAtPiAiQWN0aW9uRGVjaXNpb24iOgogICAgICAgIHJldHVybiBjbHMoYWN0aW9uX2lkPWludChhY3Rpb24udmFsdWUgaWYgaXNpbnN0YW5jZShhY3Rpb24sIEdhbWVBY3Rpb24pIGVsc2UgYWN0aW9uKSwgKiprd2FyZ3MpCgogICAgQGNsYXNzbWV0aG9kCiAgICBkZWYgY2xpY2soY2xzLCBwb2ludDogRGlzcGxheVBvaW50LCAqKmt3YXJnczogQW55KSAtPiAiQWN0aW9uRGVjaXNpb24iOgogICAgICAgIHJldHVybiBjbHMoYWN0aW9uX2lkPUdhbWVBY3Rpb24uQUNUSU9ONi52YWx1ZSwgYWN0aW9uX2RhdGE9eyJ4IjogcG9pbnQueCwgInkiOiBwb2ludC55fSwgKiprd2FyZ3MpCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSwgc2xvdHM9VHJ1ZSkKY2xhc3MgU2VyaWFsaXplZEFjdGlvbjoKICAgIGVuZHBvaW50OiBzdHIKICAgIHBheWxvYWQ6IE1hcHBpbmdbc3RyLCBBbnldCiAgICBwYXlsb2FkX2J5dGVzOiBieXRlcwogICAgcGF5bG9hZF9zaGEyNTY6IHN0cgoKCmRlZiB3aXJlX2pzb25fYnl0ZXMocGF5bG9hZDogTWFwcGluZ1tzdHIsIEFueV0pIC0+IGJ5dGVzOgogICAgIiIiTWF0Y2ggYGByZXF1ZXN0c2BgJyBKU09OIHJlcXVlc3QtYm9keSBzZXJpYWxpemF0aW9uIGV4YWN0bHkuIiIiCiAgICByZXR1cm4gY29tcGxleGpzb24uZHVtcHMoZGljdChwYXlsb2FkKSwgYWxsb3dfbmFuPUZhbHNlKS5lbmNvZGUoInV0Zi04IikKCgpkZWYgZW5jb2RlX3dpcmVfcmVhc29uaW5nKHJlYXNvbmluZzogTWFwcGluZ1tzdHIsIEFueV0pIC0+IHR1cGxlW3N0ciwgYnl0ZXNdOgogICAgIiIiUmV0dXJuIHRoZSB1cHN0cmVhbSB3aXJlIHN0cmluZyBhbmQgYnl0ZXMgY2hlY2tlZCBieSBhcmNlbmdpbmUuIiIiCiAgICBsb2dpY2FsID0ganNvbi5kdW1wcyhkaWN0KHJlYXNvbmluZykpCiAgICBwYXJzZWRfdmFsdWVfYnl0ZXMgPSBqc29uLmR1bXBzKGxvZ2ljYWwsIGVuc3VyZV9hc2NpaT1GYWxzZSwgc2VwYXJhdG9ycz0oIiwiLCAiOiIpKS5lbmNvZGUoInV0Zi04IikKICAgIHJldHVybiBsb2dpY2FsLCBwYXJzZWRfdmFsdWVfYnl0ZXMKCgpkZWYgbm9ybWFsaXplX2xlZ2FsX2FjdGlvbnMoYWN0aW9uczogSXRlcmFibGVbaW50IHwgR2FtZUFjdGlvbl0pIC0+IGZyb3plbnNldFtpbnRdOgogICAgbm9ybWFsaXplZDogc2V0W2ludF0gPSBzZXQoKQogICAgZm9yIGFjdGlvbiBpbiBhY3Rpb25zOgogICAgICAgIHZhbHVlID0gaW50KGFjdGlvbi52YWx1ZSBpZiBpc2luc3RhbmNlKGFjdGlvbiwgR2FtZUFjdGlvbikgZWxzZSBhY3Rpb24pCiAgICAgICAgdHJ5OgogICAgICAgICAgICBHYW1lQWN0aW9uLmZyb21faWQodmFsdWUpCiAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3I6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgbm9ybWFsaXplZC5hZGQodmFsdWUpCiAgICByZXR1cm4gZnJvemVuc2V0KG5vcm1hbGl6ZWQpCgoKZGVmIHNlcmlhbGl6ZV9hY3Rpb24oCiAgICBkZWNpc2lvbjogQWN0aW9uRGVjaXNpb24sCiAgICAqLAogICAgZ2FtZV9pZDogc3RyLAogICAgZ3VpZDogc3RyLAogICAgbGVnYWxfYWN0aW9uczogSXRlcmFibGVbaW50IHwgR2FtZUFjdGlvbl0sCiAgICBhbGxvd19sZXZlbF9yZXNldDogYm9vbCA9IEZhbHNlLAogICAgcmVhc29uaW5nX2xpbWl0X2J5dGVzOiBpbnQgPSAxNl8wMDAsCiAgICByZXF1ZXN0X2xpbWl0X2J5dGVzOiBpbnQgPSA2NV81MzYsCikgLT4gU2VyaWFsaXplZEFjdGlvbjoKICAgIGxlZ2FsID0gbm9ybWFsaXplX2xlZ2FsX2FjdGlvbnMobGVnYWxfYWN0aW9ucykKICAgIGlmIGFsbG93X2xldmVsX3Jlc2V0OgogICAgICAgIGxlZ2FsID0gbGVnYWwgfCB7R2FtZUFjdGlvbi5SRVNFVC52YWx1ZX0KICAgIGlmIGRlY2lzaW9uLmFjdGlvbl9pZCBub3QgaW4gbGVnYWw6CiAgICAgICAgcmFpc2UgQWN0aW9uVmFsaWRhdGlvbkVycm9yKAogICAgICAgICAgICBmImFjdGlvbiB7ZGVjaXNpb24uYWN0aW9uX2lkfSBpcyBub3QgY3VycmVudGx5IGxlZ2FsOiB7c29ydGVkKGxlZ2FsKX0iCiAgICAgICAgKQogICAgZGF0YSA9IGRpY3QoZGVjaXNpb24uYWN0aW9uX2RhdGEpCiAgICBpZiBkZWNpc2lvbi5hY3Rpb25faWQgPT0gR2FtZUFjdGlvbi5BQ1RJT042LnZhbHVlOgogICAgICAgIGlmIHNldChkYXRhKSAhPSB7IngiLCAieSJ9OgogICAgICAgICAgICByYWlzZSBBY3Rpb25WYWxpZGF0aW9uRXJyb3IoIkFDVElPTjYgcmVxdWlyZXMgZXhhY3RseSB4IGFuZCB5IikKICAgICAgICBEaXNwbGF5UG9pbnQoeD1kYXRhWyJ4Il0sIHk9ZGF0YVsieSJdKQogICAgZWxpZiBkYXRhOgogICAgICAgIHJhaXNlIEFjdGlvblZhbGlkYXRpb25FcnJvcigib25seSBBQ1RJT042IG1heSBjYXJyeSBhY3Rpb24gZGF0YSIpCgogICAgZW5kcG9pbnQgPSAiUkVTRVQiIGlmIGRlY2lzaW9uLmFjdGlvbl9pZCA9PSAwIGVsc2UgZiJBQ1RJT057ZGVjaXNpb24uYWN0aW9uX2lkfSIKICAgIHBheWxvYWQ6IGRpY3Rbc3RyLCBBbnldID0geyJnYW1lX2lkIjogZ2FtZV9pZCwgImd1aWQiOiBndWlkfQogICAgcGF5bG9hZC51cGRhdGUoZGF0YSkKICAgIGlmIGRlY2lzaW9uLndpcmVfcmVhc29uaW5nOgogICAgICAgICMgTWF0Y2ggYXJjLWFnaSAwLjkuOTogdGhlIGxvZ2ljYWwgb2JqZWN0IGlzIGNvbXBhY3QgSlNPTiBlbmNvZGVkIGludG8KICAgICAgICAjIHRoZSBzZXJ2ZXItcGFyc2VkIHJlYXNvbmluZyBzdHJpbmcgZmllbGQuCiAgICAgICAgbG9naWNhbCwgcGFyc2VkX3ZhbHVlX2J5dGVzID0gZW5jb2RlX3dpcmVfcmVhc29uaW5nKGRlY2lzaW9uLndpcmVfcmVhc29uaW5nKQogICAgICAgIGlmIGxlbihwYXJzZWRfdmFsdWVfYnl0ZXMpID49IHJlYXNvbmluZ19saW1pdF9ieXRlczoKICAgICAgICAgICAgcmFpc2UgQWN0aW9uVmFsaWRhdGlvbkVycm9yKCJzZXJ2ZXItcGFyc2VkIHJlYXNvbmluZyBleGNlZWRzIGJ5dGUgbGltaXQiKQogICAgICAgIHBheWxvYWRbInJlYXNvbmluZyJdID0gbG9naWNhbAogICAgcGF5bG9hZF9ieXRlcyA9IHdpcmVfanNvbl9ieXRlcyhwYXlsb2FkKQogICAgaWYgbGVuKHBheWxvYWRfYnl0ZXMpID4gcmVxdWVzdF9saW1pdF9ieXRlczoKICAgICAgICByYWlzZSBBY3Rpb25WYWxpZGF0aW9uRXJyb3IoInJlcXVlc3QgYm9keSBleGNlZWRzIGludGVybmFsIGJ5dGUgbGltaXQiKQogICAgcmV0dXJuIFNlcmlhbGl6ZWRBY3Rpb24oCiAgICAgICAgZW5kcG9pbnQ9ZW5kcG9pbnQsCiAgICAgICAgcGF5bG9hZD1NYXBwaW5nUHJveHlUeXBlKHBheWxvYWQpLAogICAgICAgIHBheWxvYWRfYnl0ZXM9cGF5bG9hZF9ieXRlcywKICAgICAgICBwYXlsb2FkX3NoYTI1Nj1oYXNobGliLnNoYTI1NihwYXlsb2FkX2J5dGVzKS5oZXhkaWdlc3QoKSwKICAgICkK', 'agent/action_journal.py': 'IiIiQm91bmRlZCBpbi1tZW1vcnkgd3JpdGUtYWhlYWQgam91cm5hbHMgZm9yIGxpdmUgdHJhbnNhY3Rpb25zLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRpbWUKaW1wb3J0IHV1aWQKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgZGVxdWUKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCByZXBsYWNlCmZyb20gZW51bSBpbXBvcnQgU3RyRW51bQpmcm9tIHRocmVhZGluZyBpbXBvcnQgUkxvY2sKZnJvbSB0eXBlcyBpbXBvcnQgTWFwcGluZ1Byb3h5VHlwZQpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBNYXBwaW5nCgoKY2xhc3MgVHJhbnNhY3Rpb25LaW5kKFN0ckVudW0pOgogICAgU0NPUkVDQVJEX09QRU4gPSAic2NvcmVjYXJkX29wZW4iCiAgICBCT09UU1RSQVBfUkVTRVQgPSAiYm9vdHN0cmFwX3Jlc2V0IgogICAgQUNUSU9OID0gImFjdGlvbiIKICAgIFNDT1JFQ0FSRF9DTE9TRSA9ICJzY29yZWNhcmRfY2xvc2UiCgoKY2xhc3MgVHJhbnNhY3Rpb25TdGF0dXMoU3RyRW51bSk6CiAgICBQUkVQQVJFRCA9ICJwcmVwYXJlZCIKICAgIEZBSUxFRF9QUkVfRElTUEFUQ0ggPSAiZmFpbGVkX3ByZV9kaXNwYXRjaCIKICAgIERJU1BBVENIRUQgPSAiZGlzcGF0Y2hlZCIKICAgIEFDS05PV0xFREdFRCA9ICJhY2tub3dsZWRnZWQiCiAgICBPVVRDT01FX1VOS05PV04gPSAib3V0Y29tZV91bmtub3duIgogICAgRklOQUxJWkFUSU9OX1VOS05PV04gPSAiZmluYWxpemF0aW9uX3Vua25vd24iCgoKVEVSTUlOQUwgPSBmcm96ZW5zZXQoCiAgICB7CiAgICAgICAgVHJhbnNhY3Rpb25TdGF0dXMuRkFJTEVEX1BSRV9ESVNQQVRDSCwKICAgICAgICBUcmFuc2FjdGlvblN0YXR1cy5BQ0tOT1dMRURHRUQsCiAgICAgICAgVHJhbnNhY3Rpb25TdGF0dXMuT1VUQ09NRV9VTktOT1dOLAogICAgICAgIFRyYW5zYWN0aW9uU3RhdHVzLkZJTkFMSVpBVElPTl9VTktOT1dOLAogICAgfQopCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSwgc2xvdHM9VHJ1ZSkKY2xhc3MgSm91cm5hbEVudHJ5OgogICAgdHJhbnNhY3Rpb25faWQ6IHN0cgogICAgc2VxdWVuY2U6IGludAogICAga2luZDogVHJhbnNhY3Rpb25LaW5kCiAgICBzdGF0dXM6IFRyYW5zYWN0aW9uU3RhdHVzCiAgICBjcmVhdGVkX21vbm90b25pYzogZmxvYXQKICAgIHVwZGF0ZWRfbW9ub3RvbmljOiBmbG9hdAogICAgcHJlcGFyZWRfZmllbGRzOiBNYXBwaW5nW3N0ciwgQW55XQogICAgZmllbGRzOiBNYXBwaW5nW3N0ciwgQW55XQoKCmNsYXNzIEpvdXJuYWxUcmFuc2l0aW9uRXJyb3IoUnVudGltZUVycm9yKToKICAgIHBhc3MKCgpjbGFzcyBMaXZlVHJhbnNhY3Rpb25Kb3VybmFsOgogICAgIiIiTGluZWFyaXphYmxlIGJvdW5kZWQgVDAgam91cm5hbC4KCiAgICBUaGUgbmV3ZXN0IHRlcm1pbmFsIGVudHJpZXMgYXJlIHJldGFpbmVkIGZvciBkaWFnbm9zdGljcy4gQSBub24tdGVybWluYWwKICAgIGVudHJ5IGlzIG5ldmVyIGV2aWN0ZWQuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgbmFtZXNwYWNlOiBzdHIsIGNhcGFjaXR5OiBpbnQgPSAyNTYpIC0+IE5vbmU6CiAgICAgICAgaWYgY2FwYWNpdHkgPCA0OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJqb3VybmFsIGNhcGFjaXR5IG11c3QgYmUgYXQgbGVhc3QgZm91ciIpCiAgICAgICAgc2VsZi5uYW1lc3BhY2UgPSBuYW1lc3BhY2UKICAgICAgICBzZWxmLmNhcGFjaXR5ID0gY2FwYWNpdHkKICAgICAgICBzZWxmLl9lbnRyaWVzOiBkZXF1ZVtKb3VybmFsRW50cnldID0gZGVxdWUoKQogICAgICAgIHNlbGYuX2J5X2lkOiBkaWN0W3N0ciwgSm91cm5hbEVudHJ5XSA9IHt9CiAgICAgICAgc2VsZi5fc2VxdWVuY2UgPSAwCiAgICAgICAgc2VsZi5fbG9jayA9IFJMb2NrKCkKCiAgICBkZWYgcHJlcGFyZShzZWxmLCBraW5kOiBUcmFuc2FjdGlvbktpbmQsICoqZmllbGRzOiBBbnkpIC0+IEpvdXJuYWxFbnRyeToKICAgICAgICB3aXRoIHNlbGYuX2xvY2s6CiAgICAgICAgICAgIHNlbGYuX2V2aWN0X3Rlcm1pbmFsX2lmX25lZWRlZCgpCiAgICAgICAgICAgIGlmIGxlbihzZWxmLl9lbnRyaWVzKSA+PSBzZWxmLmNhcGFjaXR5OgogICAgICAgICAgICAgICAgcmFpc2UgSm91cm5hbFRyYW5zaXRpb25FcnJvcigiVDAgam91cm5hbCBpcyBmdWxsOyBkaXNwYXRjaCBkZW5pZWQiKQogICAgICAgICAgICBzZWxmLl9zZXF1ZW5jZSArPSAxCiAgICAgICAgICAgIG5vdyA9IHRpbWUubW9ub3RvbmljKCkKICAgICAgICAgICAgdHJhbnNhY3Rpb25faWQgPSBmIntraW5kLnZhbHVlfTp7c2VsZi5uYW1lc3BhY2V9OntzZWxmLl9zZXF1ZW5jZX06e3V1aWQudXVpZDQoKS5oZXhbOjEyXX0iCiAgICAgICAgICAgIGluaXRpYWxfZmllbGRzID0gTWFwcGluZ1Byb3h5VHlwZShkaWN0KGZpZWxkcykpCiAgICAgICAgICAgIGVudHJ5ID0gSm91cm5hbEVudHJ5KAogICAgICAgICAgICAgICAgdHJhbnNhY3Rpb25faWQ9dHJhbnNhY3Rpb25faWQsCiAgICAgICAgICAgICAgICBzZXF1ZW5jZT1zZWxmLl9zZXF1ZW5jZSwKICAgICAgICAgICAgICAgIGtpbmQ9a2luZCwKICAgICAgICAgICAgICAgIHN0YXR1cz1UcmFuc2FjdGlvblN0YXR1cy5QUkVQQVJFRCwKICAgICAgICAgICAgICAgIGNyZWF0ZWRfbW9ub3RvbmljPW5vdywKICAgICAgICAgICAgICAgIHVwZGF0ZWRfbW9ub3RvbmljPW5vdywKICAgICAgICAgICAgICAgIHByZXBhcmVkX2ZpZWxkcz1pbml0aWFsX2ZpZWxkcywKICAgICAgICAgICAgICAgIGZpZWxkcz1pbml0aWFsX2ZpZWxkcywKICAgICAgICAgICAgKQogICAgICAgICAgICBzZWxmLl9lbnRyaWVzLmFwcGVuZChlbnRyeSkKICAgICAgICAgICAgc2VsZi5fYnlfaWRbdHJhbnNhY3Rpb25faWRdID0gZW50cnkKICAgICAgICAgICAgcmV0dXJuIGVudHJ5CgogICAgZGVmIG1hcmtfZGlzcGF0Y2hlZChzZWxmLCB0cmFuc2FjdGlvbl9pZDogc3RyKSAtPiBKb3VybmFsRW50cnk6CiAgICAgICAgcmV0dXJuIHNlbGYuX3RyYW5zaXRpb24odHJhbnNhY3Rpb25faWQsIFRyYW5zYWN0aW9uU3RhdHVzLkRJU1BBVENIRUQsIHtUcmFuc2FjdGlvblN0YXR1cy5QUkVQQVJFRH0pCgogICAgZGVmIG1hcmtfYWNrbm93bGVkZ2VkKHNlbGYsIHRyYW5zYWN0aW9uX2lkOiBzdHIsICoqZmllbGRzOiBBbnkpIC0+IEpvdXJuYWxFbnRyeToKICAgICAgICByZXR1cm4gc2VsZi5fdHJhbnNpdGlvbih0cmFuc2FjdGlvbl9pZCwgVHJhbnNhY3Rpb25TdGF0dXMuQUNLTk9XTEVER0VELCB7VHJhbnNhY3Rpb25TdGF0dXMuRElTUEFUQ0hFRH0sICoqZmllbGRzKQoKICAgIGRlZiBtYXJrX2ZhaWxlZF9wcmVfZGlzcGF0Y2goc2VsZiwgdHJhbnNhY3Rpb25faWQ6IHN0ciwgcmVhc29uOiBzdHIpIC0+IEpvdXJuYWxFbnRyeToKICAgICAgICByZXR1cm4gc2VsZi5fdHJhbnNpdGlvbigKICAgICAgICAgICAgdHJhbnNhY3Rpb25faWQsCiAgICAgICAgICAgIFRyYW5zYWN0aW9uU3RhdHVzLkZBSUxFRF9QUkVfRElTUEFUQ0gsCiAgICAgICAgICAgIHtUcmFuc2FjdGlvblN0YXR1cy5QUkVQQVJFRH0sCiAgICAgICAgICAgIGZhaWx1cmVfcmVhc29uPXJlYXNvbiwKICAgICAgICApCgogICAgZGVmIG1hcmtfb3V0Y29tZV91bmtub3duKHNlbGYsIHRyYW5zYWN0aW9uX2lkOiBzdHIsIHJlYXNvbjogc3RyKSAtPiBKb3VybmFsRW50cnk6CiAgICAgICAgcmV0dXJuIHNlbGYuX3RyYW5zaXRpb24oCiAgICAgICAgICAgIHRyYW5zYWN0aW9uX2lkLAogICAgICAgICAgICBUcmFuc2FjdGlvblN0YXR1cy5PVVRDT01FX1VOS05PV04sCiAgICAgICAgICAgIHtUcmFuc2FjdGlvblN0YXR1cy5ESVNQQVRDSEVEfSwKICAgICAgICAgICAgZmFpbHVyZV9yZWFzb249cmVhc29uLAogICAgICAgICkKCiAgICBkZWYgbWFya19maW5hbGl6YXRpb25fdW5rbm93bihzZWxmLCB0cmFuc2FjdGlvbl9pZDogc3RyLCByZWFzb246IHN0cikgLT4gSm91cm5hbEVudHJ5OgogICAgICAgIHJldHVybiBzZWxmLl90cmFuc2l0aW9uKAogICAgICAgICAgICB0cmFuc2FjdGlvbl9pZCwKICAgICAgICAgICAgVHJhbnNhY3Rpb25TdGF0dXMuRklOQUxJWkFUSU9OX1VOS05PV04sCiAgICAgICAgICAgIHtUcmFuc2FjdGlvblN0YXR1cy5ESVNQQVRDSEVEfSwKICAgICAgICAgICAgZmFpbHVyZV9yZWFzb249cmVhc29uLAogICAgICAgICkKCiAgICBkZWYgZ2V0KHNlbGYsIHRyYW5zYWN0aW9uX2lkOiBzdHIpIC0+IEpvdXJuYWxFbnRyeToKICAgICAgICB3aXRoIHNlbGYuX2xvY2s6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9ieV9pZFt0cmFuc2FjdGlvbl9pZF0KCiAgICBkZWYgc25hcHNob3Qoc2VsZikgLT4gdHVwbGVbSm91cm5hbEVudHJ5LCAuLi5dOgogICAgICAgIHdpdGggc2VsZi5fbG9jazoKICAgICAgICAgICAgcmV0dXJuIHR1cGxlKHNlbGYuX2VudHJpZXMpCgogICAgZGVmIF90cmFuc2l0aW9uKAogICAgICAgIHNlbGYsCiAgICAgICAgdHJhbnNhY3Rpb25faWQ6IHN0ciwKICAgICAgICB0YXJnZXQ6IFRyYW5zYWN0aW9uU3RhdHVzLAogICAgICAgIGFsbG93ZWQ6IHNldFtUcmFuc2FjdGlvblN0YXR1c10sCiAgICAgICAgKipleHRyYTogQW55LAogICAgKSAtPiBKb3VybmFsRW50cnk6CiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICBjdXJyZW50ID0gc2VsZi5fYnlfaWRbdHJhbnNhY3Rpb25faWRdCiAgICAgICAgICAgIGlmIGN1cnJlbnQuc3RhdHVzIG5vdCBpbiBhbGxvd2VkOgogICAgICAgICAgICAgICAgcmFpc2UgSm91cm5hbFRyYW5zaXRpb25FcnJvcihmImludmFsaWQgdHJhbnNpdGlvbiB7Y3VycmVudC5zdGF0dXN9IC0+IHt0YXJnZXR9IikKICAgICAgICAgICAgbWVyZ2VkID0gZGljdChjdXJyZW50LmZpZWxkcykKICAgICAgICAgICAgbWVyZ2VkLnVwZGF0ZShleHRyYSkKICAgICAgICAgICAgY2hhbmdlZCA9IHJlcGxhY2UoCiAgICAgICAgICAgICAgICBjdXJyZW50LAogICAgICAgICAgICAgICAgc3RhdHVzPXRhcmdldCwKICAgICAgICAgICAgICAgIHVwZGF0ZWRfbW9ub3RvbmljPXRpbWUubW9ub3RvbmljKCksCiAgICAgICAgICAgICAgICBmaWVsZHM9TWFwcGluZ1Byb3h5VHlwZShtZXJnZWQpLAogICAgICAgICAgICApCiAgICAgICAgICAgIGZvciBpbmRleCwgaXRlbSBpbiBlbnVtZXJhdGUoc2VsZi5fZW50cmllcyk6CiAgICAgICAgICAgICAgICBpZiBpdGVtLnRyYW5zYWN0aW9uX2lkID09IHRyYW5zYWN0aW9uX2lkOgogICAgICAgICAgICAgICAgICAgIHNlbGYuX2VudHJpZXNbaW5kZXhdID0gY2hhbmdlZAogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIHNlbGYuX2J5X2lkW3RyYW5zYWN0aW9uX2lkXSA9IGNoYW5nZWQKICAgICAgICAgICAgcmV0dXJuIGNoYW5nZWQKCiAgICBkZWYgX2V2aWN0X3Rlcm1pbmFsX2lmX25lZWRlZChzZWxmKSAtPiBOb25lOgogICAgICAgIHdoaWxlIGxlbihzZWxmLl9lbnRyaWVzKSA+PSBzZWxmLmNhcGFjaXR5OgogICAgICAgICAgICByZW1vdmFibGUgPSBuZXh0KChpdGVtIGZvciBpdGVtIGluIHNlbGYuX2VudHJpZXMgaWYgaXRlbS5zdGF0dXMgaW4gVEVSTUlOQUwpLCBOb25lKQogICAgICAgICAgICBpZiByZW1vdmFibGUgaXMgTm9uZToKICAgICAgICAgICAgICAgIHJldHVybgogICAgICAgICAgICBzZWxmLl9lbnRyaWVzLnJlbW92ZShyZW1vdmFibGUpCiAgICAgICAgICAgIHNlbGYuX2J5X2lkLnBvcChyZW1vdmFibGUudHJhbnNhY3Rpb25faWQsIE5vbmUpCg==', 'agent/competition_loop.py': 'IiIiQm91bmRlZCBwZXItZ2FtZSBsb29wIGFuZCBhbGwtZ2FtZSBjb21wZXRpdGlvbiBvcmNoZXN0cmF0aW9uLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRpbWUKZnJvbSBjb25jdXJyZW50LmZ1dHVyZXMgaW1wb3J0IEZJUlNUX0NPTVBMRVRFRCwgRnV0dXJlLCBUaHJlYWRQb29sRXhlY3V0b3IsIHdhaXQKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIENhbGxhYmxlLCBQcm90b2NvbAoKZnJvbSBhcmNlbmdpbmUgaW1wb3J0IEdhbWVTdGF0ZQoKZnJvbSAuYWN0aW9uIGltcG9ydCBBY3Rpb25EZWNpc2lvbgpmcm9tIC5jb250cm9sbGVyIGltcG9ydCBEZXRlcm1pbmlzdGljRmFsbGJhY2sKZnJvbSAuZGlhZ25vc3RpY3MgaW1wb3J0IFRyYW5zaXRpb25EaWFnbm9zdGljUmVjb3JkZXIsIGNsYXNzaWZ5X3Byb3Bvc2FsX3JlamVjdGlvbgpmcm9tIC5mcmFtZXdvcmtfYWRhcHRlciBpbXBvcnQgRmluYWxpemF0aW9uVW5rbm93biwgT3V0Y29tZVVua25vd24sIFByZURpc3BhdGNoRmFpbHVyZQpmcm9tIC5zdGF0ZSBpbXBvcnQgR2FtZVJ1bnRpbWVTdGF0ZQpmcm9tIC53YXRjaGRvZyBpbXBvcnQgRGVhZGxpbmVXYXRjaGRvZwoKCmNsYXNzIFBvbGljeShQcm90b2NvbCk6CiAgICBkZWYgcHJvcG9zZShzZWxmLCBzdGF0ZTogR2FtZVJ1bnRpbWVTdGF0ZSkgLT4gQWN0aW9uRGVjaXNpb246IC4uLgoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUsIHNsb3RzPVRydWUpCmNsYXNzIEdhbWVSZXN1bHQ6CiAgICBnYW1lX2lkOiBzdHIKICAgIHN0YXR1czogc3RyCiAgICBsZXZlbHNfY29tcGxldGVkOiBpbnQKICAgIGFja25vd2xlZGdlZF9hY3Rpb25zOiBpbnQKICAgIGFtYmlndW91c19hY3Rpb25zOiBpbnQKICAgIHRlcm1pbmFsX3JlYXNvbjogc3RyCiAgICBjb250cm9sbGVyX2l0ZXJhdGlvbnM6IGludCA9IDAKICAgIHBvbGljeV9mYWlsdXJlczogaW50ID0gMAogICAgaW5mZXJlbmNlX3JlcXVlc3RzOiBpbnQgPSAwCiAgICBpbmZlcmVuY2VfY29tcGxldGlvbnM6IGludCA9IDAKICAgIGluZmVyZW5jZV90cmFuc3BvcnRfZmFpbHVyZXM6IGludCA9IDAKICAgIGluZmVyZW5jZV9xdWV1ZV9mYWlsdXJlczogaW50ID0gMAogICAgaW5mZXJlbmNlX3Byb21wdF90b2tlbnM6IGludCA9IDAKICAgIGluZmVyZW5jZV9jb21wbGV0aW9uX3Rva2VuczogaW50ID0gMAogICAgaW5mZXJlbmNlX2VsYXBzZWRfc2Vjb25kczogZmxvYXQgPSAwLjAKICAgIHdvcmtzcGFjZV9pbnZvY2F0aW9uczogaW50ID0gMAogICAgd29ya3NwYWNlX2VsYXBzZWRfc2Vjb25kczogZmxvYXQgPSAwLjAKICAgIHBhcnNlcl9yZXBhaXJzOiBpbnQgPSAwCgoKY2xhc3MgQ29tcGV0aXRpb25BZ2VudExvb3A6CiAgICAiIiJPd25zIG9uZSBjbGllbnQgbGlmZWN5Y2xlIHdpdGhvdXQgcmV0YWluaW5nIGFuIHVuYm91bmRlZCBmcmFtZSBsaXN0LiIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGFkYXB0ZXI6IEFueSwKICAgICAgICBjbGllbnQ6IEFueSwKICAgICAgICAqLAogICAgICAgIHBvbGljeTogUG9saWN5IHwgTm9uZSA9IE5vbmUsCiAgICAgICAgbWF4X2FjdGlvbnM6IGludCA9IDgwLAogICAgICAgIHdhdGNoZG9nOiBEZWFkbGluZVdhdGNoZG9nIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgZGlhZ25vc3RpY3M6IFRyYW5zaXRpb25EaWFnbm9zdGljUmVjb3JkZXIgfCBOb25lID0gTm9uZSwKICAgICkgLT4gTm9uZToKICAgICAgICBpZiBtYXhfYWN0aW9ucyA8IDE6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm1heF9hY3Rpb25zIG11c3QgYmUgcG9zaXRpdmUiKQogICAgICAgIHNlbGYuYWRhcHRlciA9IGFkYXB0ZXIKICAgICAgICBzZWxmLmNsaWVudCA9IGNsaWVudAogICAgICAgIHNlbGYuZmFsbGJhY2sgPSBEZXRlcm1pbmlzdGljRmFsbGJhY2soKQogICAgICAgIHNlbGYucG9saWN5ID0gcG9saWN5IG9yIHNlbGYuZmFsbGJhY2sKICAgICAgICBzZWxmLm1heF9hY3Rpb25zID0gbWF4X2FjdGlvbnMKICAgICAgICBzZWxmLndhdGNoZG9nID0gd2F0Y2hkb2cgb3IgRGVhZGxpbmVXYXRjaGRvZygzMl80MDAsIDYwMCkKICAgICAgICBzZWxmLmRpYWdub3N0aWNzID0gZGlhZ25vc3RpY3MKICAgICAgICBzZWxmLnN0YXRlID0gR2FtZVJ1bnRpbWVTdGF0ZSgKICAgICAgICAgICAgY2xpZW50Lm9ic2VydmF0aW9uLAogICAgICAgICAgICBhY3Rpb25fYnVkZ2V0X2xpbWl0PW1heF9hY3Rpb25zLAogICAgICAgICAgICBkaWFnbm9zdGljcz1kaWFnbm9zdGljcywKICAgICAgICApCiAgICAgICAgc2VsZi5zdGF0ZS5jb3VudGVycy5saWZlY3ljbGVfYXR0ZW1wdGVkID0gMQogICAgICAgIHNlbGYuc3RhdGUuY291bnRlcnMubGlmZWN5Y2xlX2Fja25vd2xlZGdlZCA9IDEKICAgICAgICBzZWxmLnN0YXRlLmNvdW50ZXJzLmJvb3RzdHJhcF9zdGFydHMgPSAxCgogICAgZGVmIHJ1bihzZWxmKSAtPiBHYW1lUmVzdWx0OgogICAgICAgIHJlYXNvbiA9ICJhY3Rpb25fY2FwIgogICAgICAgIHRyeToKICAgICAgICAgICAgd2hpbGUgc2VsZi5zdGF0ZS5jb3VudGVycy5jb25zZXJ2YXRpdmVfc3BlbnRfYWN0aW9ucyA8IHNlbGYubWF4X2FjdGlvbnM6CiAgICAgICAgICAgICAgICBpZiBzZWxmLndhdGNoZG9nLnN0b3BfYWRtaXNzaW9uOgogICAgICAgICAgICAgICAgICAgIHJlYXNvbiA9ICJnbG9iYWxfZmluYWxpemF0aW9uX3Jlc2VydmUiCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIGlmIHNlbGYuc3RhdGUub2JzZXJ2YXRpb24uc3RhdGUgaXMgR2FtZVN0YXRlLldJTjoKICAgICAgICAgICAgICAgICAgICByZWFzb24gPSAid2luIgogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBzZWxmLnN0YXRlLmNvdW50ZXJzLmNvbnRyb2xsZXJfaXRlcmF0aW9ucyArPSAxCiAgICAgICAgICAgICAgICBiZWZvcmUgPSBzZWxmLnN0YXRlLm9ic2VydmF0aW9uCiAgICAgICAgICAgICAgICBzZWxmLl9kaWFnbm9zdGljKAogICAgICAgICAgICAgICAgICAgICJiZWdpbl90cmFuc2l0aW9uIiwKICAgICAgICAgICAgICAgICAgICBiZWZvcmUsCiAgICAgICAgICAgICAgICAgICAgaXRlcmF0aW9uPXNlbGYuc3RhdGUuY291bnRlcnMuY29udHJvbGxlcl9pdGVyYXRpb25zLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIGRlY2lzaW9uID0gc2VsZi5wb2xpY3kucHJvcG9zZShzZWxmLnN0YXRlKQogICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgICAgICAgICAgICAgc2VsZi5zdGF0ZS5jb3VudGVycy5wb2xpY3lfZmFpbHVyZXMgKz0gMQogICAgICAgICAgICAgICAgICAgIHNlbGYuX2RpYWdub3N0aWMoCiAgICAgICAgICAgICAgICAgICAgICAgICJub3RlX3JlamVjdGlvbiIsCiAgICAgICAgICAgICAgICAgICAgICAgIGNsYXNzaWZ5X3Byb3Bvc2FsX3JlamVjdGlvbihleGMpLAogICAgICAgICAgICAgICAgICAgICAgICBzdHIoZXhjKSwKICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgICAgICBkZWNpc2lvbiA9IHNlbGYuZmFsbGJhY2sucHJvcG9zZShzZWxmLnN0YXRlKQogICAgICAgICAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICAgICAgICAgIHJlYXNvbiA9ICJwb2xpY3lfZmFpbHVyZV9ub19sZWdhbF9mYWxsYmFjayIKICAgICAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIHNlbGYuX2RpYWdub3N0aWMoIm5vdGVfZGVjaXNpb24iLCBkZWNpc2lvbikKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBzZWxmLnN0YXRlLm1hcmtfcGVuZGluZ19hY3Rpb24oZGVjaXNpb24uZGVjaXNpb25faWQpCiAgICAgICAgICAgICAgICAgICAgb2JzZXJ2YXRpb24gPSBzZWxmLmFkYXB0ZXIuZGlzcGF0Y2goc2VsZi5jbGllbnQsIGRlY2lzaW9uKQogICAgICAgICAgICAgICAgZXhjZXB0IFByZURpc3BhdGNoRmFpbHVyZToKICAgICAgICAgICAgICAgICAgICBzZWxmLnN0YXRlLmNsZWFyX3BlbmRpbmdfYWN0aW9uKCkKICAgICAgICAgICAgICAgICAgICBzZWxmLl9kaWFnbm9zdGljKAogICAgICAgICAgICAgICAgICAgICAgICAiZmluaXNoX3VucmVzb2x2ZWQiLAogICAgICAgICAgICAgICAgICAgICAgICBwaGFzZT0icHJlX3RyYW5zcG9ydCIsCiAgICAgICAgICAgICAgICAgICAgICAgIGNhdGVnb3J5PSJwcmVfZGlzcGF0Y2hfZmFpbHVyZSIsCiAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgICAgIHJlYXNvbiA9ICJwcmVfZGlzcGF0Y2hfZmFpbHVyZSIKICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICAgICAgZXhjZXB0IE91dGNvbWVVbmtub3duOgogICAgICAgICAgICAgICAgICAgIHNlbGYuc3RhdGUucXVhcmFudGluZWQgPSBUcnVlCiAgICAgICAgICAgICAgICAgICAgc2VsZi5zdGF0ZS5jb3VudGVycy5hY3Rpb25zX2FtYmlndW91cyArPSAxCiAgICAgICAgICAgICAgICAgICAgc2VsZi5zdGF0ZS5jb3VudGVycy5jb25zZXJ2YXRpdmVfc3BlbnRfYWN0aW9ucyArPSAxCiAgICAgICAgICAgICAgICAgICAgc2VsZi5zdGF0ZS5tYXJrX3BlbmRpbmdfYWN0aW9uKGRlY2lzaW9uLmRlY2lzaW9uX2lkKQogICAgICAgICAgICAgICAgICAgIHNlbGYuX2RpYWdub3N0aWMoCiAgICAgICAgICAgICAgICAgICAgICAgICJmaW5pc2hfdW5yZXNvbHZlZCIsCiAgICAgICAgICAgICAgICAgICAgICAgIHBoYXNlPSJhY3Rpb25fZGlzcGF0Y2hfcG9zdF9lbnRyeSIsCiAgICAgICAgICAgICAgICAgICAgICAgIGNhdGVnb3J5PSJvdXRjb21lX3Vua25vd24iLAogICAgICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgICAgICByZWFzb24gPSAib3V0Y29tZV91bmtub3duX3F1YXJhbnRpbmUiCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIHNlbGYuc3RhdGUuY291bnRlcnMuYWN0aW9uc19hY2tub3dsZWRnZWQgKz0gMQogICAgICAgICAgICAgICAgc2VsZi5zdGF0ZS5jb3VudGVycy5jb25zZXJ2YXRpdmVfc3BlbnRfYWN0aW9ucyArPSAxCiAgICAgICAgICAgICAgICBpZiBkZWNpc2lvbi5hY3Rpb25faWQgPT0gMDoKICAgICAgICAgICAgICAgICAgICBzZWxmLnN0YXRlLmNvdW50ZXJzLmxhdGVyX3Jlc2V0c19hY2tub3dsZWRnZWQgKz0gMQogICAgICAgICAgICAgICAgc2VsZi5zdGF0ZS5yZXBsYWNlX29ic2VydmF0aW9uKAogICAgICAgICAgICAgICAgICAgIG9ic2VydmF0aW9uLAogICAgICAgICAgICAgICAgICAgIGFjdGlvbl9pZD1kZWNpc2lvbi5hY3Rpb25faWQsCiAgICAgICAgICAgICAgICAgICAgYWN0aW9uX2RhdGE9ZGVjaXNpb24uYWN0aW9uX2RhdGEsCiAgICAgICAgICAgICAgICAgICAgdHJhbnNpdGlvbl9pZD1kZWNpc2lvbi5kZWNpc2lvbl9pZCwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGlmIHNlbGYuc3RhdGUuZXZpZGVuY2UudHJhbnNpdGlvbnM6CiAgICAgICAgICAgICAgICAgICAgc2VsZi5fZGlhZ25vc3RpYygKICAgICAgICAgICAgICAgICAgICAgICAgImZpbmlzaF9hY2tub3dsZWRnZWQiLAogICAgICAgICAgICAgICAgICAgICAgICBiZWZvcmUsCiAgICAgICAgICAgICAgICAgICAgICAgIG9ic2VydmF0aW9uLAogICAgICAgICAgICAgICAgICAgICAgICBzZWxmLnN0YXRlLmV2aWRlbmNlLnRyYW5zaXRpb25zWy0xXSwKICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgZmluYWxseToKICAgICAgICAgICAgZmluYWxpemUgPSBnZXRhdHRyKHNlbGYuYWRhcHRlciwgImZpbmFsaXplX2NsaWVudCIsIE5vbmUpCiAgICAgICAgICAgIGlmIGZpbmFsaXplIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgZmluYWxpemUoc2VsZi5jbGllbnQpCgogICAgICAgIHNlbGYuc3RhdGUudGVybWluYWxfcmVhc29uID0gcmVhc29uCiAgICAgICAgcmVzdWx0ID0gR2FtZVJlc3VsdCgKICAgICAgICAgICAgZ2FtZV9pZD1zZWxmLmNsaWVudC5nYW1lX2lkLAogICAgICAgICAgICBzdGF0dXM9InF1YXJhbnRpbmVkIiBpZiBzZWxmLnN0YXRlLnF1YXJhbnRpbmVkIGVsc2UgImNvbXBsZXRlIiwKICAgICAgICAgICAgbGV2ZWxzX2NvbXBsZXRlZD1zZWxmLnN0YXRlLm9ic2VydmF0aW9uLmxldmVsc19jb21wbGV0ZWQsCiAgICAgICAgICAgIGFja25vd2xlZGdlZF9hY3Rpb25zPXNlbGYuc3RhdGUuY291bnRlcnMuYWN0aW9uc19hY2tub3dsZWRnZWQsCiAgICAgICAgICAgIGFtYmlndW91c19hY3Rpb25zPXNlbGYuc3RhdGUuY291bnRlcnMuYWN0aW9uc19hbWJpZ3VvdXMsCiAgICAgICAgICAgIHRlcm1pbmFsX3JlYXNvbj1yZWFzb24sCiAgICAgICAgICAgIGNvbnRyb2xsZXJfaXRlcmF0aW9ucz1zZWxmLnN0YXRlLmNvdW50ZXJzLmNvbnRyb2xsZXJfaXRlcmF0aW9ucywKICAgICAgICAgICAgcG9saWN5X2ZhaWx1cmVzPXNlbGYuc3RhdGUuY291bnRlcnMucG9saWN5X2ZhaWx1cmVzLAogICAgICAgICAgICBpbmZlcmVuY2VfcmVxdWVzdHM9c2VsZi5zdGF0ZS5jb3VudGVycy5pbmZlcmVuY2VfcmVxdWVzdHMsCiAgICAgICAgICAgIGluZmVyZW5jZV9jb21wbGV0aW9ucz1zZWxmLnN0YXRlLmNvdW50ZXJzLmluZmVyZW5jZV9jb21wbGV0aW9ucywKICAgICAgICAgICAgaW5mZXJlbmNlX3RyYW5zcG9ydF9mYWlsdXJlcz1zZWxmLnN0YXRlLmNvdW50ZXJzLmluZmVyZW5jZV90cmFuc3BvcnRfZmFpbHVyZXMsCiAgICAgICAgICAgIGluZmVyZW5jZV9xdWV1ZV9mYWlsdXJlcz1zZWxmLnN0YXRlLmNvdW50ZXJzLmluZmVyZW5jZV9xdWV1ZV9mYWlsdXJlcywKICAgICAgICAgICAgaW5mZXJlbmNlX3Byb21wdF90b2tlbnM9c2VsZi5zdGF0ZS5jb3VudGVycy5pbmZlcmVuY2VfcHJvbXB0X3Rva2VucywKICAgICAgICAgICAgaW5mZXJlbmNlX2NvbXBsZXRpb25fdG9rZW5zPXNlbGYuc3RhdGUuY291bnRlcnMuaW5mZXJlbmNlX2NvbXBsZXRpb25fdG9rZW5zLAogICAgICAgICAgICBpbmZlcmVuY2VfZWxhcHNlZF9zZWNvbmRzPXNlbGYuc3RhdGUuY291bnRlcnMuaW5mZXJlbmNlX2VsYXBzZWRfc2Vjb25kcywKICAgICAgICAgICAgd29ya3NwYWNlX2ludm9jYXRpb25zPXNlbGYuc3RhdGUuY291bnRlcnMud29ya3NwYWNlX2ludm9jYXRpb25zLAogICAgICAgICAgICB3b3Jrc3BhY2VfZWxhcHNlZF9zZWNvbmRzPXNlbGYuc3RhdGUuY291bnRlcnMud29ya3NwYWNlX2VsYXBzZWRfc2Vjb25kcywKICAgICAgICAgICAgcGFyc2VyX3JlcGFpcnM9c2VsZi5zdGF0ZS5jb3VudGVycy5wYXJzZXJfcmVwYWlycywKICAgICAgICApCiAgICAgICAgcmV0dXJuIHJlc3VsdAoKICAgIGRlZiBfZGlhZ25vc3RpYyhzZWxmLCBtZXRob2Q6IHN0ciwgKmFyZ3M6IEFueSwgKiprd2FyZ3M6IEFueSkgLT4gTm9uZToKICAgICAgICAiIiJPYnNlcnZlIHdpdGhvdXQgYWxsb3dpbmcgZGlhZ25vc3RpYyBmYWlsdXJlcyB0byBhbHRlciBsZWdhbCBwbGF5LiIiIgoKICAgICAgICBpZiBzZWxmLmRpYWdub3N0aWNzIGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybgogICAgICAgIHRyeToKICAgICAgICAgICAgZ2V0YXR0cihzZWxmLmRpYWdub3N0aWNzLCBtZXRob2QpKCphcmdzLCAqKmt3YXJncykKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgc2VsZi5kaWFnbm9zdGljcy5jYXB0dXJlX2Vycm9yKG1ldGhvZCwgZXhjKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgcGFzcwoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUsIHNsb3RzPVRydWUpCmNsYXNzIENvbXBldGl0aW9uUnVuUmVzdWx0OgogICAgcmVzdWx0czogdHVwbGVbR2FtZVJlc3VsdCwgLi4uXQogICAgZmluYWxpemF0aW9uX3N0YXR1czogc3RyCiAgICBzY29yZWNhcmQ6IGRpY3Rbc3RyLCBBbnldIHwgTm9uZQoKCmNsYXNzIENvbXBldGl0aW9uT3JjaGVzdHJhdG9yOgogICAgIiIiT25lIHNjb3JlY2FyZCwgYXQgbW9zdCBvbmUgYm9vdHN0cmFwIHBlciBnYW1lLCBzdHJlYW1pbmcgd29ya2VyIHN0YXJ0LiIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGFkYXB0ZXI6IEFueSwKICAgICAgICBnYW1lX2lkczogbGlzdFtzdHJdIHwgdHVwbGVbc3RyLCAuLi5dLAogICAgICAgICosCiAgICAgICAgbWF4X3dvcmtlcnM6IGludCA9IDgsCiAgICAgICAgbWF4X2FjdGlvbnM6IGludCA9IDgwLAogICAgICAgIHdhdGNoZG9nOiBEZWFkbGluZVdhdGNoZG9nIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgaHVuZ193b3JrZXJfZ3JhY2Vfc2Vjb25kczogZmxvYXQgPSAxNS4wLAogICAgICAgIHBvbGljeV9mYWN0b3J5OiBDYWxsYWJsZVtbQW55XSwgUG9saWN5XSB8IE5vbmUgPSBOb25lLAogICAgICAgIGRpYWdub3N0aWNfZmFjdG9yeTogQ2FsbGFibGVbW0FueV0sIFRyYW5zaXRpb25EaWFnbm9zdGljUmVjb3JkZXJdIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgcnVuX3RhZzogc3RyID0gInBsYW44LWUwIiwKICAgICkgLT4gTm9uZToKICAgICAgICBpZiBsZW4oc2V0KGdhbWVfaWRzKSkgIT0gbGVuKGdhbWVfaWRzKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZ2FtZSBpbnZlbnRvcnkgY29udGFpbnMgZHVwbGljYXRlcyIpCiAgICAgICAgc2VsZi5hZGFwdGVyID0gYWRhcHRlcgogICAgICAgIHNlbGYuZ2FtZV9pZHMgPSB0dXBsZShnYW1lX2lkcykKICAgICAgICBzZWxmLm1heF93b3JrZXJzID0gbWF4KDEsIG1heF93b3JrZXJzKQogICAgICAgIHNlbGYubWF4X2FjdGlvbnMgPSBtYXhfYWN0aW9ucwogICAgICAgIHNlbGYud2F0Y2hkb2cgPSB3YXRjaGRvZyBvciBEZWFkbGluZVdhdGNoZG9nKDMyXzQwMCwgNjAwKQogICAgICAgIHNlbGYuaHVuZ193b3JrZXJfZ3JhY2Vfc2Vjb25kcyA9IG1heCgwLjAsIGh1bmdfd29ya2VyX2dyYWNlX3NlY29uZHMpCiAgICAgICAgc2VsZi5wb2xpY3lfZmFjdG9yeSA9IHBvbGljeV9mYWN0b3J5CiAgICAgICAgc2VsZi5kaWFnbm9zdGljX2ZhY3RvcnkgPSBkaWFnbm9zdGljX2ZhY3RvcnkKICAgICAgICBzZWxmLnJ1bl90YWcgPSBydW5fdGFnCgogICAgZGVmIHJ1bihzZWxmKSAtPiBDb21wZXRpdGlvblJ1blJlc3VsdDoKICAgICAgICBzZWxmLndhdGNoZG9nLnN0YXJ0X3N1cGVydmlzb3IoKQogICAgICAgIHJlc3VsdHM6IGRpY3Rbc3RyLCBHYW1lUmVzdWx0XSA9IHt9CiAgICAgICAgdHJ5OgogICAgICAgICAgICBzZWxmLmFkYXB0ZXIub3Blbl9zY29yZWNhcmQodGFncz1bc2VsZi5ydW5fdGFnXSkKICAgICAgICBleGNlcHQgT3V0Y29tZVVua25vd246CiAgICAgICAgICAgIHNlbGYud2F0Y2hkb2cuc3RvcF9zdXBlcnZpc29yKCkKICAgICAgICAgICAgcmV0dXJuIENvbXBldGl0aW9uUnVuUmVzdWx0KAogICAgICAgICAgICAgICAgdHVwbGUoc2VsZi5fdW50b3VjaGVkKGdhbWVfaWQsICJzY29yZWNhcmRfb3Blbl91bmtub3duIikgZm9yIGdhbWVfaWQgaW4gc2VsZi5nYW1lX2lkcyksCiAgICAgICAgICAgICAgICAib3Blbl91bmtub3duIiwKICAgICAgICAgICAgICAgIE5vbmUsCiAgICAgICAgICAgICkKCiAgICAgICAgZnV0dXJlczogZGljdFtGdXR1cmVbR2FtZVJlc3VsdF0sIHN0cl0gPSB7fQogICAgICAgIGV4ZWN1dG9yID0gVGhyZWFkUG9vbEV4ZWN1dG9yKG1heF93b3JrZXJzPXNlbGYubWF4X3dvcmtlcnMsIHRocmVhZF9uYW1lX3ByZWZpeD0iYXJjMy1jbGllbnQiKQogICAgICAgIHRyeToKICAgICAgICAgICAgZm9yIGdhbWVfaWQgaW4gc2VsZi5nYW1lX2lkczoKICAgICAgICAgICAgICAgIGlmIHNlbGYud2F0Y2hkb2cuc3RvcF9hZG1pc3Npb246CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0c1tnYW1lX2lkXSA9IHNlbGYuX3VudG91Y2hlZChnYW1lX2lkLCAiZ2xvYmFsX2ZpbmFsaXphdGlvbl9yZXNlcnZlIikKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIGNsaWVudCA9IHNlbGYuYWRhcHRlci5ib290c3RyYXAoZ2FtZV9pZCkKICAgICAgICAgICAgICAgIGV4Y2VwdCAoT3V0Y29tZVVua25vd24sIFByZURpc3BhdGNoRmFpbHVyZSk6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0c1tnYW1lX2lkXSA9IHNlbGYuX3VudG91Y2hlZChnYW1lX2lkLCAiYm9vdHN0cmFwX2ZhaWxlZF9vcl91bmtub3duIikKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgbG9vcCA9IENvbXBldGl0aW9uQWdlbnRMb29wKAogICAgICAgICAgICAgICAgICAgIHNlbGYuYWRhcHRlciwKICAgICAgICAgICAgICAgICAgICBjbGllbnQsCiAgICAgICAgICAgICAgICAgICAgcG9saWN5PXNlbGYucG9saWN5X2ZhY3RvcnkoY2xpZW50KSBpZiBzZWxmLnBvbGljeV9mYWN0b3J5IGlzIG5vdCBOb25lIGVsc2UgTm9uZSwKICAgICAgICAgICAgICAgICAgICBtYXhfYWN0aW9ucz1zZWxmLm1heF9hY3Rpb25zLAogICAgICAgICAgICAgICAgICAgIHdhdGNoZG9nPXNlbGYud2F0Y2hkb2csCiAgICAgICAgICAgICAgICAgICAgZGlhZ25vc3RpY3M9c2VsZi5kaWFnbm9zdGljX2ZhY3RvcnkoY2xpZW50KQogICAgICAgICAgICAgICAgICAgIGlmIHNlbGYuZGlhZ25vc3RpY19mYWN0b3J5IGlzIG5vdCBOb25lCiAgICAgICAgICAgICAgICAgICAgZWxzZSBOb25lLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgZnV0dXJlc1tleGVjdXRvci5zdWJtaXQobG9vcC5ydW4pXSA9IGdhbWVfaWQKICAgICAgICAgICAgcGVuZGluZyA9IHNldChmdXR1cmVzKQogICAgICAgICAgICB3aGlsZSBwZW5kaW5nIGFuZCBub3Qgc2VsZi53YXRjaGRvZy5zdG9wX2FkbWlzc2lvbjoKICAgICAgICAgICAgICAgIHVudGlsX3N0b3AgPSBtYXgoCiAgICAgICAgICAgICAgICAgICAgMC4wLAogICAgICAgICAgICAgICAgICAgIHNlbGYud2F0Y2hkb2cuaGFyZF9zZWNvbmRzCiAgICAgICAgICAgICAgICAgICAgLSBzZWxmLndhdGNoZG9nLmZpbmFsaXphdGlvbl9yZXNlcnZlX3NlY29uZHMKICAgICAgICAgICAgICAgICAgICAtIHNlbGYud2F0Y2hkb2cuZWxhcHNlZCwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGNvbXBsZXRlZCwgcGVuZGluZyA9IHdhaXQoCiAgICAgICAgICAgICAgICAgICAgcGVuZGluZywKICAgICAgICAgICAgICAgICAgICB0aW1lb3V0PW1pbigwLjI1LCB1bnRpbF9zdG9wKSwKICAgICAgICAgICAgICAgICAgICByZXR1cm5fd2hlbj1GSVJTVF9DT01QTEVURUQsCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBmb3IgZnV0dXJlIGluIGNvbXBsZXRlZDoKICAgICAgICAgICAgICAgICAgICBnYW1lX2lkID0gZnV0dXJlc1tmdXR1cmVdCiAgICAgICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgICAgICByZXN1bHRzW2dhbWVfaWRdID0gZnV0dXJlLnJlc3VsdCgpCiAgICAgICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0c1tnYW1lX2lkXSA9IHNlbGYuX3VudG91Y2hlZChnYW1lX2lkLCAiY29udGFpbmVkX3dvcmtlcl9mYWlsdXJlIikKCiAgICAgICAgICAgIGlmIHBlbmRpbmc6CiAgICAgICAgICAgICAgICBzZWxmLndhdGNoZG9nLmNhbmNlbCgpCiAgICAgICAgICAgICAgICBjb21wbGV0ZWQsIHBlbmRpbmcgPSB3YWl0KHBlbmRpbmcsIHRpbWVvdXQ9c2VsZi5odW5nX3dvcmtlcl9ncmFjZV9zZWNvbmRzKQogICAgICAgICAgICAgICAgZm9yIGZ1dHVyZSBpbiBjb21wbGV0ZWQ6CiAgICAgICAgICAgICAgICAgICAgZ2FtZV9pZCA9IGZ1dHVyZXNbZnV0dXJlXQogICAgICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0c1tnYW1lX2lkXSA9IGZ1dHVyZS5yZXN1bHQoKQogICAgICAgICAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICAgICAgICAgIHJlc3VsdHNbZ2FtZV9pZF0gPSBzZWxmLl91bnRvdWNoZWQoZ2FtZV9pZCwgImNvbnRhaW5lZF93b3JrZXJfZmFpbHVyZSIpCiAgICAgICAgICAgICAgICBmb3IgZnV0dXJlIGluIHBlbmRpbmc6CiAgICAgICAgICAgICAgICAgICAgZnV0dXJlLmNhbmNlbCgpCiAgICAgICAgICAgICAgICAgICAgcmVzdWx0c1tmdXR1cmVzW2Z1dHVyZV1dID0gc2VsZi5fdW50b3VjaGVkKGZ1dHVyZXNbZnV0dXJlXSwgImh1bmdfd29ya2VyX3F1YXJhbnRpbmVkIikKICAgICAgICBmaW5hbGx5OgogICAgICAgICAgICBleGVjdXRvci5zaHV0ZG93bih3YWl0PUZhbHNlLCBjYW5jZWxfZnV0dXJlcz1UcnVlKQoKICAgICAgICBzY29yZWNhcmQ6IGRpY3Rbc3RyLCBBbnldIHwgTm9uZSA9IE5vbmUKICAgICAgICBmaW5hbGl6YXRpb25fc3RhdHVzID0gImFja25vd2xlZGdlZCIKICAgICAgICB0cnk6CiAgICAgICAgICAgIHNjb3JlY2FyZCA9IHNlbGYuYWRhcHRlci5jbG9zZV9zY29yZWNhcmQoKQogICAgICAgIGV4Y2VwdCBGaW5hbGl6YXRpb25Vbmtub3duOgogICAgICAgICAgICBmaW5hbGl6YXRpb25fc3RhdHVzID0gImZpbmFsaXphdGlvbl91bmtub3duIgogICAgICAgIGV4Y2VwdCBQcmVEaXNwYXRjaEZhaWx1cmU6CiAgICAgICAgICAgIGZpbmFsaXphdGlvbl9zdGF0dXMgPSAiY2xvc2Vfbm90X2Rpc3BhdGNoZWQiCiAgICAgICAgb3JkZXJlZCA9IHR1cGxlKHJlc3VsdHMuZ2V0KGdhbWVfaWQsIHNlbGYuX3VudG91Y2hlZChnYW1lX2lkLCAibm90X2F0dGVtcHRlZCIpKSBmb3IgZ2FtZV9pZCBpbiBzZWxmLmdhbWVfaWRzKQogICAgICAgIHNlbGYud2F0Y2hkb2cuc3RvcF9zdXBlcnZpc29yKCkKICAgICAgICByZXR1cm4gQ29tcGV0aXRpb25SdW5SZXN1bHQob3JkZXJlZCwgZmluYWxpemF0aW9uX3N0YXR1cywgc2NvcmVjYXJkKQoKICAgIEBzdGF0aWNtZXRob2QKICAgIGRlZiBfdW50b3VjaGVkKGdhbWVfaWQ6IHN0ciwgcmVhc29uOiBzdHIpIC0+IEdhbWVSZXN1bHQ6CiAgICAgICAgcmV0dXJuIEdhbWVSZXN1bHQoZ2FtZV9pZCwgInVudG91Y2hlZCIsIDAsIDAsIDAsIHJlYXNvbikK', 'agent/controller.py': 'IiIiRGV0ZXJtaW5pc3RpYyBsZWdhbCBmYWxsYmFjayBjb250cm9sbGVyIGZvciBQaGFzZSAwLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IHJhbmRvbQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCgpmcm9tIGFyY2VuZ2luZSBpbXBvcnQgR2FtZUFjdGlvbiwgR2FtZVN0YXRlCgpmcm9tIC5hY3Rpb24gaW1wb3J0IEFjdGlvbkRlY2lzaW9uLCBEaXNwbGF5UG9pbnQsIG5vcm1hbGl6ZV9sZWdhbF9hY3Rpb25zCmZyb20gLnN0YXRlIGltcG9ydCBHYW1lUnVudGltZVN0YXRlCgoKQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFzcyBEZXRlcm1pbmlzdGljRmFsbGJhY2s6CiAgICBzdGFibGVfc2VlZDogc3RyID0gImFyYzMtZTAtdjEiCiAgICBfcm5nX2J5X3Nlc3Npb246IGRpY3Rbc3RyLCByYW5kb20uUmFuZG9tXSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0LCBpbml0PUZhbHNlKQogICAgX2NsaWNrX2luZGV4X2J5X3Nlc3Npb246IGRpY3Rbc3RyLCBpbnRdID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QsIGluaXQ9RmFsc2UpCiAgICBfZGVjaXNpb25faW5kZXhfYnlfc2Vzc2lvbjogZGljdFtzdHIsIGludF0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCwgaW5pdD1GYWxzZSkKCiAgICBkZWYgX3JuZyhzZWxmLCBndWlkOiBzdHIpIC0+IHJhbmRvbS5SYW5kb206CiAgICAgICAgaWYgZ3VpZCBub3QgaW4gc2VsZi5fcm5nX2J5X3Nlc3Npb246CiAgICAgICAgICAgICMgVGhlIEdVSUQgaXMgb25seSBhbiBpc29sYXRpb24ga2V5LiBJdCBkb2VzIG5vdCBpbmZsdWVuY2UgcG9saWN5LgogICAgICAgICAgICBzZWVkX2J5dGVzID0gaGFzaGxpYi5zaGEyNTYoc2VsZi5zdGFibGVfc2VlZC5lbmNvZGUoKSkuZGlnZXN0KCkKICAgICAgICAgICAgc2VsZi5fcm5nX2J5X3Nlc3Npb25bZ3VpZF0gPSByYW5kb20uUmFuZG9tKGludC5mcm9tX2J5dGVzKHNlZWRfYnl0ZXNbOjhdLCAiYmlnIikpCiAgICAgICAgcmV0dXJuIHNlbGYuX3JuZ19ieV9zZXNzaW9uW2d1aWRdCgogICAgZGVmIHByb3Bvc2Uoc2VsZiwgc3RhdGU6IEdhbWVSdW50aW1lU3RhdGUpIC0+IEFjdGlvbkRlY2lzaW9uOgogICAgICAgIG9icyA9IHN0YXRlLm9ic2VydmF0aW9uCiAgICAgICAgZGVjaXNpb25faW5kZXggPSBzZWxmLl9kZWNpc2lvbl9pbmRleF9ieV9zZXNzaW9uLmdldChvYnMuZ3VpZCwgMCkKICAgICAgICBzZWxmLl9kZWNpc2lvbl9pbmRleF9ieV9zZXNzaW9uW29icy5ndWlkXSA9IGRlY2lzaW9uX2luZGV4ICsgMQogICAgICAgIGRlY2lzaW9uX2lkID0gZiJmYWxsYmFjazp7ZGVjaXNpb25faW5kZXh9IgogICAgICAgIGxlZ2FsID0gbm9ybWFsaXplX2xlZ2FsX2FjdGlvbnMob2JzLmF2YWlsYWJsZV9hY3Rpb25zKQogICAgICAgIGlmIG5vdCBsZWdhbDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJlbnZpcm9ubWVudCBleHBvc2VkIG5vIGxlZ2FsIGFjdGlvbnMiKQogICAgICAgIGlmIG9icy5zdGF0ZSBpcyBHYW1lU3RhdGUuR0FNRV9PVkVSIGFuZCBHYW1lQWN0aW9uLlJFU0VULnZhbHVlIGluIGxlZ2FsOgogICAgICAgICAgICByZXR1cm4gQWN0aW9uRGVjaXNpb24uc2ltcGxlKEdhbWVBY3Rpb24uUkVTRVQsIGNvbnRyb2xsZXJfbW9kZT0icmVjb3ZlcnkiLCBkZWNpc2lvbl9pZD1kZWNpc2lvbl9pZCkKCiAgICAgICAgbm9uX3Jlc2V0ID0gc29ydGVkKHZhbHVlIGZvciB2YWx1ZSBpbiBsZWdhbCBpZiB2YWx1ZSAhPSBHYW1lQWN0aW9uLlJFU0VULnZhbHVlKQogICAgICAgIGNhbmRpZGF0ZXMgPSBub25fcmVzZXQgb3Igc29ydGVkKGxlZ2FsKQogICAgICAgIGNob3NlbiA9IHNlbGYuX3JuZyhvYnMuZ3VpZCkuY2hvaWNlKGNhbmRpZGF0ZXMpCiAgICAgICAgaWYgY2hvc2VuID09IEdhbWVBY3Rpb24uQUNUSU9ONi52YWx1ZToKICAgICAgICAgICAgIyBEZXRlcm1pbmlzdGljIGxvdy1kaXNjcmVwYW5jeSB0cmF2ZXJzYWwgb2YgdGhlIDY0eDY0IGRpc3BsYXkuCiAgICAgICAgICAgIGluZGV4ID0gc2VsZi5fY2xpY2tfaW5kZXhfYnlfc2Vzc2lvbi5nZXQob2JzLmd1aWQsIDApCiAgICAgICAgICAgIHNlbGYuX2NsaWNrX2luZGV4X2J5X3Nlc3Npb25bb2JzLmd1aWRdID0gaW5kZXggKyAxCiAgICAgICAgICAgIHBvaW50ID0gRGlzcGxheVBvaW50KHg9KGluZGV4ICogMzcpICUgNjQsIHk9KGluZGV4ICogMjMpICUgNjQpCiAgICAgICAgICAgIHJldHVybiBBY3Rpb25EZWNpc2lvbi5jbGljayhwb2ludCwgY29udHJvbGxlcl9tb2RlPSJleHBsb3JlIiwgZGVjaXNpb25faWQ9ZGVjaXNpb25faWQpCiAgICAgICAgcmV0dXJuIEFjdGlvbkRlY2lzaW9uLnNpbXBsZShjaG9zZW4sIGNvbnRyb2xsZXJfbW9kZT0iZXhwbG9yZSIsIGRlY2lzaW9uX2lkPWRlY2lzaW9uX2lkKQo=', 'agent/diagnostics.py': '...', 'agent/evidence.py': '...', 'agent/framework_adapter.py': '...', 'agent/state.py': 'IiIiQm91bmRlZCB0eXBlZCBzdGF0ZSB1c2VkIGJ5IHRoZSBjb21wZXRpdGlvbiBsb29wLgoKYGBGcmFtZURhdGEuZnJhbWVgYCBpcyBhbiBvcmRlcmVkIHRlbXBvcmFsIHNlcXVlbmNlIHByb2R1Y2VkIGJ5IG9uZSBlbmdpbmUKYWN0aW9uLiAgSXQgaXMgZGVsaWJlcmF0ZWx5IG5hbWVkIGBgZnJhbWVzYGAgaGVyZTsgdHJlYXRpbmcgaXQgYXMgdmlzdWFsCmxheWVycyB3b3VsZCBkZXN0cm95IGFuaW1hdGlvbiBldmlkZW5jZSBhbmQgbWFrZSByZXRlbnRpb24gcG9saWN5IGFtYmlndW91cy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIE1hcHBpbmcKCmltcG9ydCBudW1weSBhcyBucApmcm9tIGFyY2VuZ2luZSBpbXBvcnQgRnJhbWVEYXRhLCBGcmFtZURhdGFSYXcsIEdhbWVTdGF0ZQoKZnJvbSAuZXZpZGVuY2UgaW1wb3J0IEV2aWRlbmNlU3RvcmUKCgpkZWYgX3BhY2tfZnJhbWVzKGZyYW1lczogQW55KSAtPiB0dXBsZVtucC5uZGFycmF5LCAuLi5dOgogICAgcGFja2VkOiBsaXN0W25wLm5kYXJyYXldID0gW10KICAgIGZvciBmcmFtZSBpbiBmcmFtZXMgb3IgW106CiAgICAgICAgc291cmNlID0gbnAuYXNhcnJheShmcmFtZSkKICAgICAgICBpZiBzb3VyY2UubmRpbSAhPSAyIG9yIG5vdCBucC5pc3N1YmR0eXBlKHNvdXJjZS5kdHlwZSwgbnAuaW50ZWdlcik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm9ic2VydmF0aW9uIGZyYW1lcyBtdXN0IGJlIHR3by1kaW1lbnNpb25hbCBpbnRlZ2VyIGdyaWRzIikKICAgICAgICBpZiBzb3VyY2Uuc2l6ZSBhbmQgKGludChzb3VyY2UubWluKCkpIDwgMCBvciBpbnQoc291cmNlLm1heCgpKSA+IDI1NSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm9ic2VydmF0aW9uIGZyYW1lcyBtdXN0IGZpdCB1aW50OCB3aXRob3V0IGxvc3MiKQogICAgICAgIGl0ZW0gPSBucC5hc2NvbnRpZ3VvdXNhcnJheShzb3VyY2UsIGR0eXBlPW5wLnVpbnQ4KQogICAgICAgIGl0ZW0uc2V0ZmxhZ3Mod3JpdGU9RmFsc2UpCiAgICAgICAgcGFja2VkLmFwcGVuZChpdGVtKQogICAgaWYgbm90IHBhY2tlZDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJvYnNlcnZhdGlvbiBoYXMgbm8gcmVuZGVyZWQgZnJhbWVzIikKICAgIHJldHVybiB0dXBsZShwYWNrZWQpCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSwgc2xvdHM9VHJ1ZSkKY2xhc3MgT2JzZXJ2YXRpb246CiAgICBnYW1lX2lkOiBzdHIKICAgIGZyYW1lczogdHVwbGVbbnAubmRhcnJheSwgLi4uXQogICAgc3RhdGU6IEdhbWVTdGF0ZQogICAgbGV2ZWxzX2NvbXBsZXRlZDogaW50CiAgICB3aW5fbGV2ZWxzOiBpbnQKICAgIGd1aWQ6IHN0cgogICAgYXZhaWxhYmxlX2FjdGlvbnM6IHR1cGxlW2ludCwgLi4uXQogICAgZnVsbF9yZXNldDogYm9vbCA9IEZhbHNlCgogICAgQGNsYXNzbWV0aG9kCiAgICBkZWYgZnJvbV92YWx1ZShjbHMsIHZhbHVlOiBGcmFtZURhdGEgfCBGcmFtZURhdGFSYXcgfCBkaWN0W3N0ciwgQW55XSkgLT4gIk9ic2VydmF0aW9uIjoKICAgICAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBkaWN0KToKICAgICAgICAgICAgZnJhbWUgPSBGcmFtZURhdGEubW9kZWxfdmFsaWRhdGUodmFsdWUpCiAgICAgICAgICAgIHNvdXJjZSA9IGZyYW1lCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc291cmNlID0gdmFsdWUKICAgICAgICBndWlkID0gZ2V0YXR0cihzb3VyY2UsICJndWlkIiwgTm9uZSkKICAgICAgICBpZiBub3QgZ3VpZDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigib2JzZXJ2YXRpb24gaGFzIG5vIGJvdW5kIGd1aWQiKQogICAgICAgIHJldHVybiBjbHMoCiAgICAgICAgICAgIGdhbWVfaWQ9c3RyKGdldGF0dHIoc291cmNlLCAiZ2FtZV9pZCIsICIiKSksCiAgICAgICAgICAgIGZyYW1lcz1fcGFja19mcmFtZXMoZ2V0YXR0cihzb3VyY2UsICJmcmFtZSIsIFtdKSksCiAgICAgICAgICAgIHN0YXRlPWdldGF0dHIoc291cmNlLCAic3RhdGUiLCBHYW1lU3RhdGUuTk9UX1BMQVlFRCksCiAgICAgICAgICAgIGxldmVsc19jb21wbGV0ZWQ9aW50KGdldGF0dHIoc291cmNlLCAibGV2ZWxzX2NvbXBsZXRlZCIsIDApKSwKICAgICAgICAgICAgd2luX2xldmVscz1pbnQoZ2V0YXR0cihzb3VyY2UsICJ3aW5fbGV2ZWxzIiwgMCkpLAogICAgICAgICAgICBndWlkPXN0cihndWlkKSwKICAgICAgICAgICAgYXZhaWxhYmxlX2FjdGlvbnM9dHVwbGUoaW50KHgudmFsdWUgaWYgaGFzYXR0cih4LCAidmFsdWUiKSBlbHNlIHgpIGZvciB4IGluIGdldGF0dHIoc291cmNlLCAiYXZhaWxhYmxlX2FjdGlvbnMiLCBbXSkpLAogICAgICAgICAgICBmdWxsX3Jlc2V0PWJvb2woZ2V0YXR0cihzb3VyY2UsICJmdWxsX3Jlc2V0IiwgRmFsc2UpKSwKICAgICAgICApCgogICAgQHByb3BlcnR5CiAgICBkZWYgY2Fub25pY2FsX2hhc2goc2VsZikgLT4gc3RyOgogICAgICAgIGhlYWRlciA9IHsKICAgICAgICAgICAgInZlcnNpb24iOiAiYXJjMy1vYnNlcnZhdGlvbi12MiIsCiAgICAgICAgICAgICJnYW1lX2lkIjogc2VsZi5nYW1lX2lkLAogICAgICAgICAgICAic3RhdGUiOiBzdHIoc2VsZi5zdGF0ZS52YWx1ZSksCiAgICAgICAgICAgICJsZXZlbHNfY29tcGxldGVkIjogc2VsZi5sZXZlbHNfY29tcGxldGVkLAogICAgICAgICAgICAid2luX2xldmVscyI6IHNlbGYud2luX2xldmVscywKICAgICAgICAgICAgImF2YWlsYWJsZV9hY3Rpb25zIjogbGlzdChzZWxmLmF2YWlsYWJsZV9hY3Rpb25zKSwKICAgICAgICAgICAgImR0eXBlIjogInVpbnQ4IiwKICAgICAgICAgICAgImJ5dGVfb3JkZXIiOiAibm90X2FwcGxpY2FibGUiLAogICAgICAgICAgICAib3JkZXIiOiAiQyIsCiAgICAgICAgICAgICJmcmFtZXMiOiBbbGlzdChmcmFtZS5zaGFwZSkgZm9yIGZyYW1lIGluIHNlbGYuZnJhbWVzXSwKICAgICAgICB9CiAgICAgICAgZGlnZXN0ID0gaGFzaGxpYi5zaGEyNTYoanNvbi5kdW1wcyhoZWFkZXIsIHNvcnRfa2V5cz1UcnVlLCBzZXBhcmF0b3JzPSgiLCIsICI6IikpLmVuY29kZSgpKQogICAgICAgIGZvciBmcmFtZSBpbiBzZWxmLmZyYW1lczoKICAgICAgICAgICAgZGlnZXN0LnVwZGF0ZShiIlwwIikKICAgICAgICAgICAgZGlnZXN0LnVwZGF0ZShmcmFtZS50b2J5dGVzKG9yZGVyPSJDIikpCiAgICAgICAgcmV0dXJuIGRpZ2VzdC5oZXhkaWdlc3QoKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIGxhdGVzdF9mcmFtZShzZWxmKSAtPiBucC5uZGFycmF5OgogICAgICAgIHJldHVybiBzZWxmLmZyYW1lc1stMV0KCgpAZGF0YWNsYXNzKHNsb3RzPVRydWUpCmNsYXNzIExpZmVjeWNsZUNvdW50ZXJzOgogICAgbGlmZWN5Y2xlX2F0dGVtcHRlZDogaW50ID0gMAogICAgbGlmZWN5Y2xlX2Fja25vd2xlZGdlZDogaW50ID0gMAogICAgbGlmZWN5Y2xlX2FtYmlndW91czogaW50ID0gMAogICAgYm9vdHN0cmFwX3N0YXJ0czogaW50ID0gMAogICAgYWN0aW9uc19hY2tub3dsZWRnZWQ6IGludCA9IDAKICAgIGFjdGlvbnNfYW1iaWd1b3VzOiBpbnQgPSAwCiAgICBsYXRlcl9yZXNldHNfYWNrbm93bGVkZ2VkOiBpbnQgPSAwCiAgICBsYXRlcl9yZXNldHNfYW1iaWd1b3VzOiBpbnQgPSAwCiAgICBjb25zZXJ2YXRpdmVfc3BlbnRfYWN0aW9uczogaW50ID0gMAogICAgY29udHJvbGxlcl9pdGVyYXRpb25zOiBpbnQgPSAwCiAgICBwb2xpY3lfZmFpbHVyZXM6IGludCA9IDAKICAgIGluZmVyZW5jZV9yZXF1ZXN0czogaW50ID0gMAogICAgaW5mZXJlbmNlX2NvbXBsZXRpb25zOiBpbnQgPSAwCiAgICBpbmZlcmVuY2VfdHJhbnNwb3J0X2ZhaWx1cmVzOiBpbnQgPSAwCiAgICBpbmZlcmVuY2VfcXVldWVfZmFpbHVyZXM6IGludCA9IDAKICAgIGluZmVyZW5jZV9wcm9tcHRfdG9rZW5zOiBpbnQgPSAwCiAgICBpbmZlcmVuY2VfY29tcGxldGlvbl90b2tlbnM6IGludCA9IDAKICAgIGluZmVyZW5jZV9lbGFwc2VkX3NlY29uZHM6IGZsb2F0ID0gMC4wCiAgICBwYXJzZXJfcmVwYWlyczogaW50ID0gMAogICAgd29ya3NwYWNlX2ludm9jYXRpb25zOiBpbnQgPSAwCiAgICB3b3Jrc3BhY2VfZWxhcHNlZF9zZWNvbmRzOiBmbG9hdCA9IDAuMAogICAgZXZpZGVuY2VfcmV0cmlldmFsczogaW50ID0gMAoKCkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgR2FtZVJ1bnRpbWVTdGF0ZToKICAgIG9ic2VydmF0aW9uOiBPYnNlcnZhdGlvbgogICAgYWN0aW9uX2J1ZGdldF9saW1pdDogaW50IHwgTm9uZSA9IE5vbmUKICAgIGNvdW50ZXJzOiBMaWZlY3ljbGVDb3VudGVycyA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1MaWZlY3ljbGVDb3VudGVycykKICAgIGV2aWRlbmNlOiBFdmlkZW5jZVN0b3JlID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PUV2aWRlbmNlU3RvcmUpCiAgICAjIFdyaXRlLW9ubHkgb2JzZXJ2YXRpb25hbCBzaWRlIGNoYW5uZWwuIFJlcHJlc2VudGF0aW9uIGJ1aWxkZXJzIGRvIG5vdAogICAgIyByZWFkIGl0LCBzbyBkaWFnbm9zdGljIGNhcHR1cmUgY2Fubm90IGVudGVyIHRoZSBtb2RlbCdzIHBvbGljeSBjb250ZXh0LgogICAgZGlhZ25vc3RpY3M6IEFueSB8IE5vbmUgPSBOb25lCiAgICBxdWFyYW50aW5lZDogYm9vbCA9IEZhbHNlCiAgICB0ZXJtaW5hbF9yZWFzb246IHN0ciB8IE5vbmUgPSBOb25lCgogICAgZGVmIF9fcG9zdF9pbml0X18oc2VsZikgLT4gTm9uZToKICAgICAgICBzZWxmLmV2aWRlbmNlLmNhcHR1cmVfdDAoc2VsZi5vYnNlcnZhdGlvbiwgYnVkZ2V0cz1zZWxmLl9saXZlX2J1ZGdldHMoKSkKCiAgICBkZWYgX2xpdmVfYnVkZ2V0cyhzZWxmKSAtPiBNYXBwaW5nW3N0ciwgaW50XToKICAgICAgICBpZiBzZWxmLmFjdGlvbl9idWRnZXRfbGltaXQgaXMgTm9uZToKICAgICAgICAgICAgcmV0dXJuIHt9CiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgImFjdGlvbl9saW1pdCI6IHNlbGYuYWN0aW9uX2J1ZGdldF9saW1pdCwKICAgICAgICAgICAgImNvbnNlcnZhdGl2ZV9zcGVudF9hY3Rpb25zIjogc2VsZi5jb3VudGVycy5jb25zZXJ2YXRpdmVfc3BlbnRfYWN0aW9ucywKICAgICAgICAgICAgImFjdGlvbnNfcmVtYWluaW5nIjogbWF4KAogICAgICAgICAgICAgICAgMCwgc2VsZi5hY3Rpb25fYnVkZ2V0X2xpbWl0IC0gc2VsZi5jb3VudGVycy5jb25zZXJ2YXRpdmVfc3BlbnRfYWN0aW9ucwogICAgICAgICAgICApLAogICAgICAgIH0KCiAgICBkZWYgbWFya19wZW5kaW5nX2FjdGlvbihzZWxmLCB0cmFuc2l0aW9uX2lkOiBzdHIpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5ldmlkZW5jZS5jYXB0dXJlX3QwKAogICAgICAgICAgICBzZWxmLm9ic2VydmF0aW9uLAogICAgICAgICAgICBidWRnZXRzPXNlbGYuX2xpdmVfYnVkZ2V0cygpLAogICAgICAgICAgICBwZW5kaW5nX2FjdGlvbl9yZWZlcmVuY2U9dHJhbnNpdGlvbl9pZCwKICAgICAgICApCgogICAgZGVmIGNsZWFyX3BlbmRpbmdfYWN0aW9uKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5ldmlkZW5jZS5jYXB0dXJlX3QwKHNlbGYub2JzZXJ2YXRpb24sIGJ1ZGdldHM9c2VsZi5fbGl2ZV9idWRnZXRzKCkpCgogICAgZGVmIHJlcGxhY2Vfb2JzZXJ2YXRpb24oCiAgICAgICAgc2VsZiwKICAgICAgICBvYnNlcnZhdGlvbjogT2JzZXJ2YXRpb24sCiAgICAgICAgKiwKICAgICAgICBhY3Rpb25faWQ6IGludCB8IE5vbmUgPSBOb25lLAogICAgICAgIGFjdGlvbl9kYXRhOiBNYXBwaW5nW3N0ciwgQW55XSB8IE5vbmUgPSBOb25lLAogICAgICAgIHRyYW5zaXRpb25faWQ6IHN0ciB8IE5vbmUgPSBOb25lLAogICAgKSAtPiBOb25lOgogICAgICAgIGlmIG9ic2VydmF0aW9uLmd1aWQgIT0gc2VsZi5vYnNlcnZhdGlvbi5ndWlkOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJzZXNzaW9uIGd1aWQgY2hhbmdlZCIpCiAgICAgICAgcHJldmlvdXMgPSBzZWxmLm9ic2VydmF0aW9uCiAgICAgICAgaWYgYWN0aW9uX2lkIGlzIE5vbmU6CiAgICAgICAgICAgIHNlbGYuZXZpZGVuY2UuY2FwdHVyZV90MChvYnNlcnZhdGlvbiwgYnVkZ2V0cz1zZWxmLl9saXZlX2J1ZGdldHMoKSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBzZWxmLmV2aWRlbmNlLnJlY29yZF90cmFuc2l0aW9uKAogICAgICAgICAgICAgICAgcHJldmlvdXMsCiAgICAgICAgICAgICAgICBvYnNlcnZhdGlvbiwKICAgICAgICAgICAgICAgIGFjdGlvbl9pZD1hY3Rpb25faWQsCiAgICAgICAgICAgICAgICBhY3Rpb25fZGF0YT1hY3Rpb25fZGF0YSwKICAgICAgICAgICAgICAgIHRyYW5zaXRpb25faWQ9dHJhbnNpdGlvbl9pZCwKICAgICAgICAgICAgICAgIGJ1ZGdldHM9c2VsZi5fbGl2ZV9idWRnZXRzKCksCiAgICAgICAgICAgICkKICAgICAgICBzZWxmLm9ic2VydmF0aW9uID0gb2JzZXJ2YXRpb24K', 'agent/watchdog.py': 'IiIiSW5kZXBlbmRlbnQgZGVhZGxpbmUgc3RhdGUgc2hhcmVkIGJ5IG9yY2hlc3RyYXRpb24gYW5kIHdvcmtlcnMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdGltZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gdGhyZWFkaW5nIGltcG9ydCBFdmVudCwgVGhyZWFkCgoKQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFzcyBEZWFkbGluZVdhdGNoZG9nOgogICAgaGFyZF9zZWNvbmRzOiBmbG9hdAogICAgZmluYWxpemF0aW9uX3Jlc2VydmVfc2Vjb25kczogZmxvYXQKICAgIHN0YXJ0ZWRfbW9ub3RvbmljOiBmbG9hdCA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT10aW1lLm1vbm90b25pYykKICAgIGNhbmNlbGVkOiBFdmVudCA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1FdmVudCkKICAgIF9zdXBlcnZpc29yX3N0b3A6IEV2ZW50ID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PUV2ZW50KQogICAgX3N1cGVydmlzb3I6IFRocmVhZCB8IE5vbmUgPSBmaWVsZChkZWZhdWx0PU5vbmUsIGluaXQ9RmFsc2UpCgogICAgQHByb3BlcnR5CiAgICBkZWYgZWxhcHNlZChzZWxmKSAtPiBmbG9hdDoKICAgICAgICByZXR1cm4gdGltZS5tb25vdG9uaWMoKSAtIHNlbGYuc3RhcnRlZF9tb25vdG9uaWMKCiAgICBAcHJvcGVydHkKICAgIGRlZiBzdG9wX2FkbWlzc2lvbihzZWxmKSAtPiBib29sOgogICAgICAgIHJldHVybiBzZWxmLmNhbmNlbGVkLmlzX3NldCgpIG9yIHNlbGYuZWxhcHNlZCA+PSBzZWxmLmhhcmRfc2Vjb25kcyAtIHNlbGYuZmluYWxpemF0aW9uX3Jlc2VydmVfc2Vjb25kcwoKICAgIEBwcm9wZXJ0eQogICAgZGVmIGV4cGlyZWQoc2VsZikgLT4gYm9vbDoKICAgICAgICByZXR1cm4gc2VsZi5lbGFwc2VkID49IHNlbGYuaGFyZF9zZWNvbmRzCgogICAgZGVmIGNhbmNlbChzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuY2FuY2VsZWQuc2V0KCkKCiAgICBkZWYgc3RhcnRfc3VwZXJ2aXNvcihzZWxmLCBwb2xsX3NlY29uZHM6IGZsb2F0ID0gMC4yNSkgLT4gTm9uZToKICAgICAgICBpZiBzZWxmLl9zdXBlcnZpc29yIGlzIG5vdCBOb25lOgogICAgICAgICAgICByZXR1cm4KCiAgICAgICAgZGVmIHN1cGVydmlzZSgpIC0+IE5vbmU6CiAgICAgICAgICAgIHdoaWxlIG5vdCBzZWxmLl9zdXBlcnZpc29yX3N0b3Aud2FpdChwb2xsX3NlY29uZHMpOgogICAgICAgICAgICAgICAgaWYgc2VsZi5lbGFwc2VkID49IHNlbGYuaGFyZF9zZWNvbmRzIC0gc2VsZi5maW5hbGl6YXRpb25fcmVzZXJ2ZV9zZWNvbmRzOgogICAgICAgICAgICAgICAgICAgIHNlbGYuY2FuY2VsZWQuc2V0KCkKICAgICAgICAgICAgICAgICAgICByZXR1cm4KCiAgICAgICAgc2VsZi5fc3VwZXJ2aXNvciA9IFRocmVhZCh0YXJnZXQ9c3VwZXJ2aXNlLCBuYW1lPSJhcmMzLWRlYWRsaW5lLXdhdGNoZG9nIiwgZGFlbW9uPVRydWUpCiAgICAgICAgc2VsZi5fc3VwZXJ2aXNvci5zdGFydCgpCgogICAgZGVmIHN0b3Bfc3VwZXJ2aXNvcihzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuX3N1cGVydmlzb3Jfc3RvcC5zZXQoKQogICAgICAgIGlmIHNlbGYuX3N1cGVydmlzb3IgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHNlbGYuX3N1cGVydmlzb3Iuam9pbih0aW1lb3V0PTEuMCkK', 'certification/direct_publisher_smoke_v1/__init__.py': 'IiIiQ1BVLW9ubHkgaW50YWtlIGZvciBhIHByb3Bvc2VkIGRpcmVjdC1wdWJsaXNoZXIgc21va2UgdGVzdDsgbm8gbGl2ZSBydW5uZXIuIiIiCg==', 'certification/direct_publisher_smoke_v1/install.py': 'IiIiT2ZmbGluZSBpbnN0YWxsYXRpb24gdmVyaWZpY2F0aW9uOiBidW5kbGUgaW50ZWdyaXR5IGFnYWluc3QgdGhlIGJvdW5kIGNoZWNrc3VtcywgYSBoYXNoLXBpbm5lZCBvZmZsaW5lIGluc3RhbGwKaW50byBhIGZyZXNoIHZpcnR1YWwgZW52aXJvbm1lbnQsIHRoZW4gcGlwIGNoZWNrLCBleGFjdCB2ZXJzaW9ucywgaW1wb3J0cyBhbmQgdGhlIHRvcmNoIENVREEgYnVpbGQuIiIiCmltcG9ydCBoYXNobGliCmltcG9ydCBjdHlwZXMKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpmcm9tIGNlcnRpZmljYXRpb24uZGlyZWN0X3B1Ymxpc2hlcl9zbW9rZV92MS5zZXJ2ZXIgaW1wb3J0IE1vZGVsU2VydmVyLCBkZWZlcl9zdGFydHVwX3NpZ25hbHMKCgpjbGFzcyBJbnN0YWxsRmFpbGVkKFJ1bnRpbWVFcnJvcik6CiAgICBwYXNzCgoKZGVmIGZpbGVfc2hhMjU2KHBhdGgpOgogICAgZGlnZXN0ID0gaGFzaGxpYi5zaGEyNTYoKQogICAgd2l0aCBQYXRoKHBhdGgpLm9wZW4oJ3JiJykgYXMgc3RyZWFtOgogICAgICAgIGZvciBjaHVuayBpbiBpdGVyKGxhbWJkYTogc3RyZWFtLnJlYWQoOCAqIDEwMjQgKiAxMDI0KSwgYicnKToKICAgICAgICAgICAgZGlnZXN0LnVwZGF0ZShjaHVuaykKICAgIHJldHVybiBkaWdlc3QuaGV4ZGlnZXN0KCkKCgpkZWYgdmVyaWZ5X2J1bmRsZShidW5kbGUsIGRhdGFzZXQsIGJ1bmRsZV9waW5zLCBjaGVjaz1sYW1iZGE6IE5vbmUsICosIGlucHV0cz1Ob25lKToKICAgICIiIlZlcmlmeSB0aGUgZmxhdCBwdWJsaXNoZXIgbW91bnQgYWdhaW5zdCByZXZpZXdlZCBsb2NhbCBpbnB1dHMsIG5vdCBhbiBSMiBtYW5pZmVzdC4iIiIKICAgIGZyb20gY2VydGlmaWNhdGlvbi5kaXJlY3RfcHVibGlzaGVyX3Ntb2tlX3YxLnByZWZsaWdodCBpbXBvcnQgUEFDS0FHRSwgbG9hZF9pbnB1dHMsIHZlcmlmeV9tb3VudGVkCiAgICBpbnB1dHMgPSBQYXRoKGlucHV0cykgaWYgaW5wdXRzIGlzIG5vdCBOb25lIGVsc2UgUEFDS0FHRQogICAgcHJvcG9zYWwsIF8sIF8gPSBsb2FkX2lucHV0cyhpbnB1dHMpCiAgICBleHBlY3RlZCA9IHsncmVmJzogcHJvcG9zYWxbJ2RhdGFzZXQnXVsncmVmJ10sICd2ZXJzaW9uJzogcHJvcG9zYWxbJ2RhdGFzZXQnXVsndmVyc2lvbiddLAogICAgICAgICAgICAgICAgJ3B1Ymxpc2hlcl9tZXRhZGF0YV9zaGEyNTYnOiBwcm9wb3NhbFsncHVibGlzaGVyX21ldGFkYXRhX3NoYTI1NiddfQogICAgaWYgZGF0YXNldCAhPSBleHBlY3RlZDoKICAgICAgICByYWlzZSBJbnN0YWxsRmFpbGVkKCdkYXRhc2V0IGRpZmZlcnMgZnJvbSB0aGUgdHJ1c3RlZCBpbnRha2UgYmluZGluZycpCiAgICBleHBlY3RlZF9waW5zID0geydhcHByb3ZlZF9tYW5pZmVzdF9zaGEyNTYnOiBwcm9wb3NhbFsndHJ1c3RlZF9hcnRpZmFjdHNfc2hhMjU2J10sCiAgICAgICAgICAgICAgICAgICAgICdyZXF1aXJlbWVudHNfbG9ja19zaGEyNTYnOiBwcm9wb3NhbFsndHJ1c3RlZF9yZXF1aXJlbWVudHNfc2hhMjU2J10sCiAgICAgICAgICAgICAgICAgICAgICd3aGVlbF9jb3VudCc6IHByb3Bvc2FsWyd3aGVlbF9jb3VudCddfQogICAgaWYgYnVuZGxlX3BpbnMgIT0gZXhwZWN0ZWRfcGluczoKICAgICAgICByYWlzZSBJbnN0YWxsRmFpbGVkKCd0cnVzdGVkIGluc3RhbGxhdGlvbiBwaW5zIGRpZmZlciBmcm9tIHRoZSBwcm90b2NvbCcpCgogICAgZGVmIGNoZWNrZWRfbm93KCk6CiAgICAgICAgY2hlY2soKSAgIyBjb21iaW5lZCBpbnN0YWxsYXRpb24gZGVhZGxpbmUgY292ZXJzIGV2ZXJ5IHN0cmVhbWVkIGhhc2ggY2h1bmsKICAgICAgICByZXR1cm4gdGltZS5tb25vdG9uaWMoKQogICAgcmV0dXJuIHZlcmlmeV9tb3VudGVkKGJ1bmRsZSwgcGFja2FnZT1pbnB1dHMsIG5vdz1jaGVja2VkX25vdykKCgpjbGFzcyBJbnN0YWxsYXRpb25Qcm9jZXNzZXM6CiAgICAiIiJSZXRhaW4gb3duZXJzaGlwIGV2ZW4gd2hlbiBjb21tYW5kIHN0YXJ0dXAgbmV2ZXIgcmV0dXJucyB0byBpdHMgY2FsbGVyLgoKICAgIExpbnV4IHN1YnJlYXBlciBtb2RlIGxldHMgdGhpcyBub3RlYm9vayByZWFwIG9ycGhhbmVkIGdyYW5kY2hpbGRyZW4gaXRzZWxmOwogICAgd2FpdHBpZCBpcyBzY29wZWQgdG8gZWFjaCBvd25lZCBncm91cCwgbmV2ZXIgdG8gdW5yZWxhdGVkIGNoaWxkIHByb2Nlc3Nlcy4KICAgIFRoZSBwcmV2aW91cyBzdWJyZWFwZXIgc2V0dGluZyBpcyByZXN0b3JlZCBhZnRlciBhbGwgbGlmZWN5Y2xlIGdyb3VwcyBzdG9wLgogICAgIiIiCiAgICBkZWYgX19pbml0X18oc2VsZiwgdGVybWluYXRlX2dyYWNlPTEsIGtpbGxfZ3JhY2U9Mik6CiAgICAgICAgc2VsZi5ncm91cHMgPSBbXQogICAgICAgIHNlbGYudGVybWluYXRlX2dyYWNlLCBzZWxmLmtpbGxfZ3JhY2UgPSB0ZXJtaW5hdGVfZ3JhY2UsIGtpbGxfZ3JhY2UKICAgICAgICBzZWxmLnByZXZpb3VzX3N1YnJlYXBlciA9IE5vbmUKCiAgICBkZWYgX2Fkb3B0X2Rlc2NlbmRhbnRzKHNlbGYpOgogICAgICAgIGlmIHNlbGYucHJldmlvdXNfc3VicmVhcGVyIGlzIG5vdCBOb25lOgogICAgICAgICAgICByZXR1cm4KICAgICAgICBpZiBzeXMucGxhdGZvcm0gIT0gJ2xpbnV4JzoKICAgICAgICAgICAgcmFpc2UgSW5zdGFsbEZhaWxlZCgnaW5zdGFsbGF0aW9uIHByb2Nlc3Mgb3duZXJzaGlwIHJlcXVpcmVzIExpbnV4JykKICAgICAgICBsaWJjID0gY3R5cGVzLkNETEwoTm9uZSwgdXNlX2Vycm5vPVRydWUpCiAgICAgICAgcHJldmlvdXMgPSBjdHlwZXMuY19pbnQoKQogICAgICAgIGlmIGxpYmMucHJjdGwoMzcsIGN0eXBlcy5ieXJlZihwcmV2aW91cyksIDAsIDAsIDApICE9IDA6ICAjIFBSX0dFVF9DSElMRF9TVUJSRUFQRVIKICAgICAgICAgICAgcmFpc2UgT1NFcnJvcihjdHlwZXMuZ2V0X2Vycm5vKCksICdjYW5ub3QgcmVhZCBjaGlsZCBzdWJyZWFwZXIgc2V0dGluZycpCiAgICAgICAgc2VsZi5wcmV2aW91c19zdWJyZWFwZXIgPSBwcmV2aW91cy52YWx1ZQogICAgICAgIGlmIGxpYmMucHJjdGwoMzYsIDEsIDAsIDAsIDApICE9IDA6ICAjIFBSX1NFVF9DSElMRF9TVUJSRUFQRVIKICAgICAgICAgICAgcmFpc2UgT1NFcnJvcihjdHlwZXMuZ2V0X2Vycm5vKCksICdjYW5ub3QgYWRvcHQgaW5zdGFsbGF0aW9uIGRlc2NlbmRhbnRzJykKCiAgICBkZWYgX3N0b3BfZ3JvdXAoc2VsZiwgZW50cnksIGRlYWRsaW5lKToKICAgICAgICBvbGQgPSBlbnRyeS5nZXQoJ2NsZWFudXAnLCB7fSkKICAgICAgICBpZiBvbGQuZ2V0KCdncm91cHNfYWJzZW50Jyk6CiAgICAgICAgICAgICMgVmVyaWZpZWQgYWJzZW5jZSBpcyB0ZXJtaW5hbCBmb3IgdGhpcyBvd25lci4gQSBsYXRlciByZXVzZSBvZiBpdHMKICAgICAgICAgICAgIyBudW1lcmljIFBHSUQgbXVzdCBub3QgbGV0IGZpbmFsIGNsZWFudXAgc2lnbmFsIGFuIHVucmVsYXRlZCBncm91cC4KICAgICAgICAgICAgcmV0dXJuIGRpY3Qob2xkKQogICAgICAgIGludGVycnVwdGlvbnMgPSBbXQogICAgICAgIGZvciBfIGluIHJhbmdlKDIpOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICByZWNlaXB0ID0gZW50cnlbJ2dyb3VwJ10uc3RvcChkZWFkbGluZSwgc2VsZi50ZXJtaW5hdGVfZ3JhY2UsIHNlbGYua2lsbF9ncmFjZSkKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIGV4Y2VwdCBCYXNlRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICAgICAgICAgIGludGVycnVwdGlvbnMuYXBwZW5kKGYnY2xlYW51cDoge3R5cGUoZXhjKS5fX25hbWVfX306IHtzdHIoZXhjKVs6MTIwXX0nKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHJlY2VpcHQgPSB7J293bmVyc2hpcCc6IGVudHJ5Wydncm91cCddLm93bmVyc2hpcCwgJ3BnaWQnOiBlbnRyeVsnZ3JvdXAnXS5wZ2lkLAogICAgICAgICAgICAgICAgICAgICAgICdncm91cHNfYWJzZW50JzogRmFsc2UsICdlcnJvcic6ICdwcm9jZXNzIGNsZWFudXAgaW50ZXJydXB0ZWQgdHdpY2UnfQogICAgICAgIGZvciBrZXkgaW4gKCdzaWd0ZXJtX3NlbnQnLCAnc2lna2lsbF9zZW50Jyk6CiAgICAgICAgICAgIHJlY2VpcHRba2V5XSA9IHJlY2VpcHQuZ2V0KGtleSwgRmFsc2UpIG9yIG9sZC5nZXQoa2V5LCBGYWxzZSkKICAgICAgICByZWNlaXB0WydpbnRlcnJ1cHRlZCddID0gb2xkLmdldCgnaW50ZXJydXB0ZWQnLCBbXSkgKyBpbnRlcnJ1cHRpb25zICsgcmVjZWlwdC5nZXQoJ2ludGVycnVwdGVkJywgW10pCiAgICAgICAgZW50cnlbJ2NsZWFudXAnXSA9IHJlY2VpcHQKICAgICAgICByZXR1cm4gcmVjZWlwdAoKICAgIGRlZiBjb21tYW5kKHNlbGYsIGFyZ3YsIGxvZywgZGVhZGxpbmUsIGVudiwgd2hhdCk6CiAgICAgICAgaWYgZGVhZGxpbmUgPD0gdGltZS5tb25vdG9uaWMoKToKICAgICAgICAgICAgcmFpc2UgSW5zdGFsbEZhaWxlZChmJ3t3aGF0fTogaW5zdGFsbGF0aW9uIGRlYWRsaW5lIHJlYWNoZWQnKQogICAgICAgIHdpdGggZGVmZXJfc3RhcnR1cF9zaWduYWxzKCk6CiAgICAgICAgICAgIHNlbGYuX2Fkb3B0X2Rlc2NlbmRhbnRzKCkKICAgICAgICAgICAgZ3JvdXAgPSBNb2RlbFNlcnZlcihhcmd2LCBlbnYsIGxvZywgTm9uZSwgTm9uZSwgaW5oZXJpdF9lbnY9RmFsc2UpCiAgICAgICAgICAgIGVudHJ5ID0geydwaGFzZSc6IHdoYXQsICdncm91cCc6IGdyb3VwfQogICAgICAgICAgICBzZWxmLmdyb3Vwcy5hcHBlbmQoZW50cnkpICAjIHJlZ2lzdGVyIHRoZSBvd25lciBiZWZvcmUgZW50ZXJpbmcgUG9wZW4KICAgICAgICB3aXRoIFBhdGgobG9nKS5vcGVuKCdhYicpIGFzIHN0cmVhbToKICAgICAgICAgICAgc3RyZWFtLndyaXRlKGYnXG4kIHsiICIuam9pbihtYXAoc3RyLCBhcmd2KSl9XG4nLmVuY29kZSgpKQogICAgICAgICAgICBzdHJlYW0uZmx1c2goKQogICAgICAgICAgICBvZmZzZXQgPSBzdHJlYW0udGVsbCgpCiAgICAgICAgdHJ5OgogICAgICAgICAgICBncm91cC5zdGFydCgpCiAgICAgICAgICAgIHJlbWFpbmluZyA9IGRlYWRsaW5lIC0gdGltZS5tb25vdG9uaWMoKQogICAgICAgICAgICBpZiByZW1haW5pbmcgPD0gMDoKICAgICAgICAgICAgICAgIHJhaXNlIEluc3RhbGxGYWlsZWQoZid7d2hhdH06IGluc3RhbGxhdGlvbiBkZWFkbGluZSByZWFjaGVkJykKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgY29kZSA9IGdyb3VwLnByb2Nlc3Mud2FpdCh0aW1lb3V0PXJlbWFpbmluZykKICAgICAgICAgICAgZXhjZXB0IHN1YnByb2Nlc3MuVGltZW91dEV4cGlyZWQgYXMgZXhjOgogICAgICAgICAgICAgICAgcmFpc2UgSW5zdGFsbEZhaWxlZChmJ3t3aGF0fTogaW5zdGFsbGF0aW9uIGRlYWRsaW5lIHJlYWNoZWQnKSBmcm9tIGV4YwogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgICMgRXZlbiBLZXlib2FyZEludGVycnVwdCBvciBhbiBhbGFybSBiZWZvcmUgYXNzaWdubWVudCBtdXN0IHJlYWNoIHRoaXMuCiAgICAgICAgICAgIGNsZWFudXAgPSBzZWxmLl9zdG9wX2dyb3VwKGVudHJ5LCBkZWFkbGluZSkKICAgICAgICBpZiBub3QgY2xlYW51cFsnZ3JvdXBzX2Fic2VudCddIG9yIGNsZWFudXAuZ2V0KCdpbnRlcnJ1cHRlZCcpOgogICAgICAgICAgICByYWlzZSBJbnN0YWxsRmFpbGVkKGYne3doYXR9OiBwcm9jZXNzLWdyb3VwIGNsZWFudXAgbm90IHZlcmlmaWVkIHdpdGhvdXQgaW50ZXJydXB0aW9uJykKICAgICAgICBpZiBkZWFkbGluZSA8PSB0aW1lLm1vbm90b25pYygpOgogICAgICAgICAgICByYWlzZSBJbnN0YWxsRmFpbGVkKGYne3doYXR9OiBpbnN0YWxsYXRpb24gZGVhZGxpbmUgcmVhY2hlZCBkdXJpbmcgcHJvY2VzcyBjbGVhbnVwJykKICAgICAgICAjIEEgZmlsZSBhdm9pZHMgcGlwZSBFT0YgaGFuZ3Mgd2hlbiBpbXBvcnRlZCBjb2RlIGxlYXZlcyBhIGNoaWxkIGJlaGluZC4KICAgICAgICB3aXRoIFBhdGgobG9nKS5vcGVuKCdyYicpIGFzIHN0cmVhbToKICAgICAgICAgICAgc3RyZWFtLnNlZWsoMCwgb3MuU0VFS19FTkQpCiAgICAgICAgICAgIHN0cmVhbS5zZWVrKG1heChvZmZzZXQsIHN0cmVhbS50ZWxsKCkgLSAyICogMTAyNCAqIDEwMjQpKQogICAgICAgICAgICBvdXRwdXQgPSBzdHJlYW0ucmVhZCgpLmRlY29kZSgndXRmLTgnLCBlcnJvcnM9J3JlcGxhY2UnKQogICAgICAgIGlmIGNvZGU6CiAgICAgICAgICAgIHJhaXNlIEluc3RhbGxGYWlsZWQoZid7d2hhdH0gZXhpdGVkIHtjb2RlfToge291dHB1dC5zdHJpcCgpWy0zMDA6XX0nKQogICAgICAgIHJldHVybiBzdWJwcm9jZXNzLkNvbXBsZXRlZFByb2Nlc3MoYXJndiwgY29kZSwgc3Rkb3V0PW91dHB1dCwgc3RkZXJyPScnKQoKICAgIGRlZiBzdG9wKHNlbGYsIGRlYWRsaW5lKToKICAgICAgICByZWNlaXB0cyA9IFt7J3BoYXNlJzogZW50cnlbJ3BoYXNlJ10sICoqc2VsZi5fc3RvcF9ncm91cChlbnRyeSwgZGVhZGxpbmUpfSBmb3IgZW50cnkgaW4gc2VsZi5ncm91cHNdCiAgICAgICAgZXJyb3IgPSBOb25lCiAgICAgICAgaWYgc2VsZi5wcmV2aW91c19zdWJyZWFwZXIgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGxpYmMgPSBjdHlwZXMuQ0RMTChOb25lLCB1c2VfZXJybm89VHJ1ZSkKICAgICAgICAgICAgaWYgbGliYy5wcmN0bCgzNiwgc2VsZi5wcmV2aW91c19zdWJyZWFwZXIsIDAsIDAsIDApICE9IDA6CiAgICAgICAgICAgICAgICBlcnJvciA9ICdjb3VsZCBub3QgcmVzdG9yZSBjaGlsZCBzdWJyZWFwZXIgc2V0dGluZycKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHNlbGYucHJldmlvdXNfc3VicmVhcGVyID0gTm9uZQogICAgICAgIHJldHVybiB7J2dyb3Vwcyc6IHJlY2VpcHRzLCAnZ3JvdXBzX2Fic2VudCc6IGFsbChyWydncm91cHNfYWJzZW50J10gZm9yIHIgaW4gcmVjZWlwdHMpLAogICAgICAgICAgICAgICAgJ2ludGVycnVwdGVkJzogW21lc3NhZ2UgZm9yIHIgaW4gcmVjZWlwdHMgZm9yIG1lc3NhZ2UgaW4gci5nZXQoJ2ludGVycnVwdGVkJywgW10pXSwKICAgICAgICAgICAgICAgICdlcnJvcic6IGVycm9yfQoKCmRlZiBfcnVuKGFyZ3YsIGxvZywgZGVhZGxpbmUsIGVudiwgd2hhdCwgcHJvY2Vzc2VzPU5vbmUpOgogICAgb3duZWQgPSBwcm9jZXNzZXMgaXMgTm9uZQogICAgcHJvY2Vzc2VzID0gcHJvY2Vzc2VzIGlmIHByb2Nlc3NlcyBpcyBub3QgTm9uZSBlbHNlIEluc3RhbGxhdGlvblByb2Nlc3NlcygpCiAgICB0cnk6CiAgICAgICAgcmV0dXJuIHByb2Nlc3Nlcy5jb21tYW5kKGFyZ3YsIGxvZywgZGVhZGxpbmUsIGVudiwgd2hhdCkKICAgIGZpbmFsbHk6CiAgICAgICAgaWYgb3duZWQ6CiAgICAgICAgICAgIGNsZWFudXAgPSBwcm9jZXNzZXMuc3RvcChkZWFkbGluZSkKICAgICAgICAgICAgaWYgc3lzLmV4Y19pbmZvKClbMF0gaXMgTm9uZSBhbmQgKG5vdCBjbGVhbnVwWydncm91cHNfYWJzZW50J10gb3IgY2xlYW51cFsnZXJyb3InXSk6CiAgICAgICAgICAgICAgICByYWlzZSBJbnN0YWxsRmFpbGVkKCdpbnN0YWxsYXRpb24gcHJvY2VzcyBjbGVhbnVwIG5vdCB2ZXJpZmllZCcpCgoKUEFDS0FHRV9DSEVDSyA9ICcnJ2ltcG9ydCBpbXBvcnRsaWIsIGltcG9ydGxpYi5tZXRhZGF0YSBhcyBtLCBqc29uLCBzeXMKZXhwZWN0ZWQgPSBqc29uLmxvYWRzKHN5cy5hcmd2WzFdKQpyZXBvcnQgPSB7InZlcnNpb25zIjoge30sICJpbXBvcnRzIjoge319CmZvciBuYW1lIGluIGV4cGVjdGVkWyJwYWNrYWdlcyJdOgogICAgcmVwb3J0WyJ2ZXJzaW9ucyJdW25hbWVdID0gbS52ZXJzaW9uKG5hbWUpCmZvciBuYW1lIGluIGV4cGVjdGVkWyJpbXBvcnRzIl06CiAgICBtb2R1bGUgPSBpbXBvcnRsaWIuaW1wb3J0X21vZHVsZShuYW1lKQogICAgcmVwb3J0WyJpbXBvcnRzIl1bbmFtZV0gPSBnZXRhdHRyKG1vZHVsZSwgIl9fZmlsZV9fIiwgTm9uZSkKaWYgInRvcmNoIiBpbiBleHBlY3RlZFsiaW1wb3J0cyJdOgogICAgaW1wb3J0IHRvcmNoCiAgICByZXBvcnRbInRvcmNoX2N1ZGFfYnVpbGQiXSA9IHRvcmNoLnZlcnNpb24uY3VkYQpwcmludChqc29uLmR1bXBzKHJlcG9ydCkpCicnJwoKCmRlZiBpbnN0YWxsKGJ1bmRsZSwgdmVudiwgcnVudGltZSwgZGVhZGxpbmUsIGxvZywgcHl0aG9uPXN5cy5leGVjdXRhYmxlLCBlbnZpcm9ubWVudD1Ob25lLCByZXF1aXJlbWVudHM9Tm9uZSwKICAgICAgICAgICAgcHJvY2Vzc2VzPU5vbmUpOgogICAgIiIiSGFzaC1waW5uZWQgb2ZmbGluZSBpbnN0YWxsIGludG8gYSBuZXcgdmlydHVhbCBlbnZpcm9ubWVudCwgdGhlbiBwaXAgY2hlY2ssIHZlcnNpb25zLCBpbXBvcnRzIGFuZCB0aGUgdG9yY2gKICAgIENVREEgYnVpbGQuIE5vIGluZGV4LCBubyBjYWNoZSwgYmluYXJ5IHdoZWVscyBvbmx5LCBubyBuZXR3b3JrIGNvbmZpZ3VyYXRpb24uIiIiCiAgICBidW5kbGUsIHZlbnYgPSBQYXRoKGJ1bmRsZSksIFBhdGgodmVudikKICAgIGVudiA9IHtrOiB2IGZvciBrLCB2IGluIChlbnZpcm9ubWVudCBvciBvcy5lbnZpcm9uKS5pdGVtcygpCiAgICAgICAgICAgaWYgayBub3QgaW4gKCdQWVRIT05QQVRIJywgJ1BZVEhPTkhPTUUnLCAnVklSVFVBTF9FTlYnLCAnUElQX0lOREVYX1VSTCcsICdQSVBfRVhUUkFfSU5ERVhfVVJMJyl9CiAgICBlbnYudXBkYXRlKFBJUF9OT19JTkRFWD0nMScsIFBJUF9DT05GSUdfRklMRT1vcy5kZXZudWxsLCBQSVBfRElTQUJMRV9QSVBfVkVSU0lPTl9DSEVDSz0nMScsCiAgICAgICAgICAgICAgIFBZVEhPTk5PVVNFUlNJVEU9JzEnLCBDVURBX1ZJU0lCTEVfREVWSUNFUz1lbnYuZ2V0KCdDVURBX1ZJU0lCTEVfREVWSUNFUycsICcnKSkKICAgIG93bmVkID0gcHJvY2Vzc2VzIGlzIE5vbmUKICAgIHByb2Nlc3NlcyA9IHByb2Nlc3NlcyBpZiBwcm9jZXNzZXMgaXMgbm90IE5vbmUgZWxzZSBJbnN0YWxsYXRpb25Qcm9jZXNzZXMoKQogICAgdHJ5OgogICAgICAgIHJldHVybiBfaW5zdGFsbChidW5kbGUsIHZlbnYsIHJ1bnRpbWUsIGRlYWRsaW5lLCBsb2csIHB5dGhvbiwgZW52LCByZXF1aXJlbWVudHMsIHByb2Nlc3NlcykKICAgIGZpbmFsbHk6CiAgICAgICAgaWYgb3duZWQ6CiAgICAgICAgICAgIGNsZWFudXAgPSBwcm9jZXNzZXMuc3RvcChkZWFkbGluZSkKICAgICAgICAgICAgaWYgc3lzLmV4Y19pbmZvKClbMF0gaXMgTm9uZSBhbmQgKG5vdCBjbGVhbnVwWydncm91cHNfYWJzZW50J10gb3IgY2xlYW51cFsnZXJyb3InXSk6CiAgICAgICAgICAgICAgICByYWlzZSBJbnN0YWxsRmFpbGVkKCdpbnN0YWxsYXRpb24gcHJvY2VzcyBjbGVhbnVwIG5vdCB2ZXJpZmllZCcpCgoKZGVmIF9pbnN0YWxsKGJ1bmRsZSwgdmVudiwgcnVudGltZSwgZGVhZGxpbmUsIGxvZywgcHl0aG9uLCBlbnYsIHJlcXVpcmVtZW50cywgcHJvY2Vzc2VzKToKICAgIHJlY2VpcHQgPSB7J3Bhc3NlZCc6IEZhbHNlfQogICAgYmVnaW4gPSB0aW1lLm1vbm90b25pYygpCiAgICAjIFNvbWUgcnVudGltZSBpbWFnZXMgY2Fubm90IHNlZWQgcGlwIHRocm91Z2ggZW5zdXJlcGlwLiBLZWVwIHRoZSB2ZW52CiAgICAjIGlzb2xhdGVkIGFuZCB1c2UgdGhlIHBpbm5lZCBob3N0IGltYWdlJ3MgcGlwIHRvIG1hbmFnZSB0aGF0IGludGVycHJldGVyLgogICAgIyBwaXAgLS1weXRob24gc3VwcG9ydHMgZW52aXJvbm1lbnRzIHdpdGggbm8gcGlwIGluc3RhbGxlZCAocGlwID49IDIyLjMpLgogICAgX3J1bihbcHl0aG9uLCAnLW0nLCAndmVudicsIHN0cih2ZW52KSwgJy0td2l0aG91dC1waXAnXSwgbG9nLCBkZWFkbGluZSwgZW52LCAndmVudiBjcmVhdGlvbicsIHByb2Nlc3NlcykKICAgIHZweXRob24gPSBzdHIodmVudiAvICdiaW4nIC8gJ3B5dGhvbicpCiAgICBwaXAgPSBbcHl0aG9uLCAnLW0nLCAncGlwJywgJy0tcHl0aG9uJywgdnB5dGhvbl0KICAgIF9ydW4oWypwaXAsICdpbnN0YWxsJywgJy0tbm8taW5kZXgnLCAnLS1uby1jYWNoZS1kaXInLCAnLS1yZXF1aXJlLWhhc2hlcycsICctLW9ubHktYmluYXJ5PTphbGw6JywKICAgICAgICAgICctLWZpbmQtbGlua3MnLCBzdHIoYnVuZGxlKSwgJy1yJywgc3RyKHJlcXVpcmVtZW50cyBvciBQYXRoKF9fZmlsZV9fKS53aXRoX25hbWUoJ3RydXN0ZWRfcmVxdWlyZW1lbnRzLmxvY2snKSldLAogICAgICAgICBsb2csIGRlYWRsaW5lLCBlbnYsICdvZmZsaW5lIGluc3RhbGwnLCBwcm9jZXNzZXMpCiAgICByZWNlaXB0WydpbnN0YWxsX3NlY29uZHMnXSA9IHJvdW5kKHRpbWUubW9ub3RvbmljKCkgLSBiZWdpbiwgMykKICAgIF9ydW4oWypwaXAsICdjaGVjayddLCBsb2csIGRlYWRsaW5lLCBlbnYsICdwaXAgY2hlY2snLCBwcm9jZXNzZXMpCiAgICBjaGVja2VkID0gX3J1bihbdnB5dGhvbiwgJy1JJywgJy1jJywgUEFDS0FHRV9DSEVDSywganNvbi5kdW1wcyhydW50aW1lKV0sIGxvZywgZGVhZGxpbmUsIGVudiwKICAgICAgICAgICAgICAgICAgICdwYWNrYWdlIGNoZWNrcycsIHByb2Nlc3NlcykKICAgIHJlcG9ydCA9IGpzb24ubG9hZHMoY2hlY2tlZC5zdGRvdXQuc3RyaXAoKS5zcGxpdGxpbmVzKClbLTFdKQogICAgcmVjZWlwdC51cGRhdGUocmVwb3J0KQogICAgbWlzbWF0Y2hlZCA9IHtuOiB2IGZvciBuLCB2IGluIHJlcG9ydFsndmVyc2lvbnMnXS5pdGVtcygpIGlmIHYgIT0gcnVudGltZVsncGFja2FnZXMnXVtuXX0KICAgIGlmIG1pc21hdGNoZWQ6CiAgICAgICAgcmFpc2UgSW5zdGFsbEZhaWxlZChmJ3ZlcnNpb24gbWlzbWF0Y2ggKG5vIHVwZ3JhZGUgYXR0ZW1wdGVkKToge21pc21hdGNoZWR9JykKICAgIGlmICd0b3JjaF9jdWRhX2J1aWxkJyBpbiBydW50aW1lIGFuZCByZXBvcnQuZ2V0KCd0b3JjaF9jdWRhX2J1aWxkJykgIT0gcnVudGltZVsndG9yY2hfY3VkYV9idWlsZCddOgogICAgICAgIHJhaXNlIEluc3RhbGxGYWlsZWQoZiJ0b3JjaCBDVURBIGJ1aWxkIHtyZXBvcnQuZ2V0KCd0b3JjaF9jdWRhX2J1aWxkJyl9ICE9IHtydW50aW1lWyd0b3JjaF9jdWRhX2J1aWxkJ119IikKICAgIGlmIHRpbWUubW9ub3RvbmljKCkgPj0gZGVhZGxpbmU6CiAgICAgICAgcmFpc2UgSW5zdGFsbEZhaWxlZCgncGFja2FnZSBjaGVja3M6IGluc3RhbGxhdGlvbiBkZWFkbGluZSByZWFjaGVkJykKICAgIHJlY2VpcHQudXBkYXRlKHBhc3NlZD1UcnVlLCBweXRob249dnB5dGhvbikKICAgIHJldHVybiByZWNlaXB0Cg==', 'certification/direct_publisher_smoke_v1/preflight.py': 'IiIiVmVyaWZ5IG1vdW50ZWQgYnl0ZXMgd2l0aG91dCBpbnN0YWxsaW5nIHBhY2thZ2VzLCBpbXBvcnRpbmcgdGhlbSBvciB1c2luZyBhIEdQVS4KClRoaXMgY2hlY2tzIGludGVncml0eSBvbmx5LiBJdCBjYW5ub3QgZXN0YWJsaXNoIGFjY291bnQgYXR0YWNobWVudCwgbGljZW5jZQpwZXJtaXNzaW9uLCBydW50aW1lIGNvbXBhdGliaWxpdHkgb3IgY29tcHV0ZSBhdXRob3JpemF0aW9uLgoiIiIKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCByZQppbXBvcnQgdGltZQoKUEFDS0FHRSA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKUElOID0gcmUuY29tcGlsZShyJyhbQS1aYS16MC05Xy4tXSspPT0oW15cc10rKSg/OiAtLWhhc2g9c2hhMjU2OihbYS1mMC05XXs2NH0pKT8nKQoKCmNsYXNzIFByZWZsaWdodEZhaWxlZChWYWx1ZUVycm9yKToKICAgIHBhc3MKCgpkZWYgbm9ybWFsaXplKG5hbWUpOgogICAgcmV0dXJuIHJlLnN1YihyJ1stXy5dKycsICctJywgbmFtZSkubG93ZXIoKQoKCmRlZiBwaW5zKHRleHQsIHJlcXVpcmVfaGFzaGVzKToKICAgIHJvd3MgPSB7fQogICAgZm9yIGxpbmUgaW4gdGV4dC5zcGxpdGxpbmVzKCk6CiAgICAgICAgaWYgbm90IGxpbmUuc3RyaXAoKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBtYXRjaCA9IFBJTi5mdWxsbWF0Y2gobGluZSkKICAgICAgICBpZiBub3QgbWF0Y2ggb3IgKHJlcXVpcmVfaGFzaGVzIGFuZCBub3QgbWF0Y2hbM10pOgogICAgICAgICAgICByYWlzZSBQcmVmbGlnaHRGYWlsZWQoJ3JlcXVpcmVtZW50cyBsaW5lIGlzIG5vdCBhbiBleGFjdCBwZXJtaXR0ZWQgcGluJykKICAgICAgICBuYW1lLCB2ZXJzaW9uLCBkaWdlc3QgPSBtYXRjaC5ncm91cHMoKQogICAgICAgIG5hbWUgPSBub3JtYWxpemUobmFtZSkKICAgICAgICBpZiBuYW1lIGluIHJvd3M6CiAgICAgICAgICAgIHJhaXNlIFByZWZsaWdodEZhaWxlZCgnZHVwbGljYXRlIHJlcXVpcmVtZW50OiAnICsgbmFtZSkKICAgICAgICByb3dzW25hbWVdID0gKHZlcnNpb24sIGRpZ2VzdCkKICAgIHJldHVybiByb3dzCgoKZGVmIGRpZ2VzdF9maWxlKHBhdGgsIGNoZWNrKToKICAgIGRpZ2VzdCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggcGF0aC5vcGVuKCdyYicpIGFzIHN0cmVhbToKICAgICAgICB3aGlsZSBUcnVlOgogICAgICAgICAgICBjaGVjaygpCiAgICAgICAgICAgIGNodW5rID0gc3RyZWFtLnJlYWQoOCAqIDEwMjQgKiAxMDI0KQogICAgICAgICAgICBjaGVjaygpCiAgICAgICAgICAgIGlmIG5vdCBjaHVuazoKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIGRpZ2VzdC51cGRhdGUoY2h1bmspCiAgICByZXR1cm4gZGlnZXN0LmhleGRpZ2VzdCgpCgoKZGVmIGxvYWRfaW5wdXRzKHBhY2thZ2U9UEFDS0FHRSk6CiAgICBwYWNrYWdlID0gUGF0aChwYWNrYWdlKQogICAgcHJvcG9zYWwgPSBqc29uLmxvYWRzKChwYWNrYWdlIC8gJ3Byb3Bvc2FsLmpzb24nKS5yZWFkX2J5dGVzKCkpCiAgICBtYW5pZmVzdCA9IGpzb24ubG9hZHMoKHBhY2thZ2UgLyAndHJ1c3RlZF9tYW5pZmVzdC5qc29uJykucmVhZF9ieXRlcygpKQogICAgbG9jayA9IChwYWNrYWdlIC8gJ3RydXN0ZWRfcmVxdWlyZW1lbnRzLmxvY2snKS5yZWFkX2J5dGVzKCkKICAgIGNhbm9uaWNhbCA9IGpzb24uZHVtcHMobWFuaWZlc3RbJ2FydGlmYWN0cyddLCBzb3J0X2tleXM9VHJ1ZSwgc2VwYXJhdG9ycz0oJywnLCAnOicpKS5lbmNvZGUoKQogICAgaWYgaGFzaGxpYi5zaGEyNTYoY2Fub25pY2FsKS5oZXhkaWdlc3QoKSAhPSBwcm9wb3NhbFsndHJ1c3RlZF9hcnRpZmFjdHNfc2hhMjU2J106CiAgICAgICAgcmFpc2UgUHJlZmxpZ2h0RmFpbGVkKCd0cnVzdGVkIGFydGlmYWN0IGludmVudG9yeSBkaWZmZXJzIGZyb20gaXRzIGJpbmRpbmcnKQogICAgaWYgaGFzaGxpYi5zaGEyNTYobG9jaykuaGV4ZGlnZXN0KCkgIT0gcHJvcG9zYWxbJ3RydXN0ZWRfcmVxdWlyZW1lbnRzX3NoYTI1NiddOgogICAgICAgIHJhaXNlIFByZWZsaWdodEZhaWxlZCgndHJ1c3RlZCBoYXNoLXBpbm5lZCByZXF1aXJlbWVudHMgZGlmZmVyIGZyb20gdGhlaXIgYmluZGluZycpCiAgICByb3dzID0gbWFuaWZlc3RbJ2FydGlmYWN0cyddCiAgICBuYW1lcyA9IFtyb3dbJ2ZpbGVuYW1lJ10gZm9yIHJvdyBpbiByb3dzXQogICAgaWYgbGVuKHJvd3MpICE9IHByb3Bvc2FsWyd3aGVlbF9jb3VudCddIG9yIGxlbihzZXQobmFtZXMpKSAhPSBsZW4ocm93cyk6CiAgICAgICAgcmFpc2UgUHJlZmxpZ2h0RmFpbGVkKCd3cm9uZyB0cnVzdGVkIHdoZWVsIGNvdW50IG9yIGR1cGxpY2F0ZSBmaWxlbmFtZScpCiAgICBmb3IgbmFtZSBpbiBuYW1lczoKICAgICAgICBpZiBub3QgbmFtZS5lbmRzd2l0aCgnLndobCcpIG9yIFBhdGgobmFtZSkubmFtZSAhPSBuYW1lIG9yICcvJyBpbiBuYW1lIG9yICdcXCcgaW4gbmFtZToKICAgICAgICAgICAgcmFpc2UgUHJlZmxpZ2h0RmFpbGVkKCd1bnNhZmUgdHJ1c3RlZCBmaWxlbmFtZScpCiAgICBsb2NrX3BpbnMgPSBwaW5zKGxvY2suZGVjb2RlKCd1dGYtOCcpLCByZXF1aXJlX2hhc2hlcz1UcnVlKQogICAgZXhwZWN0ZWRfcGlucyA9IHt9CiAgICBmb3Igcm93IGluIHJvd3M6CiAgICAgICAgbmFtZSwgdmVyc2lvbiA9IHJvd1snZmlsZW5hbWUnXS5zcGxpdCgnLScpWzoyXQogICAgICAgIG5hbWUgPSBub3JtYWxpemUobmFtZSkKICAgICAgICBpZiBuYW1lIGluIGV4cGVjdGVkX3BpbnM6CiAgICAgICAgICAgIHJhaXNlIFByZWZsaWdodEZhaWxlZCgnbXVsdGlwbGUgdHJ1c3RlZCB3aGVlbHMgZm9yIG9uZSBkaXN0cmlidXRpb24nKQogICAgICAgIGV4cGVjdGVkX3BpbnNbbmFtZV0gPSAodmVyc2lvbiwgcm93WydzaGEyNTYnXSkKICAgIGlmIGxvY2tfcGlucyAhPSBleHBlY3RlZF9waW5zOgogICAgICAgIHJhaXNlIFByZWZsaWdodEZhaWxlZCgndHJ1c3RlZCBsb2NrIGFuZCB3aGVlbCBpbnZlbnRvcnkgZGlzYWdyZWUnKQogICAgcmV0dXJuIHByb3Bvc2FsLCByb3dzLCBsb2NrX3BpbnMKCgpkZWYgdmVyaWZ5X21vdW50ZWQoZGF0YXNldF9yb290LCAqLCBwYWNrYWdlPVBBQ0tBR0UsIG5vdz10aW1lLm1vbm90b25pYyk6CiAgICBzdGFydGVkID0gbm93KCkKICAgIHByb3Bvc2FsLCBhcnRpZmFjdHMsIHRydXN0ZWRfcGlucyA9IGxvYWRfaW5wdXRzKHBhY2thZ2UpCiAgICBkZWFkbGluZSA9IHN0YXJ0ZWQgKyBwcm9wb3NhbFsnaW50ZWdyaXR5X2J1ZGdldF9zZWNvbmRzJ10KCiAgICBkZWYgY2hlY2soKToKICAgICAgICBjdXJyZW50ID0gbm93KCkKICAgICAgICBpZiBjdXJyZW50ID49IGRlYWRsaW5lOgogICAgICAgICAgICByYWlzZSBQcmVmbGlnaHRGYWlsZWQoJ2ludGVncml0eSBkZWFkbGluZSBleGNlZWRlZCcpCiAgICAgICAgcmV0dXJuIGN1cnJlbnQKCiAgICByb290ID0gUGF0aChkYXRhc2V0X3Jvb3QpCiAgICBpZiBub3Qgcm9vdC5pc19kaXIoKToKICAgICAgICByYWlzZSBQcmVmbGlnaHRGYWlsZWQoJ2RhdGFzZXQgcm9vdCBpcyBub3QgbW91bnRlZCcpCiAgICBleHBlY3RlZCA9IHtyb3dbJ2ZpbGVuYW1lJ10gZm9yIHJvdyBpbiBhcnRpZmFjdHN9IHwgc2V0KHByb3Bvc2FsWydwdWJsaXNoZXJfbWV0YWRhdGFfc2hhMjU2J10pCiAgICBwcmVzZW50ID0gc2V0KCkKICAgIGZvciBwYXRoIGluIHJvb3Qucmdsb2IoJyonKToKICAgICAgICBjaGVjaygpCiAgICAgICAgaWYgcGF0aC5pc19zeW1saW5rKCk6CiAgICAgICAgICAgIHJhaXNlIFByZWZsaWdodEZhaWxlZCgnc3ltbGluayBpbiBkYXRhc2V0IGludmVudG9yeScpCiAgICAgICAgaWYgcGF0aC5pc19maWxlKCk6CiAgICAgICAgICAgIHByZXNlbnQuYWRkKHBhdGgucmVsYXRpdmVfdG8ocm9vdCkuYXNfcG9zaXgoKSkKICAgIGlmIHByZXNlbnQgIT0gZXhwZWN0ZWQ6CiAgICAgICAgcmFpc2UgUHJlZmxpZ2h0RmFpbGVkKCdkYXRhc2V0IGZpbGUgaW52ZW50b3J5IGRpZmZlcnM6IG1pc3Npbmc9JXMgZXh0cmE9JXMnICUKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKHNvcnRlZChleHBlY3RlZCAtIHByZXNlbnQpWzo1XSwgc29ydGVkKHByZXNlbnQgLSBleHBlY3RlZClbOjVdKSkKICAgIGZvciBuYW1lLCBkaWdlc3QgaW4gcHJvcG9zYWxbJ3B1Ymxpc2hlcl9tZXRhZGF0YV9zaGEyNTYnXS5pdGVtcygpOgogICAgICAgIGlmIGRpZ2VzdF9maWxlKHJvb3QgLyBuYW1lLCBjaGVjaykgIT0gZGlnZXN0OgogICAgICAgICAgICByYWlzZSBQcmVmbGlnaHRGYWlsZWQoJ3B1Ymxpc2hlciBtZXRhZGF0YSBkaWZmZXJzOiAnICsgbmFtZSkKICAgIHB1Ymxpc2hlcl9waW5zID0gcGlucygocm9vdCAvICdyZXF1aXJlbWVudHMubG9jaycpLnJlYWRfdGV4dChlbmNvZGluZz0ndXRmLTgnKSwgcmVxdWlyZV9oYXNoZXM9RmFsc2UpCiAgICBpZiB7bjogdiBmb3IgbiwgKHYsIF8pIGluIHB1Ymxpc2hlcl9waW5zLml0ZW1zKCl9ICE9IHtuOiB2IGZvciBuLCAodiwgXykgaW4gdHJ1c3RlZF9waW5zLml0ZW1zKCl9OgogICAgICAgIHJhaXNlIFByZWZsaWdodEZhaWxlZCgncHVibGlzaGVyIHBhY2thZ2UvdmVyc2lvbiBwaW5zIGRpZmZlcicpCiAgICBmb3Igcm93IGluIGFydGlmYWN0czoKICAgICAgICBjaGVjaygpCiAgICAgICAgcGF0aCA9IHJvb3QgLyByb3dbJ2ZpbGVuYW1lJ10KICAgICAgICBpZiBwYXRoLnN0YXQoKS5zdF9zaXplICE9IHJvd1snc2l6ZSddOgogICAgICAgICAgICByYWlzZSBQcmVmbGlnaHRGYWlsZWQoJ3doZWVsIHNpemUgZGlmZmVyczogJyArIHJvd1snZmlsZW5hbWUnXSkKICAgICAgICBpZiBkaWdlc3RfZmlsZShwYXRoLCBjaGVjaykgIT0gcm93WydzaGEyNTYnXToKICAgICAgICAgICAgcmFpc2UgUHJlZmxpZ2h0RmFpbGVkKCd3aGVlbCBieXRlIGhhc2ggZGlmZmVyczogJyArIHJvd1snZmlsZW5hbWUnXSkKICAgIGZpbmlzaGVkID0gY2hlY2soKSAgIyBleHBsaWNpdCBjaGVjayBhZnRlciBhbGwgcmVhZHM7IGFuIG92ZXJydW4gbmV2ZXIgcGFzc2VzCiAgICByZXR1cm4geydpbnRlZ3JpdHlfcGFzc2VkJzogVHJ1ZSwgJ3doZWVsX2J5dGVzX3ZlcmlmaWVkJzogbGVuKGFydGlmYWN0cyksCiAgICAgICAgICAgICd0cnVzdGVkX2FydGlmYWN0c19zaGEyNTYnOiBwcm9wb3NhbFsndHJ1c3RlZF9hcnRpZmFjdHNfc2hhMjU2J10sCiAgICAgICAgICAgICdpbnN0YWxsYXRpb25fcmVxdWlyZW1lbnRzX3NoYTI1Nic6IHByb3Bvc2FsWyd0cnVzdGVkX3JlcXVpcmVtZW50c19zaGEyNTYnXSwKICAgICAgICAgICAgJ2RhdGFzZXRfcmVmJzogcHJvcG9zYWxbJ2RhdGFzZXQnXVsncmVmJ10sICdyZXF1ZXN0ZWRfdmVyc2lvbic6IHByb3Bvc2FsWydkYXRhc2V0J11bJ3ZlcnNpb24nXSwKICAgICAgICAgICAgJ2FjY291bnRfYXR0YWNobWVudF92ZXJpZmllZCc6IEZhbHNlLCAncHJvdmlkZXJfYXR0YWNobWVudF92ZXJzaW9uX3ZlcmlmaWVkJzogRmFsc2UsCiAgICAgICAgICAgICdwZXJtaXNzaW9uc19hc3Nlc3NlZCc6IEZhbHNlLCAnZ3B1X2NvbXBhdGliaWxpdHlfZXN0YWJsaXNoZWQnOiBGYWxzZSwKICAgICAgICAgICAgJ2xhdW5jaF9hdXRob3JpemVkJzogRmFsc2UsICdlbGFwc2VkX3NlY29uZHMnOiByb3VuZChmaW5pc2hlZCAtIHN0YXJ0ZWQsIDMpfQo=', 'certification/direct_publisher_smoke_v1/server.py': 'IiIiTW9kZWwtc2VydmVyIHByb2Nlc3MgbGlmZWN5Y2xlOiBvd24gcHJvY2VzcyBncm91cCwgVENQLW9ubHkgcmVhZGluZXNzIHdpdGhpbiB0aGUgc3RhcnR1cCBjZWlsaW5nLCBhbmQgZ3JvdXAKdGVybWluYXRpb24gKFNJR1RFUk0sIHRoZW4gU0lHS0lMTCkgd2l0aCB2ZXJpZmljYXRpb24gdGhhdCBubyBtZW1iZXIgb2YgdGhlIGdyb3VwIHJlbWFpbnMuIiIiCmltcG9ydCBvcwpmcm9tIGNvbnRleHRsaWIgaW1wb3J0IGNvbnRleHRtYW5hZ2VyCmltcG9ydCBzaWduYWwKaW1wb3J0IHNvY2tldAppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCgpjbGFzcyBTdGFydHVwRmFpbGVkKFJ1bnRpbWVFcnJvcik6CiAgICBwYXNzCgoKQGNvbnRleHRtYW5hZ2VyCmRlZiBkZWZlcl9zdGFydHVwX3NpZ25hbHMoKToKICAgICIiIkRlZmVyIFB5dGhvbi1oYW5kbGVkIHNodXRkb3duIHNpZ25hbHMgdW50aWwgb3duZXJzaGlwIGlzIHJlZ2lzdGVyZWQuCgogICAgVW5saWtlIGJsb2NraW5nIGEgc2lnbmFsIG1hc2sgYXJvdW5kIFBvcGVuLCByZXBsYWNpbmcgdGhlIGhhbmRsZXJzIGRvZXMgbm90IGdpdmUgdGhlIGNoaWxkIGEgYmxvY2tlZCBTSUdURVJNCiAgICBtYXNrLiBUaW1lcnMga2VlcCBydW5uaW5nLiBEZWxpdmVyeSBpcyByZXBsYXllZCB1c2luZyB0aGUgb3JpZ2luYWwgaGFuZGxlciBhZnRlciB0aGUgcHJvdGVjdGVkIHNlY3Rpb24uCiAgICBEZWZhdWx0IGZhdGFsIHNpZ25hbHMgYW5kIFNJR0tJTEwgY2Fubm90IGJlIHJlY292ZXJlZCBieSBQeXRob247IHRoaXMgZ3VhcmQgY292ZXJzIHRoZSBub3RlYm9vayBjdXRvZmYgYW5kCiAgICBoYW5kbGVkIHVzZXIgaW50ZXJydXB0cywgYW5kIGFuIGV4Y2VwdGlvbmFsIFBvcGVuIG91dGNvbWUgaXMgc2VwYXJhdGVseSBtYXJrZWQgdW5jZXJ0YWluLgogICAgIiIiCiAgICBoYW5kbGVycywgcGVuZGluZyA9IHt9LCBbXQoKICAgIGRlZiBkZWZlcihzaWdudW0sIGZyYW1lKToKICAgICAgICBwZW5kaW5nLmFwcGVuZCgoc2lnbnVtLCBmcmFtZSkpCgogICAgdHJ5OgogICAgICAgIGZvciBzaWdudW0gaW4gKHNpZ25hbC5TSUdBTFJNLCBzaWduYWwuU0lHSU5ULCBzaWduYWwuU0lHVEVSTSwgc2lnbmFsLlNJR0hVUCk6CiAgICAgICAgICAgIGhhbmRsZXIgPSBzaWduYWwuZ2V0c2lnbmFsKHNpZ251bSkKICAgICAgICAgICAgaWYgY2FsbGFibGUoaGFuZGxlcik6CiAgICAgICAgICAgICAgICBoYW5kbGVyc1tzaWdudW1dID0gaGFuZGxlcgogICAgICAgICAgICAgICAgc2lnbmFsLnNpZ25hbChzaWdudW0sIGRlZmVyKQogICAgICAgIHlpZWxkCiAgICBmaW5hbGx5OgogICAgICAgIGZvciBzaWdudW0sIGhhbmRsZXIgaW4gaGFuZGxlcnMuaXRlbXMoKToKICAgICAgICAgICAgc2lnbmFsLnNpZ25hbChzaWdudW0sIGhhbmRsZXIpCiAgICAgICAgZm9yIHNpZ251bSwgZnJhbWUgaW4gcGVuZGluZzoKICAgICAgICAgICAgaGFuZGxlcnNbc2lnbnVtXShzaWdudW0sIGZyYW1lKQoKCmRlZiBhcmd2X2ZvcihzZXJ2ZXIsIHB5dGhvbiwgbW9kZWxfcGF0aCwgcG9ydCk6CiAgICB2YWx1ZXMgPSB7J3tweXRob259Jzogc3RyKHB5dGhvbiksICd7bW9kZWxfcGF0aH0nOiBzdHIobW9kZWxfcGF0aCksICd7cG9ydH0nOiBzdHIocG9ydCl9CiAgICByZXR1cm4gW3ZhbHVlcy5nZXQoaXRlbSwgaXRlbSkgZm9yIGl0ZW0gaW4gc2VydmVyWydhcmd2J11dCgoKZGVmIGdyb3VwX21lbWJlcnMocGdpZCwgcHJvYz1QYXRoKCcvcHJvYycpKToKICAgICIiIlBJRHMgd2hvc2UgcHJvY2VzcyBncm91cCBpcyBgcGdpZGAgKExpbnV4IC9wcm9jIHNjYW47IHpvbWJpZXMgaW5jbHVkZWQgdW50aWwgcmVhcGVkKS4iIiIKICAgIG1lbWJlcnMgPSBbXQogICAgZm9yIGVudHJ5IGluIHByb2MuaXRlcmRpcigpIGlmIHByb2MuaXNfZGlyKCkgZWxzZSBbXToKICAgICAgICBpZiBlbnRyeS5uYW1lLmlzZGlnaXQoKToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgZmllbGRzID0gKGVudHJ5IC8gJ3N0YXQnKS5yZWFkX3RleHQoKS5yc3BsaXQoJyknLCAxKVsxXS5zcGxpdCgpCiAgICAgICAgICAgIGV4Y2VwdCAoT1NFcnJvciwgSW5kZXhFcnJvcik6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBpZiBpbnQoZmllbGRzWzJdKSA9PSBwZ2lkOgogICAgICAgICAgICAgICAgbWVtYmVycy5hcHBlbmQoaW50KGVudHJ5Lm5hbWUpKQogICAgcmV0dXJuIHNvcnRlZChtZW1iZXJzKQoKCmNsYXNzIE1vZGVsU2VydmVyOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGFyZ3YsIGVudiwgbG9nLCBob3N0LCBwb3J0LCAqLCBpbmhlcml0X2Vudj1UcnVlKToKICAgICAgICBzZWxmLmFyZ3YsIHNlbGYuZW52LCBzZWxmLmxvZ19wYXRoLCBzZWxmLmhvc3QsIHNlbGYucG9ydCA9IGFyZ3YsIGVudiwgUGF0aChsb2cpLCBob3N0LCBwb3J0CiAgICAgICAgc2VsZi5pbmhlcml0X2VudiA9IGluaGVyaXRfZW52CiAgICAgICAgc2VsZi5wcm9jZXNzID0gc2VsZi5wZ2lkID0gTm9uZQogICAgICAgIHNlbGYub3duZXJzaGlwID0gJ25ldmVyX3NwYXduZWQnCiAgICAgICAgc2VsZi5ldmVudHMgPSBbXQogICAgICAgIHNlbGYucmVhcGVkX2Rlc2NlbmRhbnRzID0gW10KCiAgICBkZWYgc3RhcnQoc2VsZik6CiAgICAgICAgZW52ID0ge2s6IHYgZm9yIGssIHYgaW4gb3MuZW52aXJvbi5pdGVtcygpIGlmIGsgbm90IGluICgnUFlUSE9OUEFUSCcsICdQWVRIT05IT01FJywgJ1ZJUlRVQUxfRU5WJyl9IFwKICAgICAgICAgICAgaWYgc2VsZi5pbmhlcml0X2VudiBlbHNlIHt9CiAgICAgICAgZW52LnVwZGF0ZShzZWxmLmVudikKICAgICAgICBzZWxmLl9sb2cgPSBzZWxmLmxvZ19wYXRoLm9wZW4oJ2FiJykKICAgICAgICB3aXRoIGRlZmVyX3N0YXJ0dXBfc2lnbmFscygpOgogICAgICAgICAgICAjIE1pc3NpbmcgaGFuZGxlcyBhZnRlciBlbnRlcmluZyBQb3BlbiBkbyBub3QgcHJvdmUgdGhhdCBubyBjaGlsZCB3YXMgY3JlYXRlZC4KICAgICAgICAgICAgc2VsZi5vd25lcnNoaXAgPSAndW5jZXJ0YWluJwogICAgICAgICAgICBzZWxmLnByb2Nlc3MgPSBzdWJwcm9jZXNzLlBvcGVuKHNlbGYuYXJndiwgZW52PWVudiwgc3Rkb3V0PXNlbGYuX2xvZywgc3RkZXJyPXN1YnByb2Nlc3MuU1RET1VULAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0ZGluPXN1YnByb2Nlc3MuREVWTlVMTCwgc3RhcnRfbmV3X3Nlc3Npb249VHJ1ZSkKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgc2VsZi5wZ2lkID0gb3MuZ2V0cGdpZChzZWxmLnByb2Nlc3MucGlkKQogICAgICAgICAgICBleGNlcHQgUHJvY2Vzc0xvb2t1cEVycm9yOgogICAgICAgICAgICAgICAgIyBUaGUgbGVhZGVyIG1heSBhbHJlYWR5IGhhdmUgZXhpdGVkLiBzdGFydF9uZXdfc2Vzc2lvbiBmaXhlcyB0aGUgb3duZWQgZ3JvdXAgaWQgdG8gaXRzIFBJRDsKICAgICAgICAgICAgICAgICMgZGVzY2VuZGFudHMgY2FuIHN0aWxsIGJlIHByZXNlbnQgYW5kIG11c3QgYmUgY2hlY2tlZC90ZXJtaW5hdGVkIHVzaW5nIHRoYXQgc2FtZSBncm91cCBpZC4KICAgICAgICAgICAgICAgIHNlbGYucGdpZCA9IHNlbGYucHJvY2Vzcy5waWQKICAgICAgICAgICAgaWYgc2VsZi5wZ2lkICE9IHNlbGYucHJvY2Vzcy5waWQ6CiAgICAgICAgICAgICAgICBzZWxmLnBnaWQgPSBOb25lCiAgICAgICAgICAgICAgICByYWlzZSBTdGFydHVwRmFpbGVkKCdzcGF3bmVkIHByb2Nlc3MgZGlkIG5vdCBvd24gdGhlIGV4cGVjdGVkIG5ldyBwcm9jZXNzIGdyb3VwJykKICAgICAgICAgICAgc2VsZi5vd25lcnNoaXAgPSAncmVnaXN0ZXJlZCcKICAgICAgICAgICAgc2VsZi5ldmVudHMuYXBwZW5kKHsnZXZlbnQnOiAnc3RhcnRlZCcsICdwaWQnOiBzZWxmLnByb2Nlc3MucGlkLCAncGdpZCc6IHNlbGYucGdpZH0pCiAgICAgICAgcmV0dXJuIHNlbGYucHJvY2Vzcy5waWQKCiAgICBkZWYgd2FpdF9yZWFkeShzZWxmLCBkZWFkbGluZSwgaW50ZXJ2YWw9MS4wKToKICAgICAgICAiIiJQb2xsIGEgVENQIGNvbm5lY3Rpb24gKG5vIEhUVFAgcmVxdWVzdCkgdW50aWwgdGhlIHBvcnQgYWNjZXB0cywgdGhlIHByb2Nlc3MgZXhpdHMsIG9yIHRoZSBkZWFkbGluZS4iIiIKICAgICAgICBiZWdpbiA9IHRpbWUubW9ub3RvbmljKCkKICAgICAgICB3aGlsZSBUcnVlOgogICAgICAgICAgICBpZiBzZWxmLnByb2Nlc3MucG9sbCgpIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgcmFpc2UgU3RhcnR1cEZhaWxlZChmJ21vZGVsIHNlcnZlciBleGl0ZWQgd2l0aCB7c2VsZi5wcm9jZXNzLnJldHVybmNvZGV9IGJlZm9yZSByZWFkaW5lc3MnKQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICB3aXRoIHNvY2tldC5jcmVhdGVfY29ubmVjdGlvbigoc2VsZi5ob3N0LCBzZWxmLnBvcnQpLCB0aW1lb3V0PTEpOgogICAgICAgICAgICAgICAgICAgIHNlbGYuZXZlbnRzLmFwcGVuZCh7J2V2ZW50JzogJ3JlYWR5JywgJ3NlY29uZHMnOiByb3VuZCh0aW1lLm1vbm90b25pYygpIC0gYmVnaW4sIDMpfSkKICAgICAgICAgICAgICAgICAgICByZXR1cm4gdGltZS5tb25vdG9uaWMoKSAtIGJlZ2luCiAgICAgICAgICAgIGV4Y2VwdCBPU0Vycm9yOgogICAgICAgICAgICAgICAgcGFzcwogICAgICAgICAgICBpZiB0aW1lLm1vbm90b25pYygpID49IGRlYWRsaW5lOgogICAgICAgICAgICAgICAgcmFpc2UgU3RhcnR1cEZhaWxlZCgnbW9kZWwgc3RhcnR1cCBjZWlsaW5nIHJlYWNoZWQgYmVmb3JlIHJlYWRpbmVzcycpCiAgICAgICAgICAgIHRpbWUuc2xlZXAobWluKGludGVydmFsLCBtYXgoMC4wLCBkZWFkbGluZSAtIHRpbWUubW9ub3RvbmljKCkpKSkKCiAgICBkZWYgc3RvcChzZWxmLCBkZWFkbGluZSwgdGVybWluYXRlX2dyYWNlLCBraWxsX2dyYWNlKToKICAgICAgICAiIiJUZXJtaW5hdGUgdGhlIHdob2xlIGdyb3VwIGFuZCB2ZXJpZnkgaXQgaXMgZ29uZTsgcmV0dXJucyBhIHJlY2VpcHQgYW5kIG5ldmVyIHJhaXNlcy4gVGhlIFNJR0tJTEwKICAgICAgICBlc2NhbGF0aW9uIGFuZCB0aGUgcmVhcCBydW4gaW4gYGZpbmFsbHlgIGJsb2NrczogYW4gaW50ZXJydXB0aW9uIGR1cmluZyB0aGUgU0lHVEVSTSBncmFjZSAoZm9yIGV4YW1wbGUgYQogICAgICAgIHN0cmF5IGFsYXJtKSBpcyByZWNvcmRlZCBhbmQgY2Fubm90IHNraXAgdGhlIGtpbGwuIFRoZSBraWxsIHBoYXNlIGtlZXBzIGl0cyBvd24gZ3JhY2UgZXZlbiBpZiB0aGUgU0lHVEVSTQogICAgICAgIHBoYXNlIHVzZWQgdXAgdGhlIHNoYXJlZCBkZWFkbGluZS4iIiIKICAgICAgICByZWNlaXB0ID0geydwZ2lkJzogc2VsZi5wZ2lkLCAnb3duZXJzaGlwJzogc2VsZi5vd25lcnNoaXAsCiAgICAgICAgICAgICAgICAgICAnc2lndGVybV9zZW50JzogRmFsc2UsICdzaWdraWxsX3NlbnQnOiBGYWxzZSwgJ2dyb3Vwc19hYnNlbnQnOiBGYWxzZSwKICAgICAgICAgICAgICAgICAgICdyZW1haW5pbmdfbWVtYmVycyc6IE5vbmUsICdpbnRlcnJ1cHRlZCc6IFtdLCAnZXJyb3InOiBOb25lfQogICAgICAgIGlmIHNlbGYucHJvY2VzcyBpcyBOb25lOgogICAgICAgICAgICByZWNlaXB0Wydncm91cHNfYWJzZW50J10gPSBzZWxmLm93bmVyc2hpcCA9PSAnbmV2ZXJfc3Bhd25lZCcKICAgICAgICAgICAgaWYgbm90IHJlY2VpcHRbJ2dyb3Vwc19hYnNlbnQnXToKICAgICAgICAgICAgICAgIHJlY2VpcHRbJ2Vycm9yJ10gPSAnc3Bhd24gb3duZXJzaGlwIHVuY2VydGFpbjogbm8gY2hpbGQgaGFuZGxlOyBncm91cCBhYnNlbmNlIGNhbm5vdCBiZSB2ZXJpZmllZCcKICAgICAgICAgICAgaWYgaGFzYXR0cihzZWxmLCAnX2xvZycpOgogICAgICAgICAgICAgICAgc2VsZi5fbG9nLmNsb3NlKCkKICAgICAgICAgICAgcmV0dXJuIHJlY2VpcHQKICAgICAgICBpZiBzZWxmLnBnaWQgaXMgTm9uZToKICAgICAgICAgICAgIyBFbWVyZ2VuY3kgdGVybWluYXRpb24gaXMgc3RpbGwgcG9zc2libGUgd2hlbiBhIGhhbmRsZSBleGlzdHMgYnV0IHJlZ2lzdHJhdGlvbiB3YXMgaW50ZXJydXB0ZWQuCiAgICAgICAgICAgICMgUG9wZW4gdXNlZCBzdGFydF9uZXdfc2Vzc2lvbj1UcnVlLCBzbyBuZXZlciBzaWduYWwgdGhlIGNhbGxlcidzIG9yIGFuIGFyYml0cmFyeSBvYnNlcnZlZCBncm91cC4KICAgICAgICAgICAgc2VsZi5wZ2lkID0gc2VsZi5wcm9jZXNzLnBpZAogICAgICAgICAgICByZWNlaXB0WydwZ2lkJ10gPSBzZWxmLnBnaWQKCiAgICAgICAgZGVmIHdhaXQodW50aWwpOgogICAgICAgICAgICB3aGlsZSB0aW1lLm1vbm90b25pYygpIDwgdW50aWwgYW5kIHNlbGYuX2dyb3VwX2FsaXZlKCk6CiAgICAgICAgICAgICAgICB0aW1lLnNsZWVwKDAuMSkKICAgICAgICB0cnk6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGlmIHNlbGYuX2dyb3VwX2FsaXZlKCk6CiAgICAgICAgICAgICAgICAgICAgb3Mua2lsbHBnKHNlbGYucGdpZCwgc2lnbmFsLlNJR1RFUk0pCiAgICAgICAgICAgICAgICAgICAgcmVjZWlwdFsnc2lndGVybV9zZW50J10gPSBUcnVlCiAgICAgICAgICAgICAgICAgICAgd2FpdChtaW4odGltZS5tb25vdG9uaWMoKSArIHRlcm1pbmF0ZV9ncmFjZSwgZGVhZGxpbmUpKQogICAgICAgICAgICBleGNlcHQgUHJvY2Vzc0xvb2t1cEVycm9yOgogICAgICAgICAgICAgICAgcGFzcwogICAgICAgICAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6ICAjIG5vcWE6IEIwMzYgLSBtdXN0IG5vdCBieXBhc3MgZXNjYWxhdGlvbgogICAgICAgICAgICAgICAgcmVjZWlwdFsnaW50ZXJydXB0ZWQnXS5hcHBlbmQoZidzaWd0ZXJtIHBoYXNlOiB7dHlwZShleGMpLl9fbmFtZV9ffToge3N0cihleGMpWzoxMjBdfScpCiAgICAgICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgICAgICBmb3IgYXR0ZW1wdCBpbiByYW5nZSgyKTogICMgcmV0cmllZCBvbmNlIGlmIHRoZSBraWxsIHBoYXNlIGl0c2VsZiBpcyBpbnRlcnJ1cHRlZAogICAgICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICAgICAgaWYgc2VsZi5fZ3JvdXBfYWxpdmUoKToKICAgICAgICAgICAgICAgICAgICAgICAgICAgIG9zLmtpbGxwZyhzZWxmLnBnaWQsIHNpZ25hbC5TSUdLSUxMKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgcmVjZWlwdFsnc2lna2lsbF9zZW50J10gPSBUcnVlCiAgICAgICAgICAgICAgICAgICAgICAgICAgICB3YWl0KHRpbWUubW9ub3RvbmljKCkgKyBraWxsX2dyYWNlKQogICAgICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICAgICAgICAgIGV4Y2VwdCBQcm9jZXNzTG9va3VwRXJyb3I6CiAgICAgICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICAgICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZXhjOiAgIyBub3FhOiBCMDM2CiAgICAgICAgICAgICAgICAgICAgICAgIHJlY2VpcHRbJ2ludGVycnVwdGVkJ10uYXBwZW5kKGYnc2lna2lsbCBwaGFzZToge3R5cGUoZXhjKS5fX25hbWVfX306IHtzdHIoZXhjKVs6MTIwXX0nKQogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHNlbGYuX3JlYXAoKQogICAgICAgICAgICAgICAgcmVtYWluaW5nID0gZ3JvdXBfbWVtYmVycyhzZWxmLnBnaWQpCiAgICAgICAgICAgICAgICByZWNlaXB0WydyZW1haW5pbmdfbWVtYmVycyddID0gcmVtYWluaW5nCiAgICAgICAgICAgICAgICByZWNlaXB0Wydncm91cHNfYWJzZW50J10gPSAoc2VsZi5vd25lcnNoaXAgPT0gJ3JlZ2lzdGVyZWQnIGFuZCBub3QgcmVtYWluaW5nCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGFuZCBub3Qgc2VsZi5fZ3JvdXBfYWxpdmUoKSkKICAgICAgICAgICAgICAgIGlmIHNlbGYub3duZXJzaGlwICE9ICdyZWdpc3RlcmVkJzoKICAgICAgICAgICAgICAgICAgICByZWNlaXB0WydlcnJvciddID0gJ3NwYXduIG93bmVyc2hpcCB1bmNlcnRhaW47IGVtZXJnZW5jeSB0ZXJtaW5hdGlvbiBhdHRlbXB0ZWQnCiAgICAgICAgICAgICAgICByZWNlaXB0WydleGl0X2NvZGUnXSA9IHNlbGYucHJvY2Vzcy5yZXR1cm5jb2RlCiAgICAgICAgICAgICAgICByZWNlaXB0WydyZWFwZWRfZGVzY2VuZGFudHMnXSA9IHNvcnRlZChzZWxmLnJlYXBlZF9kZXNjZW5kYW50cykKICAgICAgICAgICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZXhjOiAgIyBub3FhOiBCMDM2CiAgICAgICAgICAgICAgICByZWNlaXB0WydlcnJvciddID0gZid7dHlwZShleGMpLl9fbmFtZV9ffToge3N0cihleGMpWzoyNTZdfScKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgc2VsZi5fbG9nLmNsb3NlKCkKICAgICAgICAgICAgZXhjZXB0IE9TRXJyb3I6CiAgICAgICAgICAgICAgICBwYXNzCiAgICAgICAgc2VsZi5ldmVudHMuYXBwZW5kKHsnZXZlbnQnOiAnc3RvcHBlZCcsICoqe2s6IHJlY2VpcHRba10gZm9yIGsgaW4gKCdzaWd0ZXJtX3NlbnQnLCAnc2lna2lsbF9zZW50JywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgJ2dyb3Vwc19hYnNlbnQnLCAnaW50ZXJydXB0ZWQnKX19KQogICAgICAgIHJldHVybiByZWNlaXB0CgogICAgZGVmIF9ncm91cF9hbGl2ZShzZWxmKToKICAgICAgICBzZWxmLl9yZWFwKCkKICAgICAgICB0cnk6CiAgICAgICAgICAgIG9zLmtpbGxwZyhzZWxmLnBnaWQsIDApCiAgICAgICAgZXhjZXB0IFByb2Nlc3NMb29rdXBFcnJvcjoKICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICAgICAgZXhjZXB0IFBlcm1pc3Npb25FcnJvcjoKICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgICAgICByZXR1cm4gYm9vbChncm91cF9tZW1iZXJzKHNlbGYucGdpZCkpCgogICAgZGVmIF9yZWFwKHNlbGYpOgogICAgICAgICMgUG9wZW4gb3ducyB0aGUgbGVhZGVyJ3Mgd2FpdCBzdGF0dXMuIE9ubHkgYWZ0ZXIgaXQgaGFzIGJlZW4gcmVhcGVkIG1heSB3ZQogICAgICAgICMgd2FpdCBmb3IgYWRvcHRlZCBkZXNjZW5kYW50cywgYW5kIG9ubHkgd2l0aGluIHRoaXMgb3duZWQgcHJvY2VzcyBncm91cC4KICAgICAgICBpZiBzZWxmLnByb2Nlc3MucG9sbCgpIGlzIE5vbmUgb3Igc2VsZi5wZ2lkIGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybgogICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHBpZCwgXyA9IG9zLndhaXRwaWQoLXNlbGYucGdpZCwgb3MuV05PSEFORykKICAgICAgICAgICAgZXhjZXB0IENoaWxkUHJvY2Vzc0Vycm9yOgogICAgICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgICAgIGlmIG5vdCBwaWQ6CiAgICAgICAgICAgICAgICByZXR1cm4KICAgICAgICAgICAgc2VsZi5yZWFwZWRfZGVzY2VuZGFudHMuYXBwZW5kKHBpZCkK', 'certification/phase4_multimodal_preflight_v3/__init__.py': 'IiIiQm91bmRlZCB0aHJlZS1hcm0gZGlhZ25vc3RpYzsgbm8gcHJvZHVjdGlvbiBvciBHUFUgbGF1bmNoIGF1dGhvcml0eS4iIiIK', 'certification/phase4_multimodal_preflight_v3/images.py': 'IiIiRGV0ZXJtaW5pc3RpYyBsb3NzbGVzcyBncmlkIHJlbmRlcmluZyBhbmQgcGlubmVkLXByb2Nlc3NvciByZXNpemUgYXJpdGhtZXRpYyAoc3RkbGliIG9ubHkpLiIiIgppbXBvcnQgYmFzZTY0LGhhc2hsaWIsbWF0aCxzdHJ1Y3QsemxpYgoKIyBSZW5kZXJlciBwYWxldHRlIGZvciB0aGlzIHByZWZsaWdodDogMTYgZGlzdGluY3QgUkdCIHRyaXBsZXMuIEl0IGlzIHRoaXMKIyBwYWNrYWdlJ3MgZG9jdW1lbnRlZCBsZWdlbmQsIHNlbnQgdG8gdGhlIG1vZGVsIGFzIHRleHQ7IG5vdCBhIGNsYWltIGFib3V0IGFueQojIG9mZmljaWFsIGdhbWUgcGFsZXR0ZS4KUEFMRVRURT1bKDI1NSwyNTUsMjU1KSwoMjA0LDIwNCwyMDQpLCgxNTMsMTUzLDE1MyksKDEwMiwxMDIsMTAyKSwoNTEsNTEsNTEpLCgwLDAsMCksCiAgICAoMjI5LDU4LDE2MyksKDI1NSwxMjMsMjA0KSwoMjQ5LDYwLDQ5KSwoMzAsMTQ3LDI1NSksKDEzNiwyMTYsMjQxKSwoMjU1LDIyMCwwKSwKICAgICgyNTUsMTMzLDI3KSwoMTQ2LDE4LDQ5KSwoNzksMjA0LDQ4KSwoMTYzLDg2LDIxNCldCkNFTEw9MTYKIyBQaW5uZWQgcHJlcHJvY2Vzc29yX2NvbmZpZy5qc29uIGF0IHRoZSBmcm96ZW4gcmV2aXNpb24gKHNlZSBwcm90b2NvbCkuClBBVENILE1FUkdFLE1JTl9QSVhFTFMsTUFYX1BJWEVMUz0xNiwyLDY1NTM2LDE2Nzc3MjE2CgpkZWYgY2hlY2tfZ3JpZChncmlkKToKICAgIGlmICh0eXBlKGdyaWQpIGlzIG5vdCBsaXN0IG9yIG5vdCAxPD1sZW4oZ3JpZCk8PTY0IG9yIHR5cGUoZ3JpZFswXSkgaXMgbm90IGxpc3Qgb3Igbm90IDE8PWxlbihncmlkWzBdKTw9NjQKICAgICAgICBvciBhbnkodHlwZShyKSBpcyBub3QgbGlzdCBvciBsZW4ocikhPWxlbihncmlkWzBdKSBvciBhbnkodHlwZSh2KSBpcyBub3QgaW50IG9yIG5vdCAwPD12PD0xNSBmb3IgdiBpbiByKSBmb3IgciBpbiBncmlkKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignZ3JpZCcpCgpkZWYgcG5nKGdyaWQsY2VsbD1DRUxMKToKICAgICIiIkVhY2ggY2VsbCBpcyBhIHVuaWZvcm0gY2VsbCB4IGNlbGwgYmxvY2s7IFJHQiwgOC1iaXQsIG5vIGZpbHRlcmluZywgZml4ZWQgemxpYiBsZXZlbC4iIiIKICAgIGNoZWNrX2dyaWQoZ3JpZCkKICAgIHJvd3M9Ynl0ZWFycmF5KCkKICAgIGZvciByIGluIGdyaWQ6CiAgICAgICAgbGluZT1iJycuam9pbihieXRlcyhQQUxFVFRFW3ZdKSpjZWxsIGZvciB2IGluIHIpCiAgICAgICAgZm9yIF8gaW4gcmFuZ2UoY2VsbCk6cm93cys9YidceDAwJytsaW5lCiAgICBkZWYgY2h1bmsoa2luZCxkYXRhKTpyZXR1cm4gc3RydWN0LnBhY2soJyFJJyxsZW4oZGF0YSkpK2tpbmQrZGF0YStzdHJ1Y3QucGFjaygnIUknLHpsaWIuY3JjMzIoa2luZCtkYXRhKSYweGZmZmZmZmZmKQogICAgaGVhZGVyPXN0cnVjdC5wYWNrKCchSUlCQkJCQicsbGVuKGdyaWRbMF0pKmNlbGwsbGVuKGdyaWQpKmNlbGwsOCwyLDAsMCwwKQogICAgcmV0dXJuIGInXHg4OVBOR1xyXG5ceDFhXG4nK2NodW5rKGInSUhEUicsaGVhZGVyKStjaHVuayhiJ0lEQVQnLHpsaWIuY29tcHJlc3MoYnl0ZXMocm93cyksOSkpK2NodW5rKGInSUVORCcsYicnKQoKZGVmIGRlY29kZShyYXcpOgogICAgIiIiSW5kZXBlbmRlbnQgUE5HIHJlYWRlciBmb3IgdGhlIGV4YWN0IGVuY29kaW5nIGFib3ZlOyByZXR1cm5zIHRoZSBncmlkIG9mIHBhbGV0dGUgaW5kaWNlcy4iIiIKICAgIGlmIHJhd1s6OF0hPWInXHg4OVBOR1xyXG5ceDFhXG4nOnJhaXNlIFZhbHVlRXJyb3IoJ3BuZyBzaWduYXR1cmUnKQogICAgcG9zPTg7ZGF0YT1iJyc7d2lkdGg9aGVpZ2h0PU5vbmUKICAgIHdoaWxlIHBvczxsZW4ocmF3KToKICAgICAgICBzaXplPXN0cnVjdC51bnBhY2soJyFJJyxyYXdbcG9zOnBvcys0XSlbMF07a2luZD1yYXdbcG9zKzQ6cG9zKzhdO2JvZHk9cmF3W3Bvcys4OnBvcys4K3NpemVdCiAgICAgICAgaWYgemxpYi5jcmMzMihraW5kK2JvZHkpJjB4ZmZmZmZmZmYhPXN0cnVjdC51bnBhY2soJyFJJyxyYXdbcG9zKzgrc2l6ZTpwb3MrMTIrc2l6ZV0pWzBdOnJhaXNlIFZhbHVlRXJyb3IoJ3BuZyBjcmMnKQogICAgICAgIGlmIGtpbmQ9PWInSUhEUic6CiAgICAgICAgICAgIHdpZHRoLGhlaWdodCxkZXB0aCxjb2xvcixfLF8sXz1zdHJ1Y3QudW5wYWNrKCchSUlCQkJCQicsYm9keSkKICAgICAgICAgICAgaWYgKGRlcHRoLGNvbG9yKSE9KDgsMik6cmFpc2UgVmFsdWVFcnJvcigncG5nIGZvcm1hdCcpCiAgICAgICAgZWxpZiBraW5kPT1iJ0lEQVQnOmRhdGErPWJvZHkKICAgICAgICBwb3MrPTEyK3NpemUKICAgIHBpeGVscz16bGliLmRlY29tcHJlc3MoZGF0YSk7c3RyaWRlPXdpZHRoKjMrMTtpbmRleD17cmdiOmkgZm9yIGkscmdiIGluIGVudW1lcmF0ZShQQUxFVFRFKX0KICAgIGltYWdlPVtdCiAgICBmb3IgeSBpbiByYW5nZShoZWlnaHQpOgogICAgICAgIHJvdz1waXhlbHNbeSpzdHJpZGU6KHkrMSkqc3RyaWRlXQogICAgICAgIGlmIHJvd1swXSE9MDpyYWlzZSBWYWx1ZUVycm9yKCdwbmcgZmlsdGVyJykKICAgICAgICBpbWFnZS5hcHBlbmQoW2luZGV4W3R1cGxlKHJvd1sxKzMqeDo0KzMqeF0pXSBmb3IgeCBpbiByYW5nZSh3aWR0aCldKQogICAgcmV0dXJuIHdpZHRoLGhlaWdodCxpbWFnZQoKZGVmIGdyaWRfZnJvbV9wbmcocmF3LGNlbGw9Q0VMTCk6CiAgICB3aWR0aCxoZWlnaHQsaW1hZ2U9ZGVjb2RlKHJhdykKICAgIGlmIHdpZHRoJWNlbGwgb3IgaGVpZ2h0JWNlbGw6cmFpc2UgVmFsdWVFcnJvcignY2VsbCBhbGlnbm1lbnQnKQogICAgZ3JpZD1bW2ltYWdlW3kqY2VsbF1beCpjZWxsXSBmb3IgeCBpbiByYW5nZSh3aWR0aC8vY2VsbCldIGZvciB5IGluIHJhbmdlKGhlaWdodC8vY2VsbCldCiAgICBmb3IgeSBpbiByYW5nZShoZWlnaHQpOgogICAgICAgIGZvciB4IGluIHJhbmdlKHdpZHRoKToKICAgICAgICAgICAgaWYgaW1hZ2VbeV1beF0hPWdyaWRbeS8vY2VsbF1beC8vY2VsbF06cmFpc2UgVmFsdWVFcnJvcignbm9uLXVuaWZvcm0gY2VsbCcpCiAgICByZXR1cm4gZ3JpZAoKZGVmIGRhdGFfdXJsKHJhdyk6cmV0dXJuICdkYXRhOmltYWdlL3BuZztiYXNlNjQsJytiYXNlNjQuYjY0ZW5jb2RlKHJhdykuZGVjb2RlKCkKCmRlZiBzbWFydF9yZXNpemUoaGVpZ2h0LHdpZHRoLGZhY3Rvcj1QQVRDSCpNRVJHRSxtaW5fcGl4ZWxzPU1JTl9QSVhFTFMsbWF4X3BpeGVscz1NQVhfUElYRUxTKToKICAgICIiIlByb3Zpc2lvbmFsIGFyaXRobWV0aWMgbWlycm9yaW5nIHRoZSBRd2VuMi1WTCByZXNpemUgcnVsZTsgdmVyaWZpZWQgb24gdGFyZ2V0LCBuZXZlciBhc3N1bWVkLiIiIgogICAgaD1yb3VuZChoZWlnaHQvZmFjdG9yKSpmYWN0b3I7dz1yb3VuZCh3aWR0aC9mYWN0b3IpKmZhY3RvcgogICAgaWYgaCp3Pm1heF9waXhlbHM6CiAgICAgICAgYmV0YT1tYXRoLnNxcnQoaGVpZ2h0KndpZHRoL21heF9waXhlbHMpO2g9bWF0aC5mbG9vcihoZWlnaHQvYmV0YS9mYWN0b3IpKmZhY3Rvcjt3PW1hdGguZmxvb3Iod2lkdGgvYmV0YS9mYWN0b3IpKmZhY3RvcgogICAgZWxpZiBoKnc8bWluX3BpeGVsczoKICAgICAgICBiZXRhPW1hdGguc3FydChtaW5fcGl4ZWxzLyhoZWlnaHQqd2lkdGgpKTtoPW1hdGguY2VpbChoZWlnaHQqYmV0YS9mYWN0b3IpKmZhY3Rvcjt3PW1hdGguY2VpbCh3aWR0aCpiZXRhL2ZhY3RvcikqZmFjdG9yCiAgICByZXR1cm4gaCx3CgpkZWYgZXhwZWN0ZWRfaW1hZ2VfdG9rZW5zKGhlaWdodCx3aWR0aCk6CiAgICBoLHc9c21hcnRfcmVzaXplKGhlaWdodCx3aWR0aCkKICAgIHJldHVybiB7J3Byb2Nlc3NlZF9oZWlnaHQnOmgsJ3Byb2Nlc3NlZF93aWR0aCc6dywnZ3JpZF90aHcnOlsxLGgvL1BBVENILHcvL1BBVENIXSwnaW1hZ2VfdG9rZW5zJzooaC8vUEFUQ0gpKih3Ly9QQVRDSCkvLyhNRVJHRSpNRVJHRSl9CgpkZWYgc2hhKHJhdyk6cmV0dXJuIGhhc2hsaWIuc2hhMjU2KHJhdykuaGV4ZGlnZXN0KCkK', 'research/cpu_competition_v1/__init__.py': 'IiIiQ1BVIGNvbXBldGl0aW9uIHJvdXRlOiBleGFjdCBsczIwIGRlbW9uc3RyYXRpb24gYW5kIGRpc2Nsb3NlZCBFMCBiYXNlbGluZS4iIiIK', 'research/cpu_competition_v1/requirements.lock': 'YW5ub3RhdGVkX3R5cGVzPT0wLjcuMCAtLWhhc2g9c2hhMjU2OjFmMDJlOGI0M2E4ZmJiYzNmM2UwZDRmMGY0YmZjODEzMWJjYjRlZWJlODg0OWI4ZTVjNzczZjNhMWM1ODJhNTMKYXJjX2FnaT09MC45LjggLS1oYXNoPXNoYTI1NjphZWNhMTY2M2RiMzQyZTkxY2I4ZmM5NmNmMGM4M2UyZmEzOWRiMTY0MGJjMTI5OGU4MGE4ZDUyMmUzYWY3MGYzCmFyY2VuZ2luZT09MC45LjMgLS1oYXNoPXNoYTI1Njo1Zjk3MzlkNmQwMDU1NzgwYTQ1ODFmZDZmZTA5MDY2YmIwODc3NWM0YzgyMTJjOWFkY2NhMmViMDA4YWVmNTljCmJsaW5rZXI9PTEuOS4wIC0taGFzaD1zaGEyNTY6YmEwZWZhYTkwODBiNjE5ZmYyZjM0NTlkMWQ1MDBjNTdiZGRlYTRhNmI0MjRiNjBhOTExNDFkYjZmZDJmMDhiYwpjZXJ0aWZpPT0yMDI2LjIuMjUgLS1oYXNoPXNoYTI1NjowMjc2OTJlNDQwMmFkOTk0ZjFjNDJlNTJhNDk5N2E5NzYzYzY0NmI3M2U0MDk2ZTRkNWQ2ZGI4YWYxZDZmMGZhCmNoYXJzZXRfbm9ybWFsaXplcj09My40LjcgLS1oYXNoPXNoYTI1Njo1NjQ5ZmQxYzdiYWRlMDJmMzIwYTQ2MmZkZWZkMGI0YmQzY2UwMzYwNjU4MzZkNGY0MmUwZGU5NTgwMzhlMTE2CmNsaWNrPT04LjMuMiAtLWhhc2g9c2hhMjU2OjE5MjRkMmMyN2M1NjUzNTYxY2QyY2FlNDU0OGQxNDA2MDM5Y2I3OWI4NThiNzQ3Y2ZlYTI0OTI0YmJjMTYxNmQKY29udG91cnB5PT0xLjMuMyAtLWhhc2g9c2hhMjU2OjRkMDBlNjU1ZmNlZjA4YWJhMzVlYzk2MTA1MzZiZmU5MDI2N2Q3YWI1YmE5NDRmNzAzMjU0OWM1NWExNDZkYTEKY3ljbGVyPT0wLjEyLjEgLS1oYXNoPXNoYTI1Njo4NWNlZjdjZmYyMjJkODY0NDE2MTUyOTgwODQ2NTk3MmU1MTM0MDU5OTQ1OWI4YWMzY2NiYWM1YTg1NGUwZDMwCmZsYXNrPT0zLjEuMyAtLWhhc2g9c2hhMjU2OmY0YmNiZWZjMTI0MjkxOTI1ZjFhMjY0NDZkYTMxYTUxNzhmOTQ4Mzg2MjIzM2IyM2MwYzk2YTIwNzAxZjY3MGMKZm9udHRvb2xzPT00LjYyLjEgLS1oYXNoPXNoYTI1NjoxNDlmN2Q4NGFmY2E2NTlkMWE5N2UzOWE0Nzc4Nzk0YTJmODNiZjM0NGM1ZWU1MTM0ZTA5OTk1MDg2Y2MyMzkyCmlkbmE9PTMuMTEgLS1oYXNoPXNoYTI1Njo3NzFhODdmNDlkOWRlZmFmNjQwOTFlNmU2ZmU5YzE4ZDQ4MzNmMTQwYmQxOTQ2NDc5NWJjMzJkOTY2Y2EzN2VhCml0c2Rhbmdlcm91cz09Mi4yLjAgLS1oYXNoPXNoYTI1NjpjNjI0MmZjNDllMzU5NThjOGIxNTE0MTM0M2FhNjYwZGI1ZmM1NGQ0ZjEzYTFkYjAxYTNmNTg5MWI5ODcwMGVmCmppbmphMj09My4xLjYgLS1oYXNoPXNoYTI1Njo4NWVjZTQ0NTFmNDkyZDBjMTNjNWRkN2MxM2E2NDY4MWE4NmFmYWU2M2E1ZjM0NzkwOGRhZjEwM2NlNmQyZjY3Cmtpd2lzb2x2ZXI9PTEuNS4wIC0taGFzaD1zaGEyNTY6YmI1MTM2ZmI1MzUyZDNmNDIyZGYzM2YwYzg3OWExYjBjMjA0MDA0MzI0MTUwY2MzYjVlM2M0ZjMxMGM5MDQ5ZgptYXJrdXBzYWZlPT0zLjAuMyAtLWhhc2g9c2hhMjU2OmQ2ZGQwYmU1YjViMTg5ZDMxZGI3Y2RhNDhiOTFkN2UwYTk3OTVmMzE0MzBiN2YyNzEyMTlhYjMwZjFkM2FjOWQKbWF0cGxvdGxpYj09My4xMC44IC0taGFzaD1zaGEyNTY6M2FiNGFhYmM3MmRlNGZmNzdiM2VjMzNhNmQ3OGE2ODIyN2JmMTEyMzQ2NTg4N2Y5OTA1YmE3OTE4NGExY2MwNApudW1weT09Mi40LjQgLS1oYXNoPXNoYTI1Njo4MWY0YTE0YmVlNDdhZWM1NGY4ODNlMGNhZDJkNzM5ODY2NDBjMTU5MGViOWJmYWFiYTdhZDE3Mzk0NDgxZTZlCnBhY2thZ2luZz09MjYuMSAtLWhhc2g9c2hhMjU2OjVkOWMwNjY5YzYyODVlNDkxZTBjZWQyZWVlNTg3ZWFmNjdiNjcwZDk0YTE5ZTk0ZTM5ODRhNDgxYWJhNjgwMmYKcGlsbG93PT0xMi4yLjAgLS1oYXNoPXNoYTI1Njo2MmY1NDA5MzM2YWRiMDY2M2I3Y2FhMGRhNWM3ZDllN2JkYmFhZTljZTc2MWQzNDY2OTQyMGMyYTgwMWIyNzgwCnB5ZGFudGljPT0yLjEzLjIgLS1oYXNoPXNoYTI1NjphNTI1MDg3ZjRjMDNkN2U3NDU2YTNkZTg5YjY0Y2Q2OTNkMjIyOTkzM2JiMTA2OGI5YWY2YmVmZDU1NjM2OTRlCnB5ZGFudGljX2NvcmU9PTIuNDYuMiAtLWhhc2g9c2hhMjU2OjQ4NjQ5Y2YyZDhjMzU4ZDc5NTg2ZTlmYjJmODIzNTkwMmZjYWEyZDk2OWVjMWM1MzAxZjJkMTg3M2IyZjgzMjEKcHlwYXJzaW5nPT0zLjMuMiAtLWhhc2g9c2hhMjU2Ojg1MGJhMTQ4YmQ5MDhkN2UyNDExNTg3ZTI0N2ExZTRmMDMyNzgzOWM0MGUyZTVlNmQwNWEwMDdlY2M2OTkxMWQKcHl0aG9uX2RhdGV1dGlsPT0yLjkuMC5wb3N0MCAtLWhhc2g9c2hhMjU2OmE4YjJiYzdiZmZhZTI4MjI4MWM4MTQwYTk3ZDNhYTljMTRkYTBiMTM2ZGZlODNmODUwZWVhOWE1Zjc0NzA0MjcKcHl0aG9uX2RvdGVudj09MS4yLjIgLS1oYXNoPXNoYTI1NjoxZDgyMTQ3ODlhMjRkZTQ1NWE4YjhiZDhhZTZmZTNjNmI2OWE1ZTNkNjRhYThhOGU1ZDY4ZTY5NGJiY2IyODVhCnJlcXVlc3RzPT0yLjMzLjEgLS1oYXNoPXNoYTI1Njo0ZTZkMWVmNDYyZjM2MjZhMWYwYTBhOWM0MmRkOTNjNjNiYWQzM2Y5ZjFjMTkzNzUwOWI4YzVjODcxOGFiNTZhCnNpeD09MS4xNy4wIC0taGFzaD1zaGEyNTY6NDcyMWYzOTFlZDkwNTQxZmRkYWNhYjVhY2Y5NDdhYTBkM2RjN2QyN2IyZTFlOGVkYTJiZTg5NzA1ODZjMzI3NAp0eXBpbmdfZXh0ZW5zaW9ucz09NC4xNS4wIC0taGFzaD1zaGEyNTY6ZjBmYTE5YzY4NDU3NThhYjA4MDc0YTBjZmE4YjdhZWNiNzFjOTk5Y2E3M2Q2Mjg4M2JjMjVjYzAxOGM0ZTU0OAp0eXBpbmdfaW5zcGVjdGlvbj09MC40LjIgLS1oYXNoPXNoYTI1Njo0ZWQxY2FjYmRjMjk4YzIyMGYxYmQyNDllZDUyODdjYWExNmYzNGQ0NGVmNGU5YzNkMGNiYWQ1YjUyMTU0NWU3CnVybGxpYjM9PTIuNi4zIC0taGFzaD1zaGEyNTY6YmYyNzIzMjNlNTUzZGZiMmU4N2Q5YmZkMjI1Y2E3YjBmNDY3YjkxOWQ3YmJkMzU1NDM2ZDNmZDM3Y2IwYWNkNAp3ZXJremV1Zz09My4xLjggLS1oYXNoPXNoYTI1Njo2M2E3N2ZiODg5MmJmMjhlYmMzMTc4NjgzNDQ1MjIyYWE1MDBlNDhlYmFkNWVjNzdiMGFkODBmODcyNmIxZjUwCg==', 'research/cpu_competition_v1/runtime.py': 'IiIiU2VxdWVudGlhbCwgYm91bmRlZCBDUFUgY29tcGV0aXRpb24gd29ya2VyOyBuZXZlciBpbXBvcnRzIG9yIGluc3RhbGxzIGEgbW9kZWwuIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgbWF0aAppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzaWduYWwKaW1wb3J0IHRocmVhZGluZwppbXBvcnQgdGltZQpmcm9tIHR5cGVzIGltcG9ydCBTaW1wbGVOYW1lc3BhY2UKCmltcG9ydCByZXF1ZXN0cwpmcm9tIHJlcXVlc3RzLmFkYXB0ZXJzIGltcG9ydCBIVFRQQWRhcHRlcgpmcm9tIHVybGxpYjMudXRpbC5yZXRyeSBpbXBvcnQgUmV0cnkKZnJvbSBhcmNlbmdpbmUgaW1wb3J0IEdhbWVTdGF0ZQpmcm9tIGFnZW50LmFjdGlvbiBpbXBvcnQgQWN0aW9uRGVjaXNpb24KZnJvbSBhZ2VudC5jb250cm9sbGVyIGltcG9ydCBEZXRlcm1pbmlzdGljRmFsbGJhY2sKZnJvbSBhZ2VudC5mcmFtZXdvcmtfYWRhcHRlciBpbXBvcnQgT3V0Y29tZVVua25vd24sIFJlbW90ZUZyYW1ld29ya0FkYXB0ZXIKZnJvbSByZXNlYXJjaC5pbml0aWFsX2dlb21ldHJ5X2Fzc2lzdGFuY2VfdjEuZ2VvbWV0cnkgaW1wb3J0IGRpZ2VzdApmcm9tIHJlc2VhcmNoLmxzMjBfb3ZlcmZpdF92MS5wb2xpY3kgaW1wb3J0IE92ZXJmaXRQb2xpY3kKCgpjbGFzcyBEZWFkbGluZVNlc3Npb24ocmVxdWVzdHMuU2Vzc2lvbik6CiAgICAiIiJObyByZXRyaWVzL3JlZGlyZWN0cywgd2l0aCBvbmUgYWJzb2x1dGUgZGVhZGxpbmUgb3ZlciB0aGUgY29tcGxldGUgcmVzcG9uc2UuIiIiCiAgICBkZWYgX19pbml0X18oc2VsZiwgZGVhZGxpbmUsICosIG5vdz10aW1lLm1vbm90b25pYyk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5kZWFkbGluZSwgc2VsZi5ub3cgPSBkZWFkbGluZSwgbm93CiAgICAgICAgcmV0cnkgPSBSZXRyeSh0b3RhbD0wLCBjb25uZWN0PTAsIHJlYWQ9MCwgcmVkaXJlY3Q9MCwgc3RhdHVzPTApCiAgICAgICAgc2VsZi5tb3VudCgnaHR0cDovLycsIEhUVFBBZGFwdGVyKG1heF9yZXRyaWVzPXJldHJ5KSkKICAgICAgICBzZWxmLm1vdW50KCdodHRwczovLycsIEhUVFBBZGFwdGVyKG1heF9yZXRyaWVzPXJldHJ5KSkKCiAgICBkZWYgcmVxdWVzdChzZWxmLCBtZXRob2QsIHVybCwgKiprd2FyZ3MpOgogICAgICAgIGlmIHRocmVhZGluZy5jdXJyZW50X3RocmVhZCgpIGlzIG5vdCB0aHJlYWRpbmcubWFpbl90aHJlYWQoKSBvciBub3QgaGFzYXR0cihzaWduYWwsICdzZXRpdGltZXInKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdhYnNvbHV0ZSBIVFRQIGRlYWRsaW5lcyByZXF1aXJlIHRoZSBzZXF1ZW50aWFsIExpbnV4IG1haW4gdGhyZWFkJykKICAgICAgICBpZiBzaWduYWwuZ2V0aXRpbWVyKHNpZ25hbC5JVElNRVJfUkVBTCkgIT0gKDAuMCwgMC4wKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCd1bmV4cGVjdGVkIGFjdGl2ZSBhbGFybSBpbiBDUFUgSFRUUCB3b3JrZXInKQogICAgICAgIGJvdW5kID0gc2VsZi5kZWFkbGluZSgpIGlmIGNhbGxhYmxlKHNlbGYuZGVhZGxpbmUpIGVsc2Ugc2VsZi5kZWFkbGluZQogICAgICAgIGRlYWRsaW5lID0gbWluKGJvdW5kLCBzZWxmLm5vdygpICsgMjAuMCkKICAgICAgICByZW1haW5pbmcgPSBkZWFkbGluZSAtIHNlbGYubm93KCkKICAgICAgICBpZiBub3QgbWF0aC5pc2Zpbml0ZShyZW1haW5pbmcpIG9yIHJlbWFpbmluZyA8PSAwOgogICAgICAgICAgICByYWlzZSBUaW1lb3V0RXJyb3IoJ0hUVFAgYWRtaXNzaW9uIGNsb3NlZCcpCiAgICAgICAga3dhcmdzLnVwZGF0ZSh0aW1lb3V0PXJlbWFpbmluZywgYWxsb3dfcmVkaXJlY3RzPUZhbHNlLCBzdHJlYW09RmFsc2UpCiAgICAgICAgcHJldmlvdXMgPSBzaWduYWwuZ2V0c2lnbmFsKHNpZ25hbC5TSUdBTFJNKQogICAgICAgIGRlZiBleHBpcmVkKF9zaWdudW0sIF9mcmFtZSk6CiAgICAgICAgICAgIHJhaXNlIFRpbWVvdXRFcnJvcignYWJzb2x1dGUgSFRUUCByZXNwb25zZSBkZWFkbGluZScpCiAgICAgICAgc2lnbmFsLnNpZ25hbChzaWduYWwuU0lHQUxSTSwgZXhwaXJlZCkKICAgICAgICBzaWduYWwuc2V0aXRpbWVyKHNpZ25hbC5JVElNRVJfUkVBTCwgcmVtYWluaW5nKQogICAgICAgIHRyeToKICAgICAgICAgICAgcmVzcG9uc2UgPSBzdXBlcigpLnJlcXVlc3QobWV0aG9kLCB1cmwsICoqa3dhcmdzKQogICAgICAgICAgICBpZiBzZWxmLm5vdygpID49IGRlYWRsaW5lOgogICAgICAgICAgICAgICAgcmVzcG9uc2UuY2xvc2UoKQogICAgICAgICAgICAgICAgcmFpc2UgVGltZW91dEVycm9yKCdIVFRQIHJlc3BvbnNlIGNvbXBsZXRlZCBsYXRlJykKICAgICAgICAgICAgcmV0dXJuIHJlc3BvbnNlCiAgICAgICAgZmluYWxseToKICAgICAgICAgICAgc2lnbmFsLnNldGl0aW1lcihzaWduYWwuSVRJTUVSX1JFQUwsIDApCiAgICAgICAgICAgIHNpZ25hbC5zaWduYWwoc2lnbmFsLlNJR0FMUk0sIHByZXZpb3VzKQoKCmRlZiBub3JtYWxpemUob2JzZXJ2YXRpb24pOgogICAgIiIiVGhlIGZpdHRlZCBwb2xpY3kgc2VlcyB0aGUgZmluYWwgZGlzcGxheWVkIGJvYXJkLCB3aXRoIG5vIEdVSUQgb3IgbWV0YWRhdGEuIiIiCiAgICBsZWdhbCA9IGxpc3Qob2JzZXJ2YXRpb24uYXZhaWxhYmxlX2FjdGlvbnMpCiAgICBpZiAoYW55KHR5cGUoYSkgaXMgbm90IGludCBvciBub3QgMCA8PSBhIDw9IDcgZm9yIGEgaW4gbGVnYWwpIG9yCiAgICAgICAgICAgIGxlbihsZWdhbCkgIT0gbGVuKHNldChsZWdhbCkpIG9yIHR5cGUob2JzZXJ2YXRpb24ubGV2ZWxzX2NvbXBsZXRlZCkgaXMgbm90IGludCBvcgogICAgICAgICAgICBvYnNlcnZhdGlvbi5sZXZlbHNfY29tcGxldGVkIDwgMCBvciBub3QgaXNpbnN0YW5jZShvYnNlcnZhdGlvbi5zdGF0ZSwgR2FtZVN0YXRlKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignaW52YWxpZCBvYnNlcnZhdGlvbiBjb250cmFjdCcpCiAgICByZXR1cm4gZGljdChncmlkPW9ic2VydmF0aW9uLmxhdGVzdF9mcmFtZS50b2xpc3QoKSwgc3RhdGU9b2JzZXJ2YXRpb24uc3RhdGUudmFsdWUsCiAgICAgICAgICAgICAgICBsZXZlbHNfY29tcGxldGVkPW9ic2VydmF0aW9uLmxldmVsc19jb21wbGV0ZWQsIGxlZ2FsX2FjdGlvbnM9c29ydGVkKGxlZ2FsKSkKCgpjbGFzcyBfQWRtaXNzaW9uQ2xvc2VkKEV4Y2VwdGlvbik6CiAgICBwYXNzCgoKZGVmIHJ1bihhZGFwdGVyLCBwcm9ncmFtLCAqLCB3b3JrX2RlYWRsaW5lLCBub3c9dGltZS5tb25vdG9uaWMsCiAgICAgICAgbWF4X2dhbWVzPTExMCwgbWF4X2FjdGlvbnM9ODAsIGdhdGU9Tm9uZSk6CiAgICAiIiJPbmUgc2NvcmVjYXJkIGFuZCBvbmUgYm9vdHN0cmFwIHBlciBnYW1lOyBhbWJpZ3VvdXMgd29yayBpcyBuZXZlciByZXRyaWVkLiIiIgogICAgaWYgKHR5cGUobWF4X2dhbWVzKSBpcyBub3QgaW50IG9yIG5vdCAxIDw9IG1heF9nYW1lcyA8PSAxMTAgb3IKICAgICAgICAgICAgdHlwZShtYXhfYWN0aW9ucykgaXMgbm90IGludCBvciBub3QgMSA8PSBtYXhfYWN0aW9ucyA8PSA4MCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignQ1BVIGNvbXBldGl0aW9uIGxpbWl0cyBkaWZmZXInKQogICAgT3ZlcmZpdFBvbGljeShwcm9ncmFtKSAgIyBWYWxpZGF0ZSBiZWZvcmUgdGhlIGZpcnN0IHByb3ZpZGVyLWZhY2luZyBvcGVyYXRpb24uCiAgICBkZWYgYWRtaXQoKToKICAgICAgICBpZiBnYXRlIGlzIG5vdCBOb25lOgogICAgICAgICAgICBnYXRlKCkKICAgICAgICBpZiBub3coKSA+PSB3b3JrX2RlYWRsaW5lOgogICAgICAgICAgICByYWlzZSBfQWRtaXNzaW9uQ2xvc2VkKCkKICAgIGFkbWl0KCkKICAgIGdhbWVzID0gYWRhcHRlci5saXN0X2dhbWVfaWRzKCkKICAgIGlmIChub3QgaXNpbnN0YW5jZShnYW1lcywgKGxpc3QsIHR1cGxlKSkgb3Igbm90IGdhbWVzIG9yIGxlbihnYW1lcykgPiBtYXhfZ2FtZXMgb3IKICAgICAgICAgICAgYW55KHR5cGUoZ2FtZSkgaXMgbm90IHN0ciBvciBub3QgZ2FtZS5zdHJpcCgpIGZvciBnYW1lIGluIGdhbWVzKSBvcgogICAgICAgICAgICBsZW4oZ2FtZXMpICE9IGxlbihzZXQoZ2FtZXMpKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignaW52YWxpZCBjb21wZXRpdGlvbiBpbnZlbnRvcnknKQogICAgc3VtbWFyeSA9IGRpY3QoYXZhaWxhYmxlX2dhbWVzPWxlbihnYW1lcyksIGF0dGVtcHRlZF9nYW1lcz0wLCBmaXR0ZWRfZ2FtZXM9MCwKICAgICAgICBmaXR0ZWRfdGhyZWVfbGV2ZWxfY29tcGxldGlvbnM9MCwgbGV2ZWxzX2NvbXBsZXRlZD0wLCBhY3Rpb25zX2FkbWl0dGVkPTAsCiAgICAgICAgYWN0aW9uc19hY2tub3dsZWRnZWQ9MCwgYW1iaWd1b3VzX2FjdGlvbnM9MCwgYm9vdHN0cmFwX2ZhaWx1cmVzPTAsCiAgICAgICAgZml0dGVkX21pc21hdGNoZXM9MCwgc3RvcHBlZF9nYW1lcz0wLCBjbG9zZWRfdmVyaWZpZWQ9RmFsc2UsCiAgICAgICAgZmluYWxpemF0aW9uX3N0YXR1cz0nbm90X29wZW5lZCcsIG1vZGVsX2dlbmVyYXRpb25zPTAsIGF1dG9tYXRpY19yZXRyaWVzPTApCiAgICBvcGVuZWQgPSBGYWxzZQogICAgdHJ5OgogICAgICAgIGFkbWl0KCkKICAgICAgICBhZGFwdGVyLm9wZW5fc2NvcmVjYXJkKHRhZ3M9WydjcHUtbHMyMC1tZW1vcml6ZWQtcGx1cy1lMC12MSddKQogICAgICAgIG9wZW5lZCA9IFRydWUKICAgICAgICBmb3IgZ2FtZV9pZCBpbiBnYW1lczoKICAgICAgICAgICAgYWRtaXQoKQogICAgICAgICAgICBjbGllbnQsIG9ic2VydmF0aW9uLCBmaXR0ZWQgPSBOb25lLCBOb25lLCBOb25lCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHN1bW1hcnlbJ2F0dGVtcHRlZF9nYW1lcyddICs9IDEKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBjbGllbnQgPSBhZGFwdGVyLmJvb3RzdHJhcChnYW1lX2lkKQogICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgICAgICBzdW1tYXJ5Wydib290c3RyYXBfZmFpbHVyZXMnXSArPSAxCiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIG9ic2VydmF0aW9uID0gY2xpZW50Lm9ic2VydmF0aW9uCiAgICAgICAgICAgICAgICBpZiBvYnNlcnZhdGlvbi5nYW1lX2lkICE9IGdhbWVfaWQgb3IgdHlwZShvYnNlcnZhdGlvbi5ndWlkKSBpcyBub3Qgc3RyIG9yIG5vdCBvYnNlcnZhdGlvbi5ndWlkOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2Jvb3RzdHJhcCBpZGVudGl0eSBkaWZmZXJzJykKICAgICAgICAgICAgICAgIGluaXRpYWwgPSBub3JtYWxpemUob2JzZXJ2YXRpb24pCiAgICAgICAgICAgICAgICBpZiBnYW1lX2lkID09IHByb2dyYW1bJ2dhbWVfaWQnXSBhbmQgZGlnZXN0KGluaXRpYWwpID09IGRpZ2VzdChwcm9ncmFtWydpbml0aWFsX29ic2VydmF0aW9uJ10pOgogICAgICAgICAgICAgICAgICAgIGZpdHRlZCA9IE92ZXJmaXRQb2xpY3kocHJvZ3JhbSkKICAgICAgICAgICAgICAgICAgICBzdW1tYXJ5WydmaXR0ZWRfZ2FtZXMnXSArPSAxCiAgICAgICAgICAgICAgICBiYXNlbGluZSA9IERldGVybWluaXN0aWNGYWxsYmFjaygpCiAgICAgICAgICAgICAgICBjYXAgPSBsZW4ocHJvZ3JhbVsnc3RlcHMnXSkgaWYgZml0dGVkIGlzIG5vdCBOb25lIGVsc2UgbWF4X2FjdGlvbnMKICAgICAgICAgICAgICAgIGZvciBpbmRleCBpbiByYW5nZShjYXApOgogICAgICAgICAgICAgICAgICAgIGlmIG9ic2VydmF0aW9uLnN0YXRlIGluIChHYW1lU3RhdGUuV0lOLCBHYW1lU3RhdGUuR0FNRV9PVkVSKToKICAgICAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgICAgICBpZiBub3QgYW55KGFjdGlvbiAhPSAwIGZvciBhY3Rpb24gaW4gb2JzZXJ2YXRpb24uYXZhaWxhYmxlX2FjdGlvbnMpOgogICAgICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICAgICAgICAgIGFkbWl0KCkKICAgICAgICAgICAgICAgICAgICBpZiBmaXR0ZWQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgICAgIGFjdGlvbiA9IGZpdHRlZC5zZWxlY3QoZ2FtZV9pZCwgbm9ybWFsaXplKG9ic2VydmF0aW9uKSkKICAgICAgICAgICAgICAgICAgICAgICAgZGVjaXNpb24gPSBBY3Rpb25EZWNpc2lvbihhY3Rpb25faWQ9YWN0aW9uWydhY3Rpb25faWQnXSwgYWN0aW9uX2RhdGE9YWN0aW9uWydhY3Rpb25fZGF0YSddLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgc291cmNlPSdsczIwX21lbW9yaXplZF90aHJlZV9sZXZlbHMnLCBkZWNpc2lvbl9pZD1mJ2ZpdHRlZDp7aW5kZXh9JykKICAgICAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgICAgICBkZWNpc2lvbiA9IGJhc2VsaW5lLnByb3Bvc2UoU2ltcGxlTmFtZXNwYWNlKG9ic2VydmF0aW9uPW9ic2VydmF0aW9uKSkKICAgICAgICAgICAgICAgICAgICBpZiAodHlwZShkZWNpc2lvbi5hY3Rpb25faWQpIGlzIG5vdCBpbnQgb3IgZGVjaXNpb24uYWN0aW9uX2lkID09IDAgb3IKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRlY2lzaW9uLmFjdGlvbl9pZCBub3QgaW4gb2JzZXJ2YXRpb24uYXZhaWxhYmxlX2FjdGlvbnMpOgogICAgICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdwb2xpY3kgcHJvcG9zZWQgYSByZXNldCBvciBpbGxlZ2FsIGFjdGlvbicpCiAgICAgICAgICAgICAgICAgICAgYWRtaXQoKSAgIyBJbmNsdWRlcyB0aW1lIHNwZW50IHNlbGVjdGluZywgaW1tZWRpYXRlbHkgYmVmb3JlIGRpc3BhdGNoLgogICAgICAgICAgICAgICAgICAgIHN1bW1hcnlbJ2FjdGlvbnNfYWRtaXR0ZWQnXSArPSAxCiAgICAgICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgICAgICBhZnRlciA9IGFkYXB0ZXIuZGlzcGF0Y2goY2xpZW50LCBkZWNpc2lvbikKICAgICAgICAgICAgICAgICAgICBleGNlcHQgT3V0Y29tZVVua25vd246CiAgICAgICAgICAgICAgICAgICAgICAgIHN1bW1hcnlbJ2FtYmlndW91c19hY3Rpb25zJ10gKz0gMQogICAgICAgICAgICAgICAgICAgICAgICBjbGllbnQucXVhcmFudGluZWQgPSBUcnVlCiAgICAgICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICAgICAgc3VtbWFyeVsnYWN0aW9uc19hY2tub3dsZWRnZWQnXSArPSAxCiAgICAgICAgICAgICAgICAgICAgaWYgYWZ0ZXIuZ3VpZCAhPSBvYnNlcnZhdGlvbi5ndWlkIG9yIGFmdGVyLmdhbWVfaWQgIT0gZ2FtZV9pZDoKICAgICAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignYWNrbm93bGVkZ2VkIG9ic2VydmF0aW9uIGlkZW50aXR5IGRpZmZlcnMnKQogICAgICAgICAgICAgICAgICAgIG5vcm1hbGl6ZShhZnRlcikKICAgICAgICAgICAgICAgICAgICBvYnNlcnZhdGlvbiA9IGFmdGVyCiAgICAgICAgICAgICAgICAgICAgaWYgZml0dGVkIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgICAgICBmaXR0ZWQuYWNrbm93bGVkZ2UoYWN0aW9uLCBub3JtYWxpemUoYWZ0ZXIpKQogICAgICAgICAgICAgICAgICAgICAgICBpZiBmaXR0ZWQuY29tcGxldGVkOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgc3VtbWFyeVsnZml0dGVkX3RocmVlX2xldmVsX2NvbXBsZXRpb25zJ10gKz0gMQogICAgICAgICAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgZXhjZXB0IF9BZG1pc3Npb25DbG9zZWQ6CiAgICAgICAgICAgICAgICByYWlzZQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgaWYgZml0dGVkIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgIHN1bW1hcnlbJ2ZpdHRlZF9taXNtYXRjaGVzJ10gKz0gMQogICAgICAgICAgICAgICAgc3VtbWFyeVsnc3RvcHBlZF9nYW1lcyddICs9IDEKICAgICAgICAgICAgZmluYWxseToKICAgICAgICAgICAgICAgIGlmIG9ic2VydmF0aW9uIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgIHN1bW1hcnlbJ2xldmVsc19jb21wbGV0ZWQnXSArPSBvYnNlcnZhdGlvbi5sZXZlbHNfY29tcGxldGVkCiAgICAgICAgICAgICAgICBpZiBjbGllbnQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgICAgICBhZGFwdGVyLmZpbmFsaXplX2NsaWVudChjbGllbnQpCiAgICAgICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgICAgICAgICAgc3VtbWFyeVsnY2xpZW50X2NsZWFudXBfZmFpbGVkJ10gPSBUcnVlCiAgICAgICAgICAgICAgICAgICAgICAgIHJhaXNlCiAgICBleGNlcHQgX0FkbWlzc2lvbkNsb3NlZDoKICAgICAgICBzdW1tYXJ5WydhZG1pc3Npb25fY2xvc2VkJ10gPSBUcnVlCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICBzdW1tYXJ5WydlcnJvcl90eXBlJ10gPSB0eXBlKGV4YykuX19uYW1lX18KICAgIGZpbmFsbHk6CiAgICAgICAgaWYgb3BlbmVkOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBjbGVhbnVwID0gZ2V0YXR0cihhZGFwdGVyLCAnYmVnaW5fY2xlYW51cCcsIE5vbmUpCiAgICAgICAgICAgICAgICBpZiBjbGVhbnVwIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgIGNsZWFudXAoKQogICAgICAgICAgICAgICAgYWRhcHRlci5jbG9zZV9zY29yZWNhcmQoKQogICAgICAgICAgICAgICAgc3VtbWFyeS51cGRhdGUoY2xvc2VkX3ZlcmlmaWVkPW5vdCBzdW1tYXJ5LmdldCgnY2xpZW50X2NsZWFudXBfZmFpbGVkJywgRmFsc2UpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZmluYWxpemF0aW9uX3N0YXR1cz0nYWNrbm93bGVkZ2VkJykKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgICAgICAgICBzdW1tYXJ5LnVwZGF0ZShmaW5hbGl6YXRpb25fc3RhdHVzPSd1bmtub3duJywgZXJyb3JfdHlwZT10eXBlKGV4YykuX19uYW1lX18pCiAgICByZXR1cm4gc3VtbWFyeQoKCmRlZiBtYWluKCk6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1wcm9ncmFtJywgcmVxdWlyZWQ9VHJ1ZSwgdHlwZT1QYXRoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS13b3JrLXNlY29uZHMnLCB0eXBlPWludCwgZGVmYXVsdD0zMDAwKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1maW5hbC1zZWNvbmRzJywgdHlwZT1pbnQsIGRlZmF1bHQ9MzMwMCkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICBpZiBub3Qgb3MuZW52aXJvbi5nZXQoJ0tBR0dMRV9JU19DT01QRVRJVElPTl9SRVJVTicpOgogICAgICAgIHJhaXNlIFBlcm1pc3Npb25FcnJvcignQ1BVIGNvbXBldGl0aW9uIHdvcmtlciByZXF1aXJlcyB0aGUgc2NvcmVkIHJlcnVuJykKICAgIGlmIG5vdCAxIDw9IGFyZ3Mud29ya19zZWNvbmRzIDwgYXJncy5maW5hbF9zZWNvbmRzIDw9IDMzMDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignQ1BVIGRlYWRsaW5lIGVudmVsb3BlIGRpZmZlcnMnKQogICAgc3RhcnRlZCA9IHRpbWUubW9ub3RvbmljKCkKICAgIHdvcmtfZGVhZGxpbmUsIGZpbmFsX2RlYWRsaW5lID0gc3RhcnRlZCArIGFyZ3Mud29ya19zZWNvbmRzLCBzdGFydGVkICsgYXJncy5maW5hbF9zZWNvbmRzCiAgICBwaGFzZSA9IHsnY2xlYW51cCc6IEZhbHNlfQogICAgZGVhZGxpbmUgPSBsYW1iZGE6IGZpbmFsX2RlYWRsaW5lIGlmIHBoYXNlWydjbGVhbnVwJ10gZWxzZSB3b3JrX2RlYWRsaW5lCiAgICBhZGFwdGVyID0gUmVtb3RlRnJhbWV3b3JrQWRhcHRlcihiYXNlX3VybD0naHR0cDovL2dhdGV3YXk6ODAwMScsCiAgICAgICAgYXBpX2tleT0ndGVzdC1rZXktMTIzJywgY29tcGV0aXRpb25fbW9kZT1UcnVlLAogICAgICAgIHNlc3Npb25fZmFjdG9yeT1sYW1iZGE6IERlYWRsaW5lU2Vzc2lvbihkZWFkbGluZSkpCiAgICBhZGFwdGVyLmJlZ2luX2NsZWFudXAgPSBsYW1iZGE6IHBoYXNlLnVwZGF0ZShjbGVhbnVwPVRydWUpCiAgICB0cnk6CiAgICAgICAgcmVzdWx0ID0gcnVuKGFkYXB0ZXIsIGpzb24ubG9hZHMoYXJncy5wcm9ncmFtLnJlYWRfYnl0ZXMoKSksIHdvcmtfZGVhZGxpbmU9d29ya19kZWFkbGluZSkKICAgIGZpbmFsbHk6CiAgICAgICAgYWRhcHRlci5fbWFzdGVyX3Nlc3Npb24uY2xvc2UoKQogICAgcmVzdWx0WydlbGFwc2VkX3NlY29uZHMnXSA9IHRpbWUubW9ub3RvbmljKCkgLSBzdGFydGVkCiAgICBpZiB0aW1lLm1vbm90b25pYygpID49IGZpbmFsX2RlYWRsaW5lOgogICAgICAgIHJlc3VsdC51cGRhdGUoY2xvc2VkX3ZlcmlmaWVkPUZhbHNlLCBlcnJvcl90eXBlPSdGaW5hbERlYWRsaW5lRXhjZWVkZWQnKQogICAgcHJpbnQoanNvbi5kdW1wcyhyZXN1bHQsIHNvcnRfa2V5cz1UcnVlKSwgZmx1c2g9VHJ1ZSkKICAgIGlmIG5vdCByZXN1bHRbJ2Nsb3NlZF92ZXJpZmllZCddIG9yIHJlc3VsdC5nZXQoJ2Vycm9yX3R5cGUnKToKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KDEpCgoKaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzoKICAgIG1haW4oKQo=', 'research/cpu_competition_v1/supervisor.py': 'IiIiT2ZmbGluZSBDUFUgaW5zdGFsbCBhbmQgYm91bmRlZCB3b3JrZXI7IGltcG9ydHMgdGhlIGdhbWUgU0RLIG9ubHkgYWZ0ZXIgaW5zdGFsbC4iIiIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgc2h1dGlsCmltcG9ydCBzeXMKaW1wb3J0IHRlbXBmaWxlCmltcG9ydCB0aW1lCgoKZGVmIGVudHJ5KHJvb3QsIHdoZWVscywgKiwgc2NvcmVkLCBmaW5hbGl6ZT1sYW1iZGEgY29tcGxldGVkOiBOb25lKToKICAgIGZyb20gcmVzZWFyY2guaW5pdGlhbF9nZW9tZXRyeV9hc3Npc3RhbmNlX3YxLmxpdmVfcmVzb3VyY2VzIGltcG9ydCBIYXJkRGVhZGxpbmUsIFByb2Nlc3NlcywgZW52aXJvbm1lbnQKICAgIGZyb20gcmVzZWFyY2guaW5pdGlhbF9nZW9tZXRyeV9hc3Npc3RhbmNlX3YxLmxpdmVfaW8gaW1wb3J0IHNoYQogICAgcm9vdCwgd2hlZWxzID0gUGF0aChyb290KS5yZXNvbHZlKCksIFBhdGgod2hlZWxzKS5yZXNvbHZlKCkKICAgIGlmIHN5cy52ZXJzaW9uX2luZm9bOjJdICE9ICgzLCAxMik6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCd0aGlzIHBpbm5lZCBDUFUgcGFja2FnZSByZXF1aXJlcyBQeXRob24gMy4xMicpCiAgICBzdGFydGVkID0gdGltZS5tb25vdG9uaWMoKQogICAgZmluYWwgPSBzdGFydGVkKzMzMDAKICAgIHdvcmsgPSBmaW5hbC0zMDAKICAgIHdhdGNoZXIgPSBIYXJkRGVhZGxpbmUoZmluYWwpCiAgICB3YXRjaGVyLnN0YXJ0KCkKICAgIHByb2Nlc3NlcyA9IFByb2Nlc3Nlcyh3YXRjaGVyKQogICAgdGVtcG9yYXJ5ID0gUGF0aCh0ZW1wZmlsZS5ta2R0ZW1wKHByZWZpeD0nYXJjMy1jcHUtcnVudGltZS0nKSkKICAgIGVudiA9IGRpY3QoZW52aXJvbm1lbnQoKSwgUFlUSE9OUEFUSD1zdHIocm9vdCksIENVREFfVklTSUJMRV9ERVZJQ0VTPScnLAogICAgICAgICAgICAgICBOVklESUFfVklTSUJMRV9ERVZJQ0VTPSd2b2lkJywgUFlUSE9ORE9OVFdSSVRFQllURUNPREU9JzEnKQogICAgaWYgc2NvcmVkOgogICAgICAgIGVudlsnS0FHR0xFX0lTX0NPTVBFVElUSU9OX1JFUlVOJ109JzEnCiAgICBjb21wbGV0ZWQgPSBGYWxzZQogICAgdHJ5OgogICAgICAgIHBpbnMgPSBqc29uLmxvYWRzKChyb290LydyZXNlYXJjaC9jcHVfY29tcGV0aXRpb25fdjEvd2hlZWxzLmpzb24nKS5yZWFkX2J5dGVzKCkpCiAgICAgICAgZm9yIG5hbWUsIHBpbiBpbiBwaW5zLml0ZW1zKCk6CiAgICAgICAgICAgIGlmIHRpbWUubW9ub3RvbmljKCkgPj0gd29yazoKICAgICAgICAgICAgICAgIHJhaXNlIFRpbWVvdXRFcnJvcignd2hlZWwgdmVyaWZpY2F0aW9uIGRlYWRsaW5lJykKICAgICAgICAgICAgcGF0aCA9IHdoZWVscy9uYW1lCiAgICAgICAgICAgIGlmIHBhdGguaXNfc3ltbGluaygpIG9yIG5vdCBwYXRoLmlzX2ZpbGUoKSBvciBwYXRoLnN0YXQoKS5zdF9zaXplICE9IHBpblsnYnl0ZXMnXSBvciBzaGEocGF0aCkgIT0gcGluWydzaGEyNTYnXToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2NvbXBldGl0aW9uIENQVSB3aGVlbCBkaWZmZXJzOiAnK25hbWUpCiAgICAgICAgcHl0aG9uID0gdGVtcG9yYXJ5Lyd2ZW52L2Jpbi9weXRob24nCiAgICAgICAgbG9nID0gdGVtcG9yYXJ5Lyd3b3JrZXIubG9nJwogICAgICAgIHByb2Nlc3Nlcy5jb21tYW5kKFtzeXMuZXhlY3V0YWJsZSwnLW0nLCd2ZW52JywnLS13aXRob3V0LXBpcCcsc3RyKHRlbXBvcmFyeS8ndmVudicpXSwKICAgICAgICAgICAgbG9nLG1pbih3b3JrLHRpbWUubW9ub3RvbmljKCkrMTIwKSxlbnYsJ0NQVSB2ZW52IHdpdGhvdXQgZW5zdXJlcGlwJykKICAgICAgICBwcm9jZXNzZXMuY29tbWFuZChbc3lzLmV4ZWN1dGFibGUsJy1tJywncGlwJywnLS1weXRob24nLHN0cihweXRob24pLCdpbnN0YWxsJywKICAgICAgICAgICAgJy0tbm8taW5kZXgnLCctLW5vLWNhY2hlLWRpcicsJy0tcmVxdWlyZS1oYXNoZXMnLCctLW9ubHktYmluYXJ5PTphbGw6JywKICAgICAgICAgICAgJy0tZmluZC1saW5rcycsc3RyKHdoZWVscyksJy1yJyxzdHIocm9vdC8ncmVzZWFyY2gvY3B1X2NvbXBldGl0aW9uX3YxL3JlcXVpcmVtZW50cy5sb2NrJyldLAogICAgICAgICAgICBsb2csbWluKHdvcmssdGltZS5tb25vdG9uaWMoKSs2MDApLGVudiwnQ29tcGV0aXRpb24tcHJvdmlkZWQgQ1BVIHdoZWVscycpCiAgICAgICAgIyBObyBnYXRld2F5IGNhbGxzIGluIHZhbGlkYXRpb24uIFRlc3RzIHVzZSBzeW50aGV0aWMgYWRhcHRlcnMgb25seS4KICAgICAgICBpZiBub3Qgc2NvcmVkOgogICAgICAgICAgICB0ZXN0X2Jvb3RzdHJhcCA9ICgnaW1wb3J0IHN5cyxydW5weTsgc3lzLnBhdGguaW5zZXJ0KDAsc3lzLmFyZ3YucG9wKDEpKTsgJwogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAncnVucHkucnVuX21vZHVsZSgidW5pdHRlc3QiLHJ1bl9uYW1lPSJfX21haW5fXyIsYWx0ZXJfc3lzPVRydWUpJykKICAgICAgICAgICAgcmVzdWx0ID0gcHJvY2Vzc2VzLmNvbW1hbmQoW3N0cihweXRob24pLCctQicsJy1jJyx0ZXN0X2Jvb3RzdHJhcCxzdHIocm9vdCksCiAgICAgICAgICAgICAgICAndGVzdHMudGVzdF9jcHVfY29tcGV0aXRpb25fdjEnLCd0ZXN0cy50ZXN0X2NwdV9jb21wZXRpdGlvbl9wYWNrYWdpbmdfdjEnLAogICAgICAgICAgICAgICAgJ3Rlc3RzLnRlc3RfbHMyMF9vdmVyZml0X3YxJ10sCiAgICAgICAgICAgICAgICBsb2csbWluKHdvcmssdGltZS5tb25vdG9uaWMoKSsxMjApLGVudiwnQ1BVIHN5bnRoZXRpYyBjb250cmFjdHMnKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGxlZnRfd29yayA9IGludCh3b3JrLXRpbWUubW9ub3RvbmljKCkpCiAgICAgICAgICAgIGxlZnRfZmluYWwgPSBpbnQoZmluYWwtNjAtdGltZS5tb25vdG9uaWMoKSkKICAgICAgICAgICAgaWYgbGVmdF93b3JrIDw9IDA6CiAgICAgICAgICAgICAgICByYWlzZSBUaW1lb3V0RXJyb3IoJ25vIENQVSBnYW1lIGFkbWlzc2lvbiBidWRnZXQgcmVtYWlucycpCiAgICAgICAgICAgIHJlc3VsdCA9IHByb2Nlc3Nlcy5jb21tYW5kKFtzdHIocHl0aG9uKSwnLUInLHN0cihyb290LydyZXNlYXJjaC9jcHVfY29tcGV0aXRpb25fdjEvcnVudGltZS5weScpLAogICAgICAgICAgICAgICAgJy0tcHJvZ3JhbScsc3RyKHJvb3QvJ3Jlc2VhcmNoL2xzMjBfb3ZlcmZpdF92MS9wb2xpY3lfc2VlZDAuanNvbicpLAogICAgICAgICAgICAgICAgJy0td29yay1zZWNvbmRzJyxzdHIobGVmdF93b3JrKSwnLS1maW5hbC1zZWNvbmRzJyxzdHIobGVmdF9maW5hbCldLAogICAgICAgICAgICAgICAgbG9nLGZpbmFsLTMwLGVudiwnU2luZ2xlLXNjb3JlY2FyZCBDUFUgY29tcGV0aXRpb24gYWdlbnQnKQogICAgICAgIHByaW50KHJlc3VsdC5zdGRvdXRbLTMwMDA6XSxmbHVzaD1UcnVlKQogICAgICAgIGNvbXBsZXRlZCA9IFRydWUKICAgIGZpbmFsbHk6CiAgICAgICAgcmVjZWlwdD1wcm9jZXNzZXMuc3RvcChmaW5hbCkKICAgICAgICBpZiByZWNlaXB0Wydncm91cHNfYWJzZW50J10gaXMgbm90IFRydWUgb3IgcmVjZWlwdFsnZXJyb3InXSBvciByZWNlaXB0WydpbnRlcnJ1cHRlZCddOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0NQVSBwcm9jZXNzIGNsZWFudXAgdW52ZXJpZmllZCcpCiAgICAgICAgc2h1dGlsLnJtdHJlZSh0ZW1wb3JhcnkpCiAgICAgICAgaWYgdGltZS5tb25vdG9uaWMoKSA+PSBmaW5hbDoKICAgICAgICAgICAgcmFpc2UgVGltZW91dEVycm9yKCdDUFUgY2xlYW51cCBkZWFkbGluZScpCiAgICAgICAgdHJ5OgogICAgICAgICAgICBmaW5hbGl6ZShjb21wbGV0ZWQpCiAgICAgICAgZmluYWxseToKICAgICAgICAgICAgaWYgdGltZS5tb25vdG9uaWMoKSA+PSBmaW5hbDoKICAgICAgICAgICAgICAgIHJhaXNlIFRpbWVvdXRFcnJvcignQ1BVIGFydGlmYWN0L3NvdXJjZSBmaW5hbGl6YXRpb24gZGVhZGxpbmUnKQogICAgICAgICAgICB3YXRjaGVyLmNsb3NlKCkKICAgICAgICAgICAgaWYgdGltZS5tb25vdG9uaWMoKSA+PSBmaW5hbDoKICAgICAgICAgICAgICAgIHJhaXNlIFRpbWVvdXRFcnJvcignQ1BVIGZpbmFsIGRlYWRsaW5lJykKICAgIHByaW50KGpzb24uZHVtcHMoZGljdChjcHVfbGlmZWN5Y2xlX2Nsb3NlZD1UcnVlLGVsYXBzZWRfc2Vjb25kcz10aW1lLm1vbm90b25pYygpLXN0YXJ0ZWQsCiAgICAgICAgICAgICAgICAgICAgICAgICBHUFVfam9icz0wLG1vZGVsX2dlbmVyYXRpb25zPTApKSxmbHVzaD1UcnVlKQo=', 'research/cpu_competition_v1/wheels.json': 'ew0KICJhbm5vdGF0ZWRfdHlwZXMtMC43LjAtcHkzLW5vbmUtYW55LndobCI6IHsNCiAgImJ5dGVzIjogMTM2NDMsDQogICJzaGEyNTYiOiAiMWYwMmU4YjQzYThmYmJjM2YzZTBkNGYwZjRiZmM4MTMxYmNiNGVlYmU4ODQ5YjhlNWM3NzNmM2ExYzU4MmE1MyINCiB9LA0KICJhcmNfYWdpLTAuOS44LXB5My1ub25lLWFueS53aGwiOiB7DQogICJieXRlcyI6IDQwMTUxLA0KICAic2hhMjU2IjogImFlY2ExNjYzZGIzNDJlOTFjYjhmYzk2Y2YwYzgzZTJmYTM5ZGIxNjQwYmMxMjk4ZTgwYThkNTIyZTNhZjcwZjMiDQogfSwNCiAiYXJjZW5naW5lLTAuOS4zLXB5My1ub25lLWFueS53aGwiOiB7DQogICJieXRlcyI6IDM4Mzc0LA0KICAic2hhMjU2IjogIjVmOTczOWQ2ZDAwNTU3ODBhNDU4MWZkNmZlMDkwNjZiYjA4Nzc1YzRjODIxMmM5YWRjY2EyZWIwMDhhZWY1OWMiDQogfSwNCiAiYmxpbmtlci0xLjkuMC1weTMtbm9uZS1hbnkud2hsIjogew0KICAiYnl0ZXMiOiA4NDU4LA0KICAic2hhMjU2IjogImJhMGVmYWE5MDgwYjYxOWZmMmYzNDU5ZDFkNTAwYzU3YmRkZWE0YTZiNDI0YjYwYTkxMTQxZGI2ZmQyZjA4YmMiDQogfSwNCiAiY2VydGlmaS0yMDI2LjIuMjUtcHkzLW5vbmUtYW55LndobCI6IHsNCiAgImJ5dGVzIjogMTUzNjg0LA0KICAic2hhMjU2IjogIjAyNzY5MmU0NDAyYWQ5OTRmMWM0MmU1MmE0OTk3YTk3NjNjNjQ2YjczZTQwOTZlNGQ1ZDZkYjhhZjFkNmYwZmEiDQogfSwNCiAiY2hhcnNldF9ub3JtYWxpemVyLTMuNC43LWNwMzEyLWNwMzEyLW1hbnlsaW51eDIwMTRfeDg2XzY0Lm1hbnlsaW51eF8yXzE3X3g4Nl82NC5tYW55bGludXhfMl8yOF94ODZfNjQud2hsIjogew0KICAiYnl0ZXMiOiAyMTY1ODksDQogICJzaGEyNTYiOiAiNTY0OWZkMWM3YmFkZTAyZjMyMGE0NjJmZGVmZDBiNGJkM2NlMDM2MDY1ODM2ZDRmNDJlMGRlOTU4MDM4ZTExNiINCiB9LA0KICJjbGljay04LjMuMi1weTMtbm9uZS1hbnkud2hsIjogew0KICAiYnl0ZXMiOiAxMDgzNzksDQogICJzaGEyNTYiOiAiMTkyNGQyYzI3YzU2NTM1NjFjZDJjYWU0NTQ4ZDE0MDYwMzljYjc5Yjg1OGI3NDdjZmVhMjQ5MjRiYmMxNjE2ZCINCiB9LA0KICJjb250b3VycHktMS4zLjMtY3AzMTItY3AzMTItbWFueWxpbnV4XzJfMjdfeDg2XzY0Lm1hbnlsaW51eF8yXzI4X3g4Nl82NC53aGwiOiB7DQogICJieXRlcyI6IDM2MjYwMSwNCiAgInNoYTI1NiI6ICI0ZDAwZTY1NWZjZWYwOGFiYTM1ZWM5NjEwNTM2YmZlOTAyNjdkN2FiNWJhOTQ0ZjcwMzI1NDljNTVhMTQ2ZGExIg0KIH0sDQogImN5Y2xlci0wLjEyLjEtcHkzLW5vbmUtYW55LndobCI6IHsNCiAgImJ5dGVzIjogODMyMSwNCiAgInNoYTI1NiI6ICI4NWNlZjdjZmYyMjJkODY0NDE2MTUyOTgwODQ2NTk3MmU1MTM0MDU5OTQ1OWI4YWMzY2NiYWM1YTg1NGUwZDMwIg0KIH0sDQogImZsYXNrLTMuMS4zLXB5My1ub25lLWFueS53aGwiOiB7DQogICJieXRlcyI6IDEwMzQyNCwNCiAgInNoYTI1NiI6ICJmNGJjYmVmYzEyNDI5MTkyNWYxYTI2NDQ2ZGEzMWE1MTc4Zjk0ODM4NjIyMzNiMjNjMGM5NmEyMDcwMWY2NzBjIg0KIH0sDQogImZvbnR0b29scy00LjYyLjEtY3AzMTItY3AzMTItbWFueWxpbnV4MjAxNF94ODZfNjQubWFueWxpbnV4XzJfMTdfeDg2XzY0LndobCI6IHsNCiAgImJ5dGVzIjogNDk4ODc2OCwNCiAgInNoYTI1NiI6ICIxNDlmN2Q4NGFmY2E2NTlkMWE5N2UzOWE0Nzc4Nzk0YTJmODNiZjM0NGM1ZWU1MTM0ZTA5OTk1MDg2Y2MyMzkyIg0KIH0sDQogImlkbmEtMy4xMS1weTMtbm9uZS1hbnkud2hsIjogew0KICAiYnl0ZXMiOiA3MTAwOCwNCiAgInNoYTI1NiI6ICI3NzFhODdmNDlkOWRlZmFmNjQwOTFlNmU2ZmU5YzE4ZDQ4MzNmMTQwYmQxOTQ2NDc5NWJjMzJkOTY2Y2EzN2VhIg0KIH0sDQogIml0c2Rhbmdlcm91cy0yLjIuMC1weTMtbm9uZS1hbnkud2hsIjogew0KICAiYnl0ZXMiOiAxNjIzNCwNCiAgInNoYTI1NiI6ICJjNjI0MmZjNDllMzU5NThjOGIxNTE0MTM0M2FhNjYwZGI1ZmM1NGQ0ZjEzYTFkYjAxYTNmNTg5MWI5ODcwMGVmIg0KIH0sDQogImppbmphMi0zLjEuNi1weTMtbm9uZS1hbnkud2hsIjogew0KICAiYnl0ZXMiOiAxMzQ4OTksDQogICJzaGEyNTYiOiAiODVlY2U0NDUxZjQ5MmQwYzEzYzVkZDdjMTNhNjQ2ODFhODZhZmFlNjNhNWYzNDc5MDhkYWYxMDNjZTZkMmY2NyINCiB9LA0KICJraXdpc29sdmVyLTEuNS4wLWNwMzEyLWNwMzEyLW1hbnlsaW51eDIwMTRfeDg2XzY0Lm1hbnlsaW51eF8yXzE3X3g4Nl82NC53aGwiOiB7DQogICJieXRlcyI6IDE0Nzc5MzQsDQogICJzaGEyNTYiOiAiYmI1MTM2ZmI1MzUyZDNmNDIyZGYzM2YwYzg3OWExYjBjMjA0MDA0MzI0MTUwY2MzYjVlM2M0ZjMxMGM5MDQ5ZiINCiB9LA0KICJtYXJrdXBzYWZlLTMuMC4zLWNwMzEyLWNwMzEyLW1hbnlsaW51eDIwMTRfeDg2XzY0Lm1hbnlsaW51eF8yXzE3X3g4Nl82NC5tYW55bGludXhfMl8yOF94ODZfNjQud2hsIjogew0KICAiYnl0ZXMiOiAyMjk0NywNCiAgInNoYTI1NiI6ICJkNmRkMGJlNWI1YjE4OWQzMWRiN2NkYTQ4YjkxZDdlMGE5Nzk1ZjMxNDMwYjdmMjcxMjE5YWIzMGYxZDNhYzlkIg0KIH0sDQogIm1hdHBsb3RsaWItMy4xMC44LWNwMzEyLWNwMzEyLW1hbnlsaW51eDIwMTRfeDg2XzY0Lm1hbnlsaW51eF8yXzE3X3g4Nl82NC53aGwiOiB7DQogICJieXRlcyI6IDg3MTY5NDQsDQogICJzaGEyNTYiOiAiM2FiNGFhYmM3MmRlNGZmNzdiM2VjMzNhNmQ3OGE2ODIyN2JmMTEyMzQ2NTg4N2Y5OTA1YmE3OTE4NGExY2MwNCINCiB9LA0KICJudW1weS0yLjQuNC1jcDMxMi1jcDMxMi1tYW55bGludXhfMl8yN194ODZfNjQubWFueWxpbnV4XzJfMjhfeDg2XzY0LndobCI6IHsNCiAgImJ5dGVzIjogMTY2NDU0ODAsDQogICJzaGEyNTYiOiAiODFmNGExNGJlZTQ3YWVjNTRmODgzZTBjYWQyZDczOTg2NjQwYzE1OTBlYjliZmFhYmE3YWQxNzM5NDQ4MWU2ZSINCiB9LA0KICJwYWNrYWdpbmctMjYuMS1weTMtbm9uZS1hbnkud2hsIjogew0KICAiYnl0ZXMiOiA5NTgzMSwNCiAgInNoYTI1NiI6ICI1ZDljMDY2OWM2Mjg1ZTQ5MWUwY2VkMmVlZTU4N2VhZjY3YjY3MGQ5NGExOWU5NGUzOTg0YTQ4MWFiYTY4MDJmIg0KIH0sDQogInBpbGxvdy0xMi4yLjAtY3AzMTItY3AzMTItbWFueWxpbnV4XzJfMjdfeDg2XzY0Lm1hbnlsaW51eF8yXzI4X3g4Nl82NC53aGwiOiB7DQogICJieXRlcyI6IDcwODcyMTUsDQogICJzaGEyNTYiOiAiNjJmNTQwOTMzNmFkYjA2NjNiN2NhYTBkYTVjN2Q5ZTdiZGJhYWU5Y2U3NjFkMzQ2Njk0MjBjMmE4MDFiMjc4MCINCiB9LA0KICJweWRhbnRpYy0yLjEzLjItcHkzLW5vbmUtYW55LndobCI6IHsNCiAgImJ5dGVzIjogNDcxOTQ3LA0KICAic2hhMjU2IjogImE1MjUwODdmNGMwM2Q3ZTc0NTZhM2RlODliNjRjZDY5M2QyMjI5OTMzYmIxMDY4YjlhZjZiZWZkNTU2MzY5NGUiDQogfSwNCiAicHlkYW50aWNfY29yZS0yLjQ2LjItY3AzMTItY3AzMTItbWFueWxpbnV4XzJfMTdfeDg2XzY0Lm1hbnlsaW51eDIwMTRfeDg2XzY0LndobCI6IHsNCiAgImJ5dGVzIjogMjA5NDA1OCwNCiAgInNoYTI1NiI6ICI0ODY0OWNmMmQ4YzM1OGQ3OTU4NmU5ZmIyZjgyMzU5MDJmY2FhMmQ5NjllYzFjNTMwMWYyZDE4NzNiMmY4MzIxIg0KIH0sDQogInB5cGFyc2luZy0zLjMuMi1weTMtbm9uZS1hbnkud2hsIjogew0KICAiYnl0ZXMiOiAxMjI3ODEsDQogICJzaGEyNTYiOiAiODUwYmExNDhiZDkwOGQ3ZTI0MTE1ODdlMjQ3YTFlNGYwMzI3ODM5YzQwZTJlNWU2ZDA1YTAwN2VjYzY5OTExZCINCiB9LA0KICJweXRob25fZGF0ZXV0aWwtMi45LjAucG9zdDAtcHkyLnB5My1ub25lLWFueS53aGwiOiB7DQogICJieXRlcyI6IDIyOTg5MiwNCiAgInNoYTI1NiI6ICJhOGIyYmM3YmZmYWUyODIyODFjODE0MGE5N2QzYWE5YzE0ZGEwYjEzNmRmZTgzZjg1MGVlYTlhNWY3NDcwNDI3Ig0KIH0sDQogInB5dGhvbl9kb3RlbnYtMS4yLjItcHkzLW5vbmUtYW55LndobCI6IHsNCiAgImJ5dGVzIjogMjIxMDEsDQogICJzaGEyNTYiOiAiMWQ4MjE0Nzg5YTI0ZGU0NTVhOGI4YmQ4YWU2ZmUzYzZiNjlhNWUzZDY0YWE4YThlNWQ2OGU2OTRiYmNiMjg1YSINCiB9LA0KICJyZXF1ZXN0cy0yLjMzLjEtcHkzLW5vbmUtYW55LndobCI6IHsNCiAgImJ5dGVzIjogNjQ5NDcsDQogICJzaGEyNTYiOiAiNGU2ZDFlZjQ2MmYzNjI2YTFmMGEwYTljNDJkZDkzYzYzYmFkMzNmOWYxYzE5Mzc1MDliOGM1Yzg3MThhYjU2YSINCiB9LA0KICJzaXgtMS4xNy4wLXB5Mi5weTMtbm9uZS1hbnkud2hsIjogew0KICAiYnl0ZXMiOiAxMTA1MCwNCiAgInNoYTI1NiI6ICI0NzIxZjM5MWVkOTA1NDFmZGRhY2FiNWFjZjk0N2FhMGQzZGM3ZDI3YjJlMWU4ZWRhMmJlODk3MDU4NmMzMjc0Ig0KIH0sDQogInR5cGluZ19leHRlbnNpb25zLTQuMTUuMC1weTMtbm9uZS1hbnkud2hsIjogew0KICAiYnl0ZXMiOiA0NDYxNCwNCiAgInNoYTI1NiI6ICJmMGZhMTljNjg0NTc1OGFiMDgwNzRhMGNmYThiN2FlY2I3MWM5OTljYTczZDYyODgzYmMyNWNjMDE4YzRlNTQ4Ig0KIH0sDQogInR5cGluZ19pbnNwZWN0aW9uLTAuNC4yLXB5My1ub25lLWFueS53aGwiOiB7DQogICJieXRlcyI6IDE0NjExLA0KICAic2hhMjU2IjogIjRlZDFjYWNiZGMyOThjMjIwZjFiZDI0OWVkNTI4N2NhYTE2ZjM0ZDQ0ZWY0ZTljM2QwY2JhZDViNTIxNTQ1ZTciDQogfSwNCiAidXJsbGliMy0yLjYuMy1weTMtbm9uZS1hbnkud2hsIjogew0KICAiYnl0ZXMiOiAxMzE1ODQsDQogICJzaGEyNTYiOiAiYmYyNzIzMjNlNTUzZGZiMmU4N2Q5YmZkMjI1Y2E3YjBmNDY3YjkxOWQ3YmJkMzU1NDM2ZDNmZDM3Y2IwYWNkNCINCiB9LA0KICJ3ZXJremV1Zy0zLjEuOC1weTMtbm9uZS1hbnkud2hsIjogew0KICAiYnl0ZXMiOiAyMjY0NTksDQogICJzaGEyNTYiOiAiNjNhNzdmYjg4OTJiZjI4ZWJjMzE3ODY4MzQ0NTIyMmFhNTAwZTQ4ZWJhZDVlYzc3YjBhZDgwZjg3MjZiMWY1MCINCiB9DQp9DQo=', 'research/initial_geometry_assistance_v1/__init__.py': 'IiIiTmV3IGRldmVsb3BtZW50IHN0dWR5OiBpbml0aWFsIGdlb21ldHJ5IGFzc2lzdGFuY2UgaW4gc2hvcnQgb2ZmbGluZSBlcGlzb2Rlcy4KCkJvdW5kZWQgbGl2ZSBwYWNrYWdpbmcgYW5kIENQVSByZWhlYXJzYWwuIE5vIGRlcGxveW1lbnQvY29tcHV0ZSBhdXRob3JpdHkgaXMgaW1wbGllZC4KIiIiCg==', 'research/initial_geometry_assistance_v1/geometry.py': 'IiIiTG9zc2xlc3MsIGxhYmVsLWZyZWUgZ2VvbWV0cnkgc2hhcmVkIGJ5IGJvdGggZXhwZXJpbWVudGFsIGFybXMuIiIiCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCgoKZGVmIGNhbm9uaWNhbCh2YWx1ZSk6CiAgICByZXR1cm4gKGpzb24uZHVtcHModmFsdWUsIHNvcnRfa2V5cz1UcnVlLCBzZXBhcmF0b3JzPSgnLCcsICc6JyksIGFsbG93X25hbj1GYWxzZSkgKyAnXG4nKS5lbmNvZGUoKQoKCmRlZiBkaWdlc3QodmFsdWUpOgogICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KGNhbm9uaWNhbCh2YWx1ZSkpLmhleGRpZ2VzdCgpCgoKZGVmIGNoZWNrX2dyaWQoZ3JpZCk6CiAgICBpZiAodHlwZShncmlkKSBpcyBub3QgbGlzdCBvciBsZW4oZ3JpZCkgIT0gNjQgb3IKICAgICAgICAgICAgYW55KHR5cGUocm93KSBpcyBub3QgbGlzdCBvciBsZW4ocm93KSAhPSA2NCBvcgogICAgICAgICAgICAgICAgYW55KHR5cGUodikgaXMgbm90IGludCBvciBub3QgMCA8PSB2IDw9IDE1IGZvciB2IGluIHJvdykgZm9yIHJvdyBpbiBncmlkKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignZXhwZWN0ZWQgYSA2NCBieSA2NCBwYWxldHRlIGdyaWQnKQogICAgcmV0dXJuIGdyaWQKCgpkZWYgY2hlY2tlZF9jZWxscyh2YWx1ZXMpOgogICAgaWYgdHlwZSh2YWx1ZXMpIGlzIG5vdCBsaXN0OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ21hc2sgY2VsbHMgbXVzdCBiZSBhIGxpc3QnKQogICAgb3V0ID0gW10KICAgIGZvciBwb2ludCBpbiB2YWx1ZXM6CiAgICAgICAgaWYgKHR5cGUocG9pbnQpIGlzIG5vdCBsaXN0IG9yIGxlbihwb2ludCkgIT0gMiBvcgogICAgICAgICAgICAgICAgYW55KHR5cGUodikgaXMgbm90IGludCBvciBub3QgMCA8PSB2IDwgNjQgZm9yIHYgaW4gcG9pbnQpKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignbWFzayBjb29yZGluYXRlIG11c3QgYmUgYW4gaW50ZWdlciBjZWxsJykKICAgICAgICBvdXQuYXBwZW5kKHR1cGxlKHBvaW50KSkKICAgIGlmIGxlbihzZXQob3V0KSkgIT0gbGVuKG91dCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignZHVwbGljYXRlIG1hc2sgY2VsbCcpCiAgICByZXR1cm4gc29ydGVkKG91dCwga2V5PWxhbWJkYSBwOiAocFsxXSwgcFswXSkpCgoKZGVmIGVuY29kZV9tYXNrKHZhbHVlcyk6CiAgICBjZWxscyA9IGNoZWNrZWRfY2VsbHModmFsdWVzKQogICAgcnVucyA9IFtdCiAgICBmb3IgeCwgeSBpbiBjZWxsczoKICAgICAgICBpZiBydW5zIGFuZCBydW5zWy0xXVswXSA9PSB5IGFuZCBydW5zWy0xXVsyXSA9PSB4IC0gMToKICAgICAgICAgICAgcnVuc1stMV1bMl0gPSB4CiAgICAgICAgZWxzZToKICAgICAgICAgICAgcnVucy5hcHBlbmQoW3ksIHgsIHhdKQogICAgaWYgbm90IGNlbGxzOgogICAgICAgIHJldHVybiB7J3Jvd19ydW5zX3lfeDBfeDFfaW5jbHVzaXZlJzogW10sICdiYm94X3h5eHlfaW5jbHVzaXZlJzogTm9uZSwgJ29jY3VwaWVkX2FuY2hvcl94eSc6IE5vbmV9CiAgICB4cywgeXMgPSB6aXAoKmNlbGxzKQogICAgIyBOZWFyZXN0IG9jY3VwaWVkIGNlbGwgdG8gdGhlIGNlbnRyb2lkLiBOZXZlciBhbiBlbXB0eSBnZW9tZXRyaWMgY2VudGVyLgogICAgbiwgc3gsIHN5ID0gbGVuKGNlbGxzKSwgc3VtKHhzKSwgc3VtKHlzKQogICAgYW5jaG9yID0gbWluKGNlbGxzLCBrZXk9bGFtYmRhIHA6ICgobipwWzBdLXN4KSoqMiArIChuKnBbMV0tc3kpKioyLCBwWzFdLCBwWzBdKSkKICAgIHJldHVybiB7J3Jvd19ydW5zX3lfeDBfeDFfaW5jbHVzaXZlJzogcnVucywKICAgICAgICAgICAgJ2Jib3hfeHl4eV9pbmNsdXNpdmUnOiBbbWluKHhzKSwgbWluKHlzKSwgbWF4KHhzKSwgbWF4KHlzKV0sCiAgICAgICAgICAgICdvY2N1cGllZF9hbmNob3JfeHknOiBsaXN0KGFuY2hvcil9CgoKZGVmIGRlY29kZV9tYXNrKG1hc2spOgogICAgaWYgdHlwZShtYXNrKSBpcyBub3QgZGljdCBvciBzZXQobWFzaykgIT0gewogICAgICAgICAgICAncm93X3J1bnNfeV94MF94MV9pbmNsdXNpdmUnLCAnYmJveF94eXh5X2luY2x1c2l2ZScsICdvY2N1cGllZF9hbmNob3JfeHknfToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCd1bmV4cGVjdGVkIGdlb21ldHJ5IGZpZWxkcycpCiAgICBydW5zID0gbWFza1sncm93X3J1bnNfeV94MF94MV9pbmNsdXNpdmUnXQogICAgaWYgdHlwZShydW5zKSBpcyBub3QgbGlzdDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdpbnZhbGlkIHJvdyBydW5zJykKICAgIGNlbGxzID0gW10KICAgIGZvciBydW4gaW4gcnVuczoKICAgICAgICBpZiAodHlwZShydW4pIGlzIG5vdCBsaXN0IG9yIGxlbihydW4pICE9IDMgb3IgYW55KHR5cGUodikgaXMgbm90IGludCBmb3IgdiBpbiBydW4pCiAgICAgICAgICAgICAgICBvciBub3QgKDAgPD0gcnVuWzBdIDwgNjQgYW5kIDAgPD0gcnVuWzFdIDw9IHJ1blsyXSA8IDY0KSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2ludmFsaWQgcm93IHJ1bicpCiAgICAgICAgeSwgbGVmdCwgcmlnaHQgPSBydW4KICAgICAgICBjZWxscy5leHRlbmQoW1t4LCB5XSBmb3IgeCBpbiByYW5nZShsZWZ0LCByaWdodCsxKV0pCiAgICBpZiBlbmNvZGVfbWFzayhjZWxscykgIT0gbWFzazoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdnZW9tZXRyeSBpcyBub3QgY2Fub25pY2FsIG9yIGhhcyBhIGZhbHNlIGFuY2hvci9ib3VuZGluZyBib3gnKQogICAgcmV0dXJuIGNlbGxzCgoKZGVmIGludmVudG9yeShtYXNrcyk6CiAgICAiIiJJbnB1dCBpcyAqb25seSogY2VsbCBsaXN0cywgc28gc291cmNlIGxhYmVscyBhbmQgcm9sZXMgY2Fubm90IGxlYWsuIiIiCiAgICBpZiB0eXBlKG1hc2tzKSBpcyBub3QgbGlzdDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdleHBlY3RlZCBtYXNrIGxpc3QnKQogICAgZW5jb2RlZCA9IHNvcnRlZCgoZW5jb2RlX21hc2soY2VsbHMpIGZvciBjZWxscyBpbiBtYXNrcyksIGtleT1jYW5vbmljYWwpCiAgICAjIER1cGxpY2F0ZS9lbXB0eSBwcm9wb3NhbHMgYXJlIHJldGFpbmVkOyB0aGVpciBjb3N0cyBhcmUgcGFydCBvZiBwZXJjZXB0aW9uLgogICAgcmV0dXJuIFt7J2lkJzogZidve2k6MDNkfScsICoqbWFza30gZm9yIGksIG1hc2sgaW4gZW51bWVyYXRlKGVuY29kZWQsIDEpXQoKCmRlZiBjaGVja19pbnZlbnRvcnkob2JqZWN0cyk6CiAgICBpZiB0eXBlKG9iamVjdHMpIGlzIG5vdCBsaXN0OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2V4cGVjdGVkIG9iamVjdCBsaXN0JykKICAgIG1hc2tzID0gW10KICAgIGZvciBpLCBvYmogaW4gZW51bWVyYXRlKG9iamVjdHMsIDEpOgogICAgICAgIGlmIHR5cGUob2JqKSBpcyBub3QgZGljdCBvciBvYmouZ2V0KCdpZCcpICE9IGYnb3tpOjAzZH0nOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdub24tbmV1dHJhbCBvYmplY3QgSUQnKQogICAgICAgIG1hc2tzLmFwcGVuZChkZWNvZGVfbWFzayh7azogdiBmb3IgaywgdiBpbiBvYmouaXRlbXMoKSBpZiBrICE9ICdpZCd9KSkKICAgIGlmIGludmVudG9yeShtYXNrcykgIT0gb2JqZWN0czoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdub25jYW5vbmljYWwgaW52ZW50b3J5JykKICAgIHJldHVybiBvYmplY3RzCg==', 'research/initial_geometry_assistance_v1/live_io.py': 'IiIiRHVyYWJsZSBldmlkZW5jZSBhbmQgYWJzb2x1dGUtZGVhZGxpbmUgcGlwZXM7IHN0YW5kYXJkIGxpYnJhcnkgb25seS4iIiIKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgc2VsZWN0b3JzCmltcG9ydCB0aW1lCgpmcm9tIC5nZW9tZXRyeSBpbXBvcnQgY2Fub25pY2FsCgoKZGVmIHNoYShwYXRoKToKICAgIHZhbHVlID0gaGFzaGxpYi5zaGEyNTYoKQogICAgd2l0aCBQYXRoKHBhdGgpLm9wZW4oJ3JiJykgYXMgc3RyZWFtOgogICAgICAgIGZvciBwYXJ0IGluIGl0ZXIobGFtYmRhOiBzdHJlYW0ucmVhZCg4ICogMTAyNCAqIDEwMjQpLCBiJycpOgogICAgICAgICAgICB2YWx1ZS51cGRhdGUocGFydCkKICAgIHJldHVybiB2YWx1ZS5oZXhkaWdlc3QoKQoKCmRlZiB3cml0ZV9uZXcocGF0aCwgdmFsdWUpOgogICAgd2l0aCBQYXRoKHBhdGgpLm9wZW4oJ3hiJykgYXMgc3RyZWFtOgogICAgICAgIHN0cmVhbS53cml0ZShjYW5vbmljYWwodmFsdWUpKQogICAgICAgIHN0cmVhbS5mbHVzaCgpCiAgICAgICAgb3MuZnN5bmMoc3RyZWFtLmZpbGVubygpKQoKCmNsYXNzIEpvdXJuYWw6CiAgICBkZWYgX19pbml0X18oc2VsZiwgcGF0aCk6CiAgICAgICAgc2VsZi5zdHJlYW0gPSBQYXRoKHBhdGgpLm9wZW4oJ3hiJywgYnVmZmVyaW5nPTApCgogICAgZGVmIF9fY2FsbF9fKHNlbGYsIGV2ZW50KToKICAgICAgICByYXcgPSBjYW5vbmljYWwoZXZlbnQpLnJzdHJpcChiJ1xuJykgKyBiJ1xuJwogICAgICAgICMgRmlsZUlPLndyaXRlIGNhbiBiZSBzaG9ydDsgYSBwYXJ0aWFsIHJlY29yZCBtdXN0IG5ldmVyIGFkbWl0IGFuIGVmZmVjdC4KICAgICAgICB2aWV3ID0gbWVtb3J5dmlldyhyYXcpCiAgICAgICAgd2hpbGUgdmlldzoKICAgICAgICAgICAgdXNlZCA9IHNlbGYuc3RyZWFtLndyaXRlKHZpZXcpCiAgICAgICAgICAgIGlmIG5vdCB1c2VkOgogICAgICAgICAgICAgICAgcmFpc2UgT1NFcnJvcignc2hvcnQgam91cm5hbCB3cml0ZScpCiAgICAgICAgICAgIHZpZXcgPSB2aWV3W3VzZWQ6XQogICAgICAgIG9zLmZzeW5jKHNlbGYuc3RyZWFtLmZpbGVubygpKQoKICAgIGRlZiBjbG9zZShzZWxmKToKICAgICAgICBzZWxmLnN0cmVhbS5jbG9zZSgpCgoKY2xhc3MgRGVhZGxpbmVQaXBlOgogICAgIiIiT25lIHJlcXVlc3QgLyBvbmUgcmVwbHksIG5vIHJldHJ5IGFuZCBubyB0aW1lb3V0IHJlc2V0IG9uIHRyaWNrbGVkIGJ5dGVzLiIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHJlYWRlciwgd3JpdGVyLCBsaW1pdD0xMDI0ICogMTAyNCk6CiAgICAgICAgc2VsZi5yZWFkZXIsIHNlbGYud3JpdGVyLCBzZWxmLmxpbWl0ID0gcmVhZGVyLCB3cml0ZXIsIGxpbWl0CiAgICAgICAgc2VsZi5idWZmZXIgPSBiJycKICAgICAgICBvcy5zZXRfYmxvY2tpbmcocmVhZGVyLmZpbGVubygpLCBGYWxzZSkKICAgICAgICBvcy5zZXRfYmxvY2tpbmcod3JpdGVyLmZpbGVubygpLCBGYWxzZSkKCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgd2FpdChmaWxlLCBldmVudCwgZGVhZGxpbmUpOgogICAgICAgIHJlbWFpbmluZyA9IGRlYWRsaW5lIC0gdGltZS5tb25vdG9uaWMoKQogICAgICAgIGlmIHJlbWFpbmluZyA8PSAwOgogICAgICAgICAgICByYWlzZSBUaW1lb3V0RXJyb3IoJ2VuZ2luZSBSUEMgZGVhZGxpbmUnKQogICAgICAgIHdpdGggc2VsZWN0b3JzLkRlZmF1bHRTZWxlY3RvcigpIGFzIHNlbGVjdG9yOgogICAgICAgICAgICBzZWxlY3Rvci5yZWdpc3RlcihmaWxlLCBldmVudCkKICAgICAgICAgICAgaWYgbm90IHNlbGVjdG9yLnNlbGVjdChyZW1haW5pbmcpIG9yIHRpbWUubW9ub3RvbmljKCkgPj0gZGVhZGxpbmU6CiAgICAgICAgICAgICAgICByYWlzZSBUaW1lb3V0RXJyb3IoJ2VuZ2luZSBSUEMgZGVhZGxpbmUnKQoKICAgIGRlZiBleGNoYW5nZShzZWxmLCB2YWx1ZSwgZGVhZGxpbmUpOgogICAgICAgIHJhdyA9IGNhbm9uaWNhbCh2YWx1ZSkucnN0cmlwKGInXG4nKSArIGInXG4nCiAgICAgICAgaWYgbGVuKHJhdykgPiBzZWxmLmxpbWl0OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdlbmdpbmUgcmVxdWVzdCB0b28gbGFyZ2UnKQogICAgICAgIHdoaWxlIHJhdzoKICAgICAgICAgICAgc2VsZi53YWl0KHNlbGYud3JpdGVyLCBzZWxlY3RvcnMuRVZFTlRfV1JJVEUsIGRlYWRsaW5lKQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICB1c2VkID0gb3Mud3JpdGUoc2VsZi53cml0ZXIuZmlsZW5vKCksIHJhdykKICAgICAgICAgICAgZXhjZXB0IEJsb2NraW5nSU9FcnJvcjoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGlmIG5vdCB1c2VkOgogICAgICAgICAgICAgICAgcmFpc2UgRU9GRXJyb3IoJ2VuZ2luZSBwaXBlIGNsb3NlZCBkdXJpbmcgd3JpdGUnKQogICAgICAgICAgICByYXcgPSByYXdbdXNlZDpdCiAgICAgICAgd2hpbGUgYidcbicgbm90IGluIHNlbGYuYnVmZmVyOgogICAgICAgICAgICBzZWxmLndhaXQoc2VsZi5yZWFkZXIsIHNlbGVjdG9ycy5FVkVOVF9SRUFELCBkZWFkbGluZSkKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgcGFydCA9IG9zLnJlYWQoc2VsZi5yZWFkZXIuZmlsZW5vKCksIDY1NTM2KQogICAgICAgICAgICBleGNlcHQgQmxvY2tpbmdJT0Vycm9yOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgaWYgbm90IHBhcnQ6CiAgICAgICAgICAgICAgICByYWlzZSBFT0ZFcnJvcignZW5naW5lIHBpcGUgY2xvc2VkIGR1cmluZyByZWFkJykKICAgICAgICAgICAgc2VsZi5idWZmZXIgKz0gcGFydAogICAgICAgICAgICBpZiBsZW4oc2VsZi5idWZmZXIpID4gc2VsZi5saW1pdDoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2VuZ2luZSByZXBseSB0b28gbGFyZ2UnKQogICAgICAgIHJhdywgc2VsZi5idWZmZXIgPSBzZWxmLmJ1ZmZlci5zcGxpdChiJ1xuJywgMSkKICAgICAgICBpZiBzZWxmLmJ1ZmZlciBvciB0aW1lLm1vbm90b25pYygpID49IGRlYWRsaW5lOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCd1bnNvbGljaXRlZCBvciBsYXRlIGVuZ2luZSByZXBseScpCiAgICAgICAgcmV0dXJuIGpzb24ubG9hZHMocmF3KQo=', 'research/initial_geometry_assistance_v1/live_resources.py': 'IiIiT3duZWQgZ3JvdXBzIHBsdXMgYW4gaW5kZXBlbmRlbnQsIG5vbi1jZW5zdXMgaGFyZC1kZWFkbGluZSB3YXRjaGRvZy4iIiIKaW1wb3J0IG11bHRpcHJvY2Vzc2luZwppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzaWduYWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHRpbWUKCmZyb20gY2VydGlmaWNhdGlvbi5kaXJlY3RfcHVibGlzaGVyX3Ntb2tlX3YxLmluc3RhbGwgaW1wb3J0IEluc3RhbGxhdGlvblByb2Nlc3Nlcwpmcm9tIGNlcnRpZmljYXRpb24uZGlyZWN0X3B1Ymxpc2hlcl9zbW9rZV92MS5zZXJ2ZXIgaW1wb3J0IE1vZGVsU2VydmVyLCBkZWZlcl9zdGFydHVwX3NpZ25hbHMKCgpkZWYgZW52aXJvbm1lbnQoaG9zdD1Ob25lKToKICAgIGhvc3QgPSBvcy5lbnZpcm9uIGlmIGhvc3QgaXMgTm9uZSBlbHNlIGhvc3QKICAgIHZhbHVlID0gaG9zdC5nZXQoJ0xEX0xJQlJBUllfUEFUSCcsICcnKQogICAgaWYgdHlwZSh2YWx1ZSkgaXMgbm90IHN0ciBvciBsZW4odmFsdWUpID4gMTYzODQgb3IgJ1wwJyBpbiB2YWx1ZSBvciAnXG4nIGluIHZhbHVlOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2RyaXZlciBsb2FkZXIgcGF0aCBjb250cmFjdCcpCiAgICBwYXJ0cyA9IHZhbHVlLnNwbGl0KG9zLnBhdGhzZXApCiAgICBpZiBsZW4ocGFydHMpID4gNjQ6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignZHJpdmVyIGxvYWRlciBwYXRoIGludmVudG9yeScpCiAgICBwYXRocyA9IGxpc3QoZGljdC5mcm9ta2V5cyhwIGZvciBwIGluIHBhcnRzIGlmIHAgYW5kIG9zLnBhdGguaXNhYnMocCkpKQogICAgZW52ID0geydQQVRIJzogaG9zdC5nZXQoJ1BBVEgnLCBvcy5kZWZwYXRoKSwgJ1BZVEhPTk5PVVNFUlNJVEUnOiAnMScsICdQSVBfTk9fSU5ERVgnOiAnMScsCiAgICAgICAgICAgJ1BJUF9DT05GSUdfRklMRSc6IG9zLmRldm51bGwsICdQSVBfRElTQUJMRV9QSVBfVkVSU0lPTl9DSEVDSyc6ICcxJywgJ0hGX0hVQl9PRkZMSU5FJzogJzEnLAogICAgICAgICAgICdUUkFOU0ZPUk1FUlNfT0ZGTElORSc6ICcxJywgJ0hGX0hVQl9ESVNBQkxFX1RFTEVNRVRSWSc6ICcxJywgJ01QTEJBQ0tFTkQnOiAnQWdnJywKICAgICAgICAgICAnVE9LRU5JWkVSU19QQVJBTExFTElTTSc6ICdmYWxzZScsICdUUkFOU0ZPUk1FUlNfTk9fVEYnOiAnMScsICdVU0VfVEYnOiAnMCcsCiAgICAgICAgICAgJ1ZMTE1fTk9fVVNBR0VfU1RBVFMnOiAnMScsICdDVURBX1ZJU0lCTEVfREVWSUNFUyc6ICcwJ30KICAgIGlmIHBhdGhzOgogICAgICAgIGVudlsnTERfTElCUkFSWV9QQVRIJ10gPSBvcy5wYXRoc2VwLmpvaW4ocGF0aHMpCiAgICAjIE5vIGFjY291bnQgY3JlZGVudGlhbHMsIGhvbWUgZGlyZWN0b3J5IG9yIGh1YiB0b2tlbnMgZW50ZXIgY2hpbGQgaW50ZXJwcmV0ZXJzLgogICAgcmV0dXJuIGVudgoKCmRlZiBfd2F0Y2gocGFyZW50LCBkZWFkbGluZSwgZ3JvdXBzLCBkb25lKToKICAgIGlmIG5vdCBkb25lLndhaXQobWF4KDAsIGRlYWRsaW5lLXRpbWUubW9ub3RvbmljKCkpKToKICAgICAgICAjIE5vIC9wcm9jIHNjYW4sIG5vIFB5dGhvbiBjbGVhbnVwIGNhbGxiYWNrLCBubyB3YWl0IGZvciBncmFjZWZ1bCBleGl0LgogICAgICAgIGZvciBwZ2lkIGluIGdyb3VwczoKICAgICAgICAgICAgaWYgcGdpZCA+IDA6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgb3Mua2lsbHBnKHBnaWQsIHNpZ25hbC5TSUdLSUxMKQogICAgICAgICAgICAgICAgZXhjZXB0IFByb2Nlc3NMb29rdXBFcnJvcjoKICAgICAgICAgICAgICAgICAgICBwYXNzCiAgICAgICAgdHJ5OgogICAgICAgICAgICBvcy5raWxsKHBhcmVudCwgc2lnbmFsLlNJR0tJTEwpCiAgICAgICAgZXhjZXB0IFByb2Nlc3NMb29rdXBFcnJvcjoKICAgICAgICAgICAgcGFzcwoKCmNsYXNzIEhhcmREZWFkbGluZToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBkZWFkbGluZSk6CiAgICAgICAgaWYgb3MubmFtZSAhPSAncG9zaXgnIG9yIG5vdCBQYXRoKCcvcHJvYycpLmlzX2RpcigpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdMaW51eCBwcm9jZXNzIG93bmVyc2hpcCByZXF1aXJlZCcpCiAgICAgICAgY3R4ID0gbXVsdGlwcm9jZXNzaW5nLmdldF9jb250ZXh0KCdzcGF3bicpCiAgICAgICAgc2VsZi5ncm91cHMgPSBjdHguUmF3QXJyYXkoJ3EnLCA2NCkKICAgICAgICBzZWxmLmRvbmUgPSBjdHguRXZlbnQoKQogICAgICAgIHNlbGYuZGVhZGxpbmUgPSBkZWFkbGluZQogICAgICAgIHNlbGYucHJvY2VzcyA9IGN0eC5Qcm9jZXNzKHRhcmdldD1fd2F0Y2gsIGFyZ3M9KG9zLmdldHBpZCgpLCBkZWFkbGluZSwgc2VsZi5ncm91cHMsIHNlbGYuZG9uZSkpCiAgICAgICAgc2VsZi5wcm9jZXNzLmRhZW1vbiA9IEZhbHNlCgogICAgZGVmIHN0YXJ0KHNlbGYpOgogICAgICAgIHdpdGggZGVmZXJfc3RhcnR1cF9zaWduYWxzKCk6CiAgICAgICAgICAgIHNlbGYucHJvY2Vzcy5zdGFydCgpCgogICAgZGVmIHJlZ2lzdGVyKHNlbGYsIGdyb3VwKToKICAgICAgICBpZiBncm91cC5wZ2lkIGlzIE5vbmUgb3IgZ3JvdXAub3duZXJzaGlwICE9ICdyZWdpc3RlcmVkJzoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigndW5jZXJ0YWluIG93bmVyc2hpcCBjYW5ub3QgYmUgcmVnaXN0ZXJlZCBhcyBhYnNlbnQnKQogICAgICAgIGZvciBpLCB2YWx1ZSBpbiBlbnVtZXJhdGUoc2VsZi5ncm91cHMpOgogICAgICAgICAgICBpZiB2YWx1ZSA9PSAwOgogICAgICAgICAgICAgICAgc2VsZi5ncm91cHNbaV0gPSBncm91cC5wZ2lkCiAgICAgICAgICAgICAgICByZXR1cm4gaQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ293bmVkIHByb2Nlc3MgaW52ZW50b3J5IGZ1bGwnKQoKICAgIGRlZiBjbGVhcihzZWxmLCBzbG90LCByZWNlaXB0KToKICAgICAgICBpZiByZWNlaXB0LmdldCgnZ3JvdXBzX2Fic2VudCcpIGlzIG5vdCBUcnVlOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdjYW5ub3QgZm9yZ2V0IGEgbGl2ZS91bmNlcnRhaW4gZ3JvdXAnKQogICAgICAgIHNlbGYuZ3JvdXBzW3Nsb3RdID0gMAoKICAgIGRlZiBjbG9zZShzZWxmKToKICAgICAgICBpZiBhbnkoc2VsZi5ncm91cHMpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdoYXJkIHdhdGNoZG9nIHN0aWxsIG93bnMgZ3JvdXBzJykKICAgICAgICBpZiB0aW1lLm1vbm90b25pYygpID49IHNlbGYuZGVhZGxpbmU6CiAgICAgICAgICAgIHJhaXNlIFRpbWVvdXRFcnJvcignaGFyZCBkZWFkbGluZSByZWFjaGVkJykKICAgICAgICBzZWxmLmRvbmUuc2V0KCkKICAgICAgICBzZWxmLnByb2Nlc3Muam9pbih0aW1lb3V0PW1pbigyLCBzZWxmLmRlYWRsaW5lLXRpbWUubW9ub3RvbmljKCkpKQogICAgICAgIGlmIHNlbGYucHJvY2Vzcy5pc19hbGl2ZSgpIG9yIHNlbGYucHJvY2Vzcy5leGl0Y29kZSAhPSAwOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ2hhcmQgd2F0Y2hkb2cgZGlkIG5vdCBzdG9wIGNsZWFubHknKQoKCmNsYXNzIFByb2Nlc3NlcyhJbnN0YWxsYXRpb25Qcm9jZXNzZXMpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHdhdGNoZG9nKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKHRlcm1pbmF0ZV9ncmFjZT0wLjIsIGtpbGxfZ3JhY2U9MSkKICAgICAgICBzZWxmLndhdGNoZG9nID0gd2F0Y2hkb2cKCiAgICBkZWYgY29tbWFuZChzZWxmLCBhcmd2LCBsb2csIGRlYWRsaW5lLCBlbnYsIHdoYXQpOgogICAgICAgIGlmIHRpbWUubW9ub3RvbmljKCkgPj0gZGVhZGxpbmU6CiAgICAgICAgICAgIHJhaXNlIFRpbWVvdXRFcnJvcih3aGF0ICsgJzogY29tbWFuZCBhZG1pc3Npb24gY2xvc2VkJykKICAgICAgICB3aXRoIFBhdGgobG9nKS5vcGVuKCdhYicpIGFzIHN0cmVhbToKICAgICAgICAgICAgc3RyZWFtLndyaXRlKCgnXG4kICcrJyAnLmpvaW4obWFwKHN0ciwgYXJndikpKydcbicpLmVuY29kZSgpKQogICAgICAgICAgICBvZmZzZXQgPSBzdHJlYW0udGVsbCgpCiAgICAgICAgZW50cnkgPSBOb25lCiAgICAgICAgIyBzcGF3biBhcHBlbmRzIGl0cyBvd25lciBiZWZvcmUgUG9wZW4sIGV2ZW4gaWYgaXQgbmV2ZXIgcmV0dXJucy4KICAgICAgICB0cnk6CiAgICAgICAgICAgIGVudHJ5ID0gc2VsZi5zcGF3bihhcmd2LCBsb2csIGVudiwgd2hhdCkKICAgICAgICAgICAgY29kZSA9IGVudHJ5Wydncm91cCddLnByb2Nlc3Mud2FpdCh0aW1lb3V0PW1heCgwLjAwMSwgZGVhZGxpbmUtdGltZS5tb25vdG9uaWMoKSkpCiAgICAgICAgZmluYWxseToKICAgICAgICAgICAgaWYgZW50cnkgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBjbGVhbnVwID0gc2VsZi5fc3RvcF9ncm91cChlbnRyeSwgZGVhZGxpbmUpCiAgICAgICAgaWYgbm90IGNsZWFudXBbJ2dyb3Vwc19hYnNlbnQnXSBvciBjbGVhbnVwLmdldCgnaW50ZXJydXB0ZWQnKSBvciB0aW1lLm1vbm90b25pYygpID49IGRlYWRsaW5lOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3Iod2hhdCArICc6IGNvbW1hbmQgbGlmZWN5Y2xlIG5vdCB2ZXJpZmllZCB3aXRoaW4gZGVhZGxpbmUnKQogICAgICAgIHdpdGggUGF0aChsb2cpLm9wZW4oJ3JiJykgYXMgc3RyZWFtOgogICAgICAgICAgICBzdHJlYW0uc2VlaygwLCBvcy5TRUVLX0VORCkKICAgICAgICAgICAgc3RyZWFtLnNlZWsobWF4KG9mZnNldCwgc3RyZWFtLnRlbGwoKS0yKjEwMjQqMTAyNCkpCiAgICAgICAgICAgIHRleHQgPSBzdHJlYW0ucmVhZCgpLmRlY29kZSgndXRmLTgnLCBlcnJvcnM9J3JlcGxhY2UnKQogICAgICAgIGlmIGNvZGU6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcih3aGF0ICsgJzogY29tbWFuZCBleGl0ZWQgJytzdHIoY29kZSkrJzogJyt0ZXh0Wy0zMDA6XSkKICAgICAgICByZXR1cm4gc3VicHJvY2Vzcy5Db21wbGV0ZWRQcm9jZXNzKGFyZ3YsIGNvZGUsIHN0ZG91dD10ZXh0LCBzdGRlcnI9JycpCgogICAgZGVmIF9zdG9wX2dyb3VwKHNlbGYsIGVudHJ5LCBkZWFkbGluZSk6CiAgICAgICAgcmVzdWx0ID0gc3VwZXIoKS5fc3RvcF9ncm91cChlbnRyeSwgZGVhZGxpbmUpCiAgICAgICAgaWYgcmVzdWx0Wydncm91cHNfYWJzZW50J10gYW5kICd3YXRjaF9zbG90JyBpbiBlbnRyeToKICAgICAgICAgICAgc2VsZi53YXRjaGRvZy5jbGVhcihlbnRyeS5wb3AoJ3dhdGNoX3Nsb3QnKSwgcmVzdWx0KQogICAgICAgIHJldHVybiByZXN1bHQKCiAgICBkZWYgc3Bhd24oc2VsZiwgYXJndiwgbG9nLCBlbnYsIHdoYXQpOgogICAgICAgIGVudHJ5ID0geydwaGFzZSc6IHdoYXQsICdncm91cCc6IE1vZGVsU2VydmVyKGFyZ3YsIGVudiwgbG9nLCBOb25lLCBOb25lLCBpbmhlcml0X2Vudj1GYWxzZSl9CiAgICAgICAgc2VsZi5ncm91cHMuYXBwZW5kKGVudHJ5KQogICAgICAgIHdpdGggZGVmZXJfc3RhcnR1cF9zaWduYWxzKCk6CiAgICAgICAgICAgIHNlbGYuX2Fkb3B0X2Rlc2NlbmRhbnRzKCkKICAgICAgICAgICAgZW50cnlbJ2dyb3VwJ10uc3RhcnQoKQogICAgICAgICAgICBlbnRyeVsnd2F0Y2hfc2xvdCddID0gc2VsZi53YXRjaGRvZy5yZWdpc3RlcihlbnRyeVsnZ3JvdXAnXSkKICAgICAgICByZXR1cm4gZW50cnkK', 'research/initial_geometry_assistance_v1/policy.py': 'IiIiRGV0ZXJtaW5pc3RpYyByZXF1ZXN0IGNvbnN0cnVjdGlvbiBhbmQgc3RyaWN0LCByZXBhaXItZnJlZSBhY3Rpb24gcGFyc2luZy4iIiIKaW1wb3J0IGpzb24KCmZyb20gY2VydGlmaWNhdGlvbi5waGFzZTRfbXVsdGltb2RhbF9wcmVmbGlnaHRfdjMuaW1hZ2VzIGltcG9ydCBkYXRhX3VybCwgcG5nCmZyb20gLmdlb21ldHJ5IGltcG9ydCBjaGVja19ncmlkLCBjaGVja19pbnZlbnRvcnksIGRpZ2VzdAoKU1lTVEVNID0gKAogICAgJ1lvdSBjb250cm9sIGFuIHVuZmFtaWxpYXIgcGl4ZWwgZ2FtZS4gVHJ5IHRvIGNvbXBsZXRlIGl0cyBmaXJzdCBsZXZlbCB1c2luZyB2aXNpYmxlIGZlZWRiYWNrLiAnCiAgICAnWW91IHJlY2VpdmUgbm8gZ2FtZSBydWxlcyBvciBnYW1lLXNwZWNpZmljIGNvbnRyb2wgZXhwbGFuYXRpb25zLiBPbmx5IGxpc3RlZCBhY3Rpb24gSURzIGFyZSBsZWdhbC4gJwogICAgJ0FDVElPTjYgYWxvbmUgdGFrZXMgaW50ZWdlciB4LHkgY29vcmRpbmF0ZXMgaW4gWzAsNjNdOyBhbGwgb3RoZXIgYWN0aW9ucyB0YWtlIGFuIGVtcHR5IGRhdGEgb2JqZWN0LiAnCiAgICAnRWFjaCBpbWFnZSBpcyBleGFjdGx5IDQ0OCBieSA0NDg6IG9yaWdpbmFsIGNlbGwgKHgseSkgb2NjdXBpZXMgcGl4ZWxzIFs3KngsNyp4KzZdIGJ5IFs3KnksNyp5KzZdLCAnCiAgICAnd2l0aCB4IGluY3JlYXNpbmcgcmlnaHQgYW5kIHkgZG93bi4gTm8gaW1hZ2UgaGFzIG1hcmdpbnMgb3Igb3ZlcmxheXMuICcKICAgICdUaGUgc3VwcGxpZWQgZ2VvbWV0cnkgYmVsb25ncyBPTkxZIHRvIHRoZSBpbml0aWFsIHN0ZXAtMCBpbWFnZS4gT2JqZWN0cyBtYXkgc2luY2UgaGF2ZSBtb3ZlZCwgJwogICAgJ2NoYW5nZWQgb3IgZGlzYXBwZWFyZWQ7IGRvIG5vdCB0cmVhdCBpdHMgY2VsbHMgYXMgY3VycmVudCBwb3NpdGlvbnMuIFRoZSBpbnZlbnRvcnkgY2FuIGJlIGluY29tcGxldGUgJwogICAgJ29yIHdyb25nLiBJdCBzdXBwbGllcyBnZW9tZXRyeSwgbm90IGNvbnRyb2wgbWVhbmluZ3Mgb3IgZ2FtZSBydWxlcy4gJwogICAgJ09ic2VydmUgdGhlIGN1cnJlbnQgaW1hZ2UgYW5kIHJlY2VudCB0cmFuc2l0aW9ucywgY2hvb3NlIG9uZSBhY3Rpb24sIGFuZCBwcmVkaWN0IHdoZXRoZXIgdGhlIG5leHQgJwogICAgJ2ZpbmFsIGltYWdlIHdpbGwgZGlmZmVyIGluIGFueSB2aXNpYmxlIGNlbGwgKGluY2x1ZGluZyBpbnRlcmZhY2UgY2hhbmdlcykuIFVzZSB1bmtub3duIGlmIHVuY2VydGFpbi4gJwogICAgJ1JldHVybiBPTkxZIGNvbXBhY3QgSlNPTiB3aXRoIGV4YWN0bHkgYWN0aW9uX2lkLCBhY3Rpb25fZGF0YSBhbmQgcHJlZGljdGVkX3Zpc2libGVfY2hhbmdlLiAnCiAgICAnRXhhbXBsZSBzeW50YXg6IHsiYWN0aW9uX2lkIjo2LCJhY3Rpb25fZGF0YSI6eyJ4IjoyMCwieSI6MzB9LCcKICAgICcicHJlZGljdGVkX3Zpc2libGVfY2hhbmdlIjoidW5rbm93biJ9LiBEbyBub3QgcmVzZXQsIGV4cGxhaW4sIG9yIG91dHB1dCBtdWx0aXBsZSBhY3Rpb25zLicKKQoKCmRlZiBjaGVja19vYnNlcnZhdGlvbihvYnNlcnZhdGlvbik6CiAgICBpZiB0eXBlKG9ic2VydmF0aW9uKSBpcyBub3QgZGljdCBvciBzZXQob2JzZXJ2YXRpb24pICE9IHsnZ3JpZCcsICdsZWdhbF9hY3Rpb25zJywgJ3N0YXRlJywgJ2xldmVsc19jb21wbGV0ZWQnfToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdvYnNlcnZhdGlvbiBjb250cmFjdCBkaWZmZXJzJykKICAgIGNoZWNrX2dyaWQob2JzZXJ2YXRpb25bJ2dyaWQnXSkKICAgIGFjdGlvbnMgPSBvYnNlcnZhdGlvblsnbGVnYWxfYWN0aW9ucyddCiAgICBpZiAodHlwZShhY3Rpb25zKSBpcyBub3QgbGlzdCBvciBsZW4oc2V0KGFjdGlvbnMpKSAhPSBsZW4oYWN0aW9ucykgb3IKICAgICAgICAgICAgYW55KHR5cGUodikgaXMgbm90IGludCBvciBub3QgMSA8PSB2IDw9IDcgZm9yIHYgaW4gYWN0aW9ucykpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2ludmFsaWQgbGVnYWwgYWN0aW9ucycpCiAgICBpZiBvYnNlcnZhdGlvblsnc3RhdGUnXSBub3QgaW4gKCdOT1RfRklOSVNIRUQnLCAnV0lOJywgJ0dBTUVfT1ZFUicpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2ludmFsaWQgZW5naW5lIHN0YXRlJykKICAgIGlmIHR5cGUob2JzZXJ2YXRpb25bJ2xldmVsc19jb21wbGV0ZWQnXSkgaXMgbm90IGludCBvciBvYnNlcnZhdGlvblsnbGV2ZWxzX2NvbXBsZXRlZCddIDwgMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdpbnZhbGlkIGxldmVsIGNvdW50ZXInKQogICAgcmV0dXJuIG9ic2VydmF0aW9uCgoKZGVmIHJlcXVlc3RfZm9yKGluaXRpYWwsIGdlb21ldHJ5LCBjdXJyZW50LCBoaXN0b3J5LCBzdGVwLCBtb2RlbCk6CiAgICBjaGVja19vYnNlcnZhdGlvbihpbml0aWFsKQogICAgY2hlY2tfb2JzZXJ2YXRpb24oY3VycmVudCkKICAgIGNoZWNrX2ludmVudG9yeShnZW9tZXRyeSkKICAgIGlmIHR5cGUoc3RlcCkgaXMgbm90IGludCBvciBub3QgMCA8PSBzdGVwIDwgMjQgb3IgbGVuKGhpc3RvcnkpICE9IHN0ZXA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignaGlzdG9yeS9zdGVwIG1pc21hdGNoJykKICAgIG1ldGEgPSB7J3N0ZXAnOiBzdGVwLCAnY3VycmVudF9sZWdhbF9hY3Rpb25zJzogY3VycmVudFsnbGVnYWxfYWN0aW9ucyddLAogICAgICAgICAgICAnY3VycmVudF9zdGF0ZSc6IGN1cnJlbnRbJ3N0YXRlJ10sICdjdXJyZW50X2xldmVsc19jb21wbGV0ZWQnOiBjdXJyZW50WydsZXZlbHNfY29tcGxldGVkJ10sCiAgICAgICAgICAgICdnZW9tZXRyeV9vYnNlcnZlZF9hdF9zdGVwJzogMCwgJ2dlb21ldHJ5X2luaXRpYWxfZ3JpZF9zaGEyNTYnOiBkaWdlc3QoaW5pdGlhbFsnZ3JpZCddKSwKICAgICAgICAgICAgJ2luaXRpYWxfb2JqZWN0cyc6IGdlb21ldHJ5LCAnb21pdHRlZF90cmFuc2l0aW9uX2NvdW50JzogbWF4KDAsIGxlbihoaXN0b3J5KS0yKX0KICAgIGNvbnRlbnQgPSBbeyd0eXBlJzogJ3RleHQnLCAndGV4dCc6IGpzb24uZHVtcHMobWV0YSwgc29ydF9rZXlzPVRydWUsIHNlcGFyYXRvcnM9KCcsJywgJzonKSl9XQoKICAgIGRlZiBpbWFnZShsYWJlbCwgZ3JpZCk6CiAgICAgICAgY29udGVudC5leHRlbmQoW3sndHlwZSc6ICd0ZXh0JywgJ3RleHQnOiBsYWJlbH0sCiAgICAgICAgICAgICAgICAgICAgICAgIHsndHlwZSc6ICdpbWFnZV91cmwnLCAnaW1hZ2VfdXJsJzogeyd1cmwnOiBkYXRhX3VybChwbmcoZ3JpZCwgY2VsbD03KSl9fV0pCgogICAgaW1hZ2UoJ0luaXRpYWwgaW1hZ2UsIHN0ZXAgMCAoaGlzdG9yaWNhbCk6JywgaW5pdGlhbFsnZ3JpZCddKQogICAgZm9yIG9sZCBpbiBoaXN0b3J5Wy0yOl06CiAgICAgICAgY29udGVudC5hcHBlbmQoeyd0eXBlJzogJ3RleHQnLCAndGV4dCc6IGpzb24uZHVtcHMoewogICAgICAgICAgICAnaGlzdG9yaWNhbF9zdGVwJzogb2xkWydzdGVwJ10sICdhY3Rpb24nOiBvbGRbJ2FjdGlvbiddLAogICAgICAgICAgICAnYmVmb3JlX3N0YXRlJzogb2xkWydiZWZvcmUnXVsnc3RhdGUnXSwgJ2FmdGVyX3N0YXRlJzogb2xkWydhZnRlciddWydzdGF0ZSddLAogICAgICAgICAgICAnYWZ0ZXJfbGV2ZWxzX2NvbXBsZXRlZCc6IG9sZFsnYWZ0ZXInXVsnbGV2ZWxzX2NvbXBsZXRlZCddfSwgc29ydF9rZXlzPVRydWUpfSkKICAgICAgICBpbWFnZSgnSGlzdG9yaWNhbCBiZWZvcmU6Jywgb2xkWydiZWZvcmUnXVsnZ3JpZCddKQogICAgICAgIGltYWdlKCdIaXN0b3JpY2FsIGFmdGVyOicsIG9sZFsnYWZ0ZXInXVsnZ3JpZCddKQogICAgaW1hZ2UoJ0N1cnJlbnQgaW1hZ2U7IGNob29zZSB0aGUgbmV4dCBhY3Rpb24gaGVyZTonLCBjdXJyZW50WydncmlkJ10pCiAgICByZXR1cm4geydtb2RlbCc6IG1vZGVsLCAnbWVzc2FnZXMnOiBbeydyb2xlJzogJ3N5c3RlbScsICdjb250ZW50JzogU1lTVEVNfSwgeydyb2xlJzogJ3VzZXInLCAnY29udGVudCc6IGNvbnRlbnR9XSwKICAgICAgICAgICAgJ3RlbXBlcmF0dXJlJzogMCwgJ3NlZWQnOiAwLCAnbWF4X3Rva2Vucyc6IDI1NiwKICAgICAgICAgICAgJ3Jlc3BvbnNlX2Zvcm1hdCc6IHsndHlwZSc6ICdqc29uX29iamVjdCd9LCAnY2hhdF90ZW1wbGF0ZV9rd2FyZ3MnOiB7J2VuYWJsZV90aGlua2luZyc6IEZhbHNlfX0KCgpkZWYgcGFyc2VfYW5zd2VyKHRleHQsIGxlZ2FsX2FjdGlvbnMpOgogICAgZGVmIHBhaXJzKGl0ZW1zKToKICAgICAgICBvdXQgPSB7fQogICAgICAgIGZvciBrLCB2IGluIGl0ZW1zOgogICAgICAgICAgICBpZiBrIGluIG91dDoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2R1cGxpY2F0ZSByZXNwb25zZSBrZXknKQogICAgICAgICAgICBvdXRba10gPSB2CiAgICAgICAgcmV0dXJuIG91dAogICAgaWYgdHlwZSh0ZXh0KSBpcyBub3Qgc3RyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ25vbi10ZXh0IGFuc3dlcicpCiAgICBhbnN3ZXIgPSBqc29uLmxvYWRzKHRleHQsIG9iamVjdF9wYWlyc19ob29rPXBhaXJzLAogICAgICAgICAgICAgICAgICAgICAgICBwYXJzZV9jb25zdGFudD1sYW1iZGEgXzogKF8gZm9yIF8gaW4gKCkpLnRocm93KFZhbHVlRXJyb3IoJ25vbi1maW5pdGUgSlNPTicpKSkKICAgIGlmIHR5cGUoYW5zd2VyKSBpcyBub3QgZGljdCBvciBzZXQoYW5zd2VyKSAhPSB7J2FjdGlvbl9pZCcsICdhY3Rpb25fZGF0YScsICdwcmVkaWN0ZWRfdmlzaWJsZV9jaGFuZ2UnfToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdyZXNwb25zZSBmaWVsZHMgZGlmZmVyJykKICAgIGFjdGlvbl9pZCwgZGF0YSA9IGFuc3dlclsnYWN0aW9uX2lkJ10sIGFuc3dlclsnYWN0aW9uX2RhdGEnXQogICAgaWYgdHlwZShhY3Rpb25faWQpIGlzIG5vdCBpbnQgb3IgYWN0aW9uX2lkIG5vdCBpbiBsZWdhbF9hY3Rpb25zIG9yIG5vdCAxIDw9IGFjdGlvbl9pZCA8PSA3OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2lsbGVnYWwgYWN0aW9uIElEJykKICAgIGlmIHR5cGUoZGF0YSkgaXMgbm90IGRpY3Qgb3Igc2V0KGRhdGEpICE9ICh7J3gnLCAneSd9IGlmIGFjdGlvbl9pZCA9PSA2IGVsc2Ugc2V0KCkpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2FjdGlvbiBkYXRhIGZpZWxkcyBkaWZmZXInKQogICAgaWYgYW55KHR5cGUodikgaXMgbm90IGludCBvciBub3QgMCA8PSB2IDwgNjQgZm9yIHYgaW4gZGF0YS52YWx1ZXMoKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignaW52YWxpZCBhY3Rpb24gY29vcmRpbmF0ZScpCiAgICBpZiBhbnN3ZXJbJ3ByZWRpY3RlZF92aXNpYmxlX2NoYW5nZSddIG5vdCBpbiAoJ3llcycsICdubycsICd1bmtub3duJyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignaW52YWxpZCBjaGFuZ2UgcHJlZGljdGlvbicpCiAgICByZXR1cm4gYW5zd2VyCg==', 'research/ls20_overfit_v1/__init__.py': 'IiIiRXhhY3Qtb2JzZXJ2YXRpb24gbWVtb3J5IGRpc3RpbGxlZCBmcm9tIG9uZSBsczIwIGh1bWFuIGRlbW9uc3RyYXRpb24uIiIiCgpmcm9tIC5wb2xpY3kgaW1wb3J0IE92ZXJmaXRQb2xpY3ksIGNvbXBpbGVfZGVtbwoKX19hbGxfXyA9IFsnT3ZlcmZpdFBvbGljeScsICdjb21waWxlX2RlbW8nXQo=', 'research/ls20_overfit_v1/policy.py': 'IiIiRGVtb25zdHJhdGlvbiBtZW1vcml6YXRpb247IG5vIG1vZGVsIGluZmVyZW5jZSBvciBpbmZlcnJlZCBnYW1lIHJ1bGVzLiIiIgpmcm9tIGNvcHkgaW1wb3J0IGRlZXBjb3B5CmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lZGVsdGEKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCByZQoKZnJvbSByZXNlYXJjaC5pbml0aWFsX2dlb21ldHJ5X2Fzc2lzdGFuY2VfdjEuZ2VvbWV0cnkgaW1wb3J0IGNoZWNrX2dyaWQsIGRpZ2VzdApmcm9tIHJlc2VhcmNoLmluaXRpYWxfZ2VvbWV0cnlfYXNzaXN0YW5jZV92MS5wb2xpY3kgaW1wb3J0IGNoZWNrX29ic2VydmF0aW9uCgpHQU1FX0lEID0gJ2xzMjAtOTYwNzYyN2InClNDSEVNQSA9ICdsczIwX292ZXJmaXRfdjEnCk1BWF9NT1ZFUyA9IDI1NgpRVUFMSUZJQ0FUSU9OID0gKAogICAgJ01lbW9yaXplZCBodW1hbiBkZW1vbnN0cmF0aW9uIG9uIGxzMjAtOTYwNzYyN2IsIHNlZWQgMCwgZmlyc3QgdGhyZWUgbGV2ZWxzIG9ubHkuICcKICAgICdFeGFjdCBvYnNlcnZhdGlvbiBtYXRjaGluZyBpbmNsdWRlcyB0aGUgdmlzaWJsZSB0aW1lIGJhci4gVGhpcyBpcyBub3QgYSBnZW5lcmFsIGxlYXJuZWQgcG9saWN5LicKKQoKCmRlZiBfc2hhKHZhbHVlKToKICAgIGlmIHR5cGUodmFsdWUpIGlzIG5vdCBzdHIgb3IgcmUuZnVsbG1hdGNoKCdbMC05YS1mXXs2NH0nLCB2YWx1ZSkgaXMgTm9uZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdpbnZhbGlkIFNIQTI1NicpCiAgICByZXR1cm4gdmFsdWUKCgpkZWYgX2FjdGlvbih2YWx1ZSk6CiAgICBpZiAodHlwZSh2YWx1ZSkgaXMgbm90IGRpY3Qgb3Igc2V0KHZhbHVlKSAhPSB7J2FjdGlvbl9pZCcsICdhY3Rpb25fZGF0YSd9IG9yCiAgICAgICAgICAgIHR5cGUodmFsdWVbJ2FjdGlvbl9pZCddKSBpcyBub3QgaW50IG9yIHZhbHVlWydhY3Rpb25faWQnXSBub3QgaW4gKDEsIDIsIDMsIDQpIG9yCiAgICAgICAgICAgIHR5cGUodmFsdWVbJ2FjdGlvbl9kYXRhJ10pIGlzIG5vdCBkaWN0IG9yIHZhbHVlWydhY3Rpb25fZGF0YSddKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdleHBlY3RlZCBvbmUgZGlyZWN0aW9uYWwgYWN0aW9uIHdpdGggZW1wdHkgZGF0YScpCiAgICByZXR1cm4gdmFsdWUKCgpkZWYgX3NuYXBzaG90KHZhbHVlKToKICAgIGNoZWNrX29ic2VydmF0aW9uKHZhbHVlKQogICAgaWYgdmFsdWVbJ2xlZ2FsX2FjdGlvbnMnXSAhPSBzb3J0ZWQodmFsdWVbJ2xlZ2FsX2FjdGlvbnMnXSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignbGVnYWwgYWN0aW9ucyBtdXN0IHVzZSB0aGUgZW5naW5lIHNuYXBzaG90IG9yZGVyaW5nJykKICAgIHJldHVybiB2YWx1ZQoKCmRlZiBfcmF3X3NuYXBzaG90KHZhbHVlKToKICAgIGV4cGVjdGVkID0geydnYW1lX2lkJywgJ3N0YXRlJywgJ2xldmVsc19jb21wbGV0ZWQnLCAnd2luX2xldmVscycsCiAgICAgICAgICAgICAgICAnYXZhaWxhYmxlX2FjdGlvbnMnLCAnZnJhbWVzJywgJ2Nhbm9uaWNhbF9oYXNoJ30KICAgIGlmIHR5cGUodmFsdWUpIGlzIG5vdCBkaWN0IG9yIHNldCh2YWx1ZSkgIT0gZXhwZWN0ZWQgb3IgdmFsdWVbJ2dhbWVfaWQnXSAhPSBHQU1FX0lEOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2RlbW9uc3RyYXRpb24gb2JzZXJ2YXRpb24gY29udHJhY3QgZGlmZmVycycpCiAgICBpZiB0eXBlKHZhbHVlWyd3aW5fbGV2ZWxzJ10pIGlzIG5vdCBpbnQgb3IgdmFsdWVbJ3dpbl9sZXZlbHMnXSAhPSA3OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2RlbW9uc3RyYXRpb24gd2lubmluZy1sZXZlbCBjb3VudCBkaWZmZXJzJykKICAgIF9zaGEodmFsdWVbJ2Nhbm9uaWNhbF9oYXNoJ10pCiAgICBpZiB0eXBlKHZhbHVlWydmcmFtZXMnXSkgaXMgbm90IGxpc3Qgb3Igbm90IHZhbHVlWydmcmFtZXMnXToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdkZW1vbnN0cmF0aW9uIGhhcyBubyB2aXNpYmxlIGZyYW1lJykKICAgIGZvciBmcmFtZSBpbiB2YWx1ZVsnZnJhbWVzJ106CiAgICAgICAgY2hlY2tfZ3JpZChmcmFtZSkKICAgIGFjdGlvbnMgPSB2YWx1ZVsnYXZhaWxhYmxlX2FjdGlvbnMnXQogICAgaWYgKHR5cGUoYWN0aW9ucykgaXMgbm90IGxpc3Qgb3IgYW55KHR5cGUoYSkgaXMgbm90IGludCBmb3IgYSBpbiBhY3Rpb25zKSBvcgogICAgICAgICAgICBsZW4oc2V0KGFjdGlvbnMpKSAhPSBsZW4oYWN0aW9ucykpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2RlbW9uc3RyYXRpb24gbGVnYWwgYWN0aW9ucyBkaWZmZXInKQogICAgcmV0dXJuIF9zbmFwc2hvdCh7J2dyaWQnOiB2YWx1ZVsnZnJhbWVzJ11bLTFdLCAnc3RhdGUnOiB2YWx1ZVsnc3RhdGUnXSwKICAgICAgICAgICAgICAgICAgICAgICdsZXZlbHNfY29tcGxldGVkJzogdmFsdWVbJ2xldmVsc19jb21wbGV0ZWQnXSwKICAgICAgICAgICAgICAgICAgICAgICdsZWdhbF9hY3Rpb25zJzogc29ydGVkKGFjdGlvbnMpfSkKCgpkZWYgX3V0Yyh2YWx1ZSk6CiAgICBpZiB0eXBlKHZhbHVlKSBpcyBub3Qgc3RyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2RlbW9uc3RyYXRpb24gdGltZXN0YW1wIGlzIG5vdCB0ZXh0JykKICAgIHRyeToKICAgICAgICBwYXJzZWQgPSBkYXRldGltZS5mcm9taXNvZm9ybWF0KHZhbHVlKQogICAgZXhjZXB0IFZhbHVlRXJyb3IgYXMgZXhjOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2ludmFsaWQgZGVtb25zdHJhdGlvbiB0aW1lc3RhbXAnKSBmcm9tIGV4YwogICAgaWYgcGFyc2VkLnR6aW5mbyBpcyBOb25lIG9yIHBhcnNlZC51dGNvZmZzZXQoKSAhPSB0aW1lZGVsdGEoMCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignZGVtb25zdHJhdGlvbiB0aW1lc3RhbXAgbXVzdCBiZSBVVEMnKQogICAgcmV0dXJuIHBhcnNlZAoKCmRlZiBfcGFpcnMoaXRlbXMpOgogICAgcmVzdWx0ID0ge30KICAgIGZvciBrZXksIHZhbHVlIGluIGl0ZW1zOgogICAgICAgIGlmIGtleSBpbiByZXN1bHQ6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2R1cGxpY2F0ZSBKU09OIGZpZWxkJykKICAgICAgICByZXN1bHRba2V5XSA9IHZhbHVlCiAgICByZXR1cm4gcmVzdWx0CgoKZGVmIF9ub3RfZmluaXRlKF8pOgogICAgcmFpc2UgVmFsdWVFcnJvcignbm9uLWZpbml0ZSBKU09OIHZhbHVlJykKCgpkZWYgY29tcGlsZV9kZW1vKHBhdGgsIGV4cGVjdGVkX3NoYTI1Nik6CiAgICAiIiJFeHRyYWN0IHRoZSBmaXJzdCB1bmludGVycnVwdGVkIHRocmVlLWxldmVsIHN1Y2Nlc3M7IGxlYXZlIHRoZSBzb3VyY2UgdW50b3VjaGVkLiIiIgogICAgX3NoYShleHBlY3RlZF9zaGEyNTYpCiAgICByYXcgPSBQYXRoKHBhdGgpLnJlYWRfYnl0ZXMoKQogICAgaWYgaGFzaGxpYi5zaGEyNTYocmF3KS5oZXhkaWdlc3QoKSAhPSBleHBlY3RlZF9zaGEyNTY6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignaHVtYW4gZGVtb25zdHJhdGlvbiBTSEEyNTYgZGlmZmVycycpCiAgICByb3dzID0gW2pzb24ubG9hZHMobGluZSwgb2JqZWN0X3BhaXJzX2hvb2s9X3BhaXJzLCBwYXJzZV9jb25zdGFudD1fbm90X2Zpbml0ZSkKICAgICAgICAgICAgZm9yIGxpbmUgaW4gcmF3LmRlY29kZSgndXRmLTgnKS5zcGxpdGxpbmVzKCldCiAgICBpZiBub3Qgcm93cyBvciB0eXBlKHJvd3NbMF0pIGlzIG5vdCBkaWN0OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ21pc3NpbmcgZGVtb25zdHJhdGlvbiBzZXNzaW9uJykKICAgIGhlYWRlciA9IHJvd3NbMF0KICAgIGlmIChoZWFkZXIuZ2V0KCd0eXBlJykgIT0gJ3Nlc3Npb25fc3RhcnQnIG9yIGhlYWRlci5nZXQoJ3NjaGVtYScpICE9ICdwdWJsaWNfZ2FtZV9tYW51YWxfdjEnIG9yCiAgICAgICAgICAgIGhlYWRlci5nZXQoJ2dhbWVfaWQnKSAhPSBHQU1FX0lEIG9yIHR5cGUoaGVhZGVyLmdldCgnc2VlZCcpKSBpcyBub3QgaW50IG9yCiAgICAgICAgICAgIGhlYWRlclsnc2VlZCddICE9IDAgb3IgaGVhZGVyLmdldCgnYWN0b3InKSAhPSAnaHVtYW5fbWFudWFsJyBvcgogICAgICAgICAgICBoZWFkZXIuZ2V0KCdzb3VyY2UnKSAhPSAnb2ZmbGluZV9wdWJsaWNfZGV2ZWxvcG1lbnRfZW5naW5lJyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignZGVtb25zdHJhdGlvbiBnYW1lLCBzZWVkIG9yIG9yaWdpbiBkaWZmZXJzJykKICAgIF91dGMoaGVhZGVyLmdldCgndXRjJykpCiAgICBwcmV2aW91cyA9IGhlYWRlci5nZXQoJ2luaXRpYWxfb2JzZXJ2YXRpb24nKQogICAgaW5pdGlhbCA9IF9yYXdfc25hcHNob3QocHJldmlvdXMpCiAgICBpZiBpbml0aWFsWydsZXZlbHNfY29tcGxldGVkJ10gIT0gMCBvciBpbml0aWFsWydzdGF0ZSddICE9ICdOT1RfRklOSVNIRUQnOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2RlbW9uc3RyYXRpb24gZG9lcyBub3Qgc3RhcnQgYXQgbGV2ZWwgb25lJykKICAgIHN0ZXBzLCBib3VuZGFyaWVzLCByZXF1ZXN0X2lkcyA9IFtdLCBbXSwgc2V0KCkKICAgIGxhc3RfYm91bmRhcnkgPSAwCiAgICBmb3IgaW5kZXggaW4gcmFuZ2UoTUFYX01PVkVTKToKICAgICAgICBwb3NpdGlvbiA9IDIgKiBpbmRleCArIDEKICAgICAgICBpZiBwb3NpdGlvbiArIDEgPj0gbGVuKHJvd3MpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdkZW1vbnN0cmF0aW9uIGVuZHMgYmVmb3JlIHRocmVlIGxldmVscyBhcmUgY29tcGxldGUnKQogICAgICAgIGVudGVyZWQsIGFja25vd2xlZGdlZCA9IHJvd3NbcG9zaXRpb246cG9zaXRpb24gKyAyXQogICAgICAgIGlmICh0eXBlKGVudGVyZWQpIGlzIG5vdCBkaWN0IG9yIHR5cGUoYWNrbm93bGVkZ2VkKSBpcyBub3QgZGljdCBvcgogICAgICAgICAgICAgICAgZW50ZXJlZC5nZXQoJ3R5cGUnKSAhPSAnYWN0aW9uX2VudGVyZWQnIG9yCiAgICAgICAgICAgICAgICBhY2tub3dsZWRnZWQuZ2V0KCd0eXBlJykgIT0gJ2FjdGlvbl9hY2tub3dsZWRnZWQnKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignZGVtb25zdHJhdGlvbiBhY3Rpb24gYW5kIGFja25vd2xlZGdlbWVudCBwYWlyaW5nIGRpZmZlcnMnKQogICAgICAgIGZvciBldmVudCBpbiAoZW50ZXJlZCwgYWNrbm93bGVkZ2VkKToKICAgICAgICAgICAgaWYgdHlwZShldmVudC5nZXQoJ3NlcXVlbmNlJykpIGlzIG5vdCBpbnQgb3IgZXZlbnRbJ3NlcXVlbmNlJ10gIT0gaW5kZXggKyAxOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignZGVtb25zdHJhdGlvbiBzZXF1ZW5jZSBkaWZmZXJzJykKICAgICAgICByZXF1ZXN0X2lkID0gZW50ZXJlZC5nZXQoJ3JlcXVlc3RfaWQnKQogICAgICAgIGlmICh0eXBlKHJlcXVlc3RfaWQpIGlzIG5vdCBzdHIgb3Igbm90IHJlcXVlc3RfaWQgb3IgcmVxdWVzdF9pZCBpbiByZXF1ZXN0X2lkcyBvcgogICAgICAgICAgICAgICAgYWNrbm93bGVkZ2VkLmdldCgncmVxdWVzdF9pZCcpICE9IHJlcXVlc3RfaWQpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdkZW1vbnN0cmF0aW9uIHJlcXVlc3QgcGFpcmluZyBkaWZmZXJzJykKICAgICAgICByZXF1ZXN0X2lkcy5hZGQocmVxdWVzdF9pZCkKICAgICAgICAjIFRoZSByZXRhaW5lZCBkZW1vIGhhcyB0d28gd2FsbC1jbG9jayByZXZlcnNhbHMuIFNlcXVlbmNlIG51bWJlcnMsCiAgICAgICAgIyByZXF1ZXN0IHBhaXJpbmcgYW5kIHRoZSBleGFjdCBvYnNlcnZhdGlvbiBjaGFpbiBlc3RhYmxpc2ggZXZlbnQgb3JkZXIuCiAgICAgICAgX3V0YyhlbnRlcmVkLmdldCgndXRjJykpCiAgICAgICAgX3V0YyhhY2tub3dsZWRnZWQuZ2V0KCd1dGMnKSkKICAgICAgICBzZWNvbmRzID0gYWNrbm93bGVkZ2VkLmdldCgnc2Vjb25kcycpCiAgICAgICAgaWYgdHlwZShzZWNvbmRzKSBub3QgaW4gKGludCwgZmxvYXQpIG9yIG5vdCBtYXRoLmlzZmluaXRlKHNlY29uZHMpIG9yIHNlY29uZHMgPCAwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdkZW1vbnN0cmF0aW9uIGFja25vd2xlZGdlbWVudCBkdXJhdGlvbiBkaWZmZXJzJykKICAgICAgICBiZWZvcmUsIGFmdGVyID0gX3Jhd19zbmFwc2hvdChlbnRlcmVkLmdldCgnYmVmb3JlJykpLCBfcmF3X3NuYXBzaG90KGFja25vd2xlZGdlZC5nZXQoJ2FmdGVyJykpCiAgICAgICAgaWYgZW50ZXJlZFsnYmVmb3JlJ10gIT0gcHJldmlvdXMgb3IgYmVmb3JlWydzdGF0ZSddICE9ICdOT1RfRklOSVNIRUQnOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdkZW1vbnN0cmF0aW9uIG9ic2VydmF0aW9uIGNoYWluIGRpZmZlcnMnKQogICAgICAgIGFjdGlvbiA9IF9hY3Rpb24oZW50ZXJlZC5nZXQoJ2FjdGlvbicpKQogICAgICAgIGlmIGFjdGlvblsnYWN0aW9uX2lkJ10gbm90IGluIGJlZm9yZVsnbGVnYWxfYWN0aW9ucyddOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdkZW1vbnN0cmF0aW9uIGFjdGlvbiBpcyBub3QgbGVnYWwnKQogICAgICAgIG9sZF9sZXZlbCwgbmV3X2xldmVsID0gYmVmb3JlWydsZXZlbHNfY29tcGxldGVkJ10sIGFmdGVyWydsZXZlbHNfY29tcGxldGVkJ10KICAgICAgICBpZiBub3Qgb2xkX2xldmVsIDw9IG5ld19sZXZlbCA8PSBvbGRfbGV2ZWwgKyAxIG9yIGFmdGVyWydzdGF0ZSddID09ICdHQU1FX09WRVInOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdkZW1vbnN0cmF0aW9uIGxldmVsIHByb2dyZXNzaW9uIGRpZmZlcnMnKQogICAgICAgIGlmIGFmdGVyWydzdGF0ZSddICE9ICdOT1RfRklOSVNIRUQnIGFuZCBuZXdfbGV2ZWwgPCAzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdkZW1vbnN0cmF0aW9uIHRlcm1pbmF0ZXMgYmVmb3JlIGxldmVsIHRocmVlJykKICAgICAgICBzdGVwcy5hcHBlbmQoeydpbmRleCc6IGluZGV4LCAnYWN0aW9uJzogZGVlcGNvcHkoYWN0aW9uKSwKICAgICAgICAgICAgICAgICAgICAgICdiZWZvcmVfc2hhMjU2JzogZGlnZXN0KGJlZm9yZSksICdhZnRlcl9zaGEyNTYnOiBkaWdlc3QoYWZ0ZXIpLAogICAgICAgICAgICAgICAgICAgICAgJ2JlZm9yZV9sZXZlbHMnOiBvbGRfbGV2ZWwsICdhZnRlcl9sZXZlbHMnOiBuZXdfbGV2ZWx9KQogICAgICAgIGlmIG5ld19sZXZlbCA+IG9sZF9sZXZlbDoKICAgICAgICAgICAgYm91bmRhcmllcy5hcHBlbmQoeydsZXZlbCc6IG5ld19sZXZlbCwgJ2FmdGVyX21vdmVzJzogaW5kZXggKyAxLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgJ21vdmVzJzogaW5kZXggKyAxIC0gbGFzdF9ib3VuZGFyeX0pCiAgICAgICAgICAgIGxhc3RfYm91bmRhcnkgPSBpbmRleCArIDEKICAgICAgICBpZiBuZXdfbGV2ZWwgPT0gMzoKICAgICAgICAgICAgcHJvZ3JhbSA9IHsnc2NoZW1hJzogU0NIRU1BLCAnZ2FtZV9pZCc6IEdBTUVfSUQsICdzZWVkJzogMCwgJ2dvYWxfbGV2ZWxzJzogMywKICAgICAgICAgICAgICAgICAgICAgICAnc291cmNlX2RlbW9fc2hhMjU2JzogZXhwZWN0ZWRfc2hhMjU2LCAnaW5pdGlhbF9vYnNlcnZhdGlvbic6IGRlZXBjb3B5KGluaXRpYWwpLAogICAgICAgICAgICAgICAgICAgICAgICdzdGVwcyc6IHN0ZXBzLCAnbGV2ZWxfYm91bmRhcmllcyc6IGJvdW5kYXJpZXMsICdxdWFsaWZpY2F0aW9uJzogUVVBTElGSUNBVElPTn0KICAgICAgICAgICAgT3ZlcmZpdFBvbGljeShwcm9ncmFtKQogICAgICAgICAgICByZXR1cm4gcHJvZ3JhbQogICAgICAgIHByZXZpb3VzID0gYWNrbm93bGVkZ2VkWydhZnRlciddCiAgICByYWlzZSBWYWx1ZUVycm9yKCd0aHJlZS1sZXZlbCBkZW1vbnN0cmF0aW9uIGV4Y2VlZHMgdGhlIDI1Ni1tb3ZlIGNhcCcpCgoKY2xhc3MgT3ZlcmZpdFBvbGljeToKICAgICIiIkFja25vd2xlZGdlIGV2ZXJ5IG9ic2VydmVkIHRyYW5zaXRpb24gYmVmb3JlIHNlbGVjdGluZyB0aGUgbmV4dCBtZW1vcml6ZWQgYWN0aW9uLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBwcm9ncmFtKToKICAgICAgICBzZWxmLl9wcm9ncmFtID0gZGVlcGNvcHkocHJvZ3JhbSkKICAgICAgICBwID0gc2VsZi5fcHJvZ3JhbQogICAgICAgIGZpZWxkcyA9IHsnc2NoZW1hJywgJ2dhbWVfaWQnLCAnc2VlZCcsICdnb2FsX2xldmVscycsICdzb3VyY2VfZGVtb19zaGEyNTYnLAogICAgICAgICAgICAgICAgICAnaW5pdGlhbF9vYnNlcnZhdGlvbicsICdzdGVwcycsICdsZXZlbF9ib3VuZGFyaWVzJywgJ3F1YWxpZmljYXRpb24nfQogICAgICAgIGlmICh0eXBlKHApIGlzIG5vdCBkaWN0IG9yIHNldChwKSAhPSBmaWVsZHMgb3IgcFsnc2NoZW1hJ10gIT0gU0NIRU1BIG9yCiAgICAgICAgICAgICAgICBwWydnYW1lX2lkJ10gIT0gR0FNRV9JRCBvciB0eXBlKHBbJ3NlZWQnXSkgaXMgbm90IGludCBvciBwWydzZWVkJ10gIT0gMCBvcgogICAgICAgICAgICAgICAgdHlwZShwWydnb2FsX2xldmVscyddKSBpcyBub3QgaW50IG9yIHBbJ2dvYWxfbGV2ZWxzJ10gIT0gMyBvcgogICAgICAgICAgICAgICAgcFsncXVhbGlmaWNhdGlvbiddICE9IFFVQUxJRklDQVRJT04pOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdvdmVyZml0IHByb2dyYW0gY29udHJhY3QgZGlmZmVycycpCiAgICAgICAgX3NoYShwWydzb3VyY2VfZGVtb19zaGEyNTYnXSkKICAgICAgICBpbml0aWFsID0gX3NuYXBzaG90KHBbJ2luaXRpYWxfb2JzZXJ2YXRpb24nXSkKICAgICAgICBpZiBpbml0aWFsWydzdGF0ZSddICE9ICdOT1RfRklOSVNIRUQnIG9yIGluaXRpYWxbJ2xldmVsc19jb21wbGV0ZWQnXSAhPSAwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdvdmVyZml0IHByb2dyYW0gaW5pdGlhbCBvYnNlcnZhdGlvbiBkaWZmZXJzJykKICAgICAgICBpZiB0eXBlKHBbJ3N0ZXBzJ10pIGlzIG5vdCBsaXN0IG9yIG5vdCAxIDw9IGxlbihwWydzdGVwcyddKSA8PSBNQVhfTU9WRVM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ292ZXJmaXQgcHJvZ3JhbSBtb3ZlIGNhcCBkaWZmZXJzJykKICAgICAgICBwcmV2aW91c19oYXNoLCBsZXZlbCwgYm91bmRhcnkgPSBkaWdlc3QoaW5pdGlhbCksIDAsIDAKICAgICAgICBib3VuZGFyaWVzID0gW10KICAgICAgICBmb3IgaW5kZXgsIHN0ZXAgaW4gZW51bWVyYXRlKHBbJ3N0ZXBzJ10pOgogICAgICAgICAgICBpZiAodHlwZShzdGVwKSBpcyBub3QgZGljdCBvciBzZXQoc3RlcCkgIT0geydpbmRleCcsICdhY3Rpb24nLCAnYmVmb3JlX3NoYTI1NicsCiAgICAgICAgICAgICAgICAgICAgJ2FmdGVyX3NoYTI1NicsICdiZWZvcmVfbGV2ZWxzJywgJ2FmdGVyX2xldmVscyd9IG9yCiAgICAgICAgICAgICAgICAgICAgdHlwZShzdGVwWydpbmRleCddKSBpcyBub3QgaW50IG9yIHN0ZXBbJ2luZGV4J10gIT0gaW5kZXggb3IKICAgICAgICAgICAgICAgICAgICB0eXBlKHN0ZXBbJ2JlZm9yZV9sZXZlbHMnXSkgaXMgbm90IGludCBvciBzdGVwWydiZWZvcmVfbGV2ZWxzJ10gIT0gbGV2ZWwgb3IKICAgICAgICAgICAgICAgICAgICB0eXBlKHN0ZXBbJ2FmdGVyX2xldmVscyddKSBpcyBub3QgaW50IG9yCiAgICAgICAgICAgICAgICAgICAgbm90IGxldmVsIDw9IHN0ZXBbJ2FmdGVyX2xldmVscyddIDw9IG1pbigzLCBsZXZlbCArIDEpIG9yIGxldmVsID09IDMpOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignb3ZlcmZpdCBwcm9ncmFtIHN0ZXAgb3IgbGV2ZWwgY2hhaW4gZGlmZmVycycpCiAgICAgICAgICAgIF9hY3Rpb24oc3RlcFsnYWN0aW9uJ10pCiAgICAgICAgICAgIGlmIF9zaGEoc3RlcFsnYmVmb3JlX3NoYTI1NiddKSAhPSBwcmV2aW91c19oYXNoOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignb3ZlcmZpdCBwcm9ncmFtIG9ic2VydmF0aW9uIGhhc2ggY2hhaW4gZGlmZmVycycpCiAgICAgICAgICAgIHByZXZpb3VzX2hhc2ggPSBfc2hhKHN0ZXBbJ2FmdGVyX3NoYTI1NiddKQogICAgICAgICAgICBpZiBzdGVwWydhZnRlcl9sZXZlbHMnXSAhPSBsZXZlbDoKICAgICAgICAgICAgICAgIGJvdW5kYXJpZXMuYXBwZW5kKHsnbGV2ZWwnOiBzdGVwWydhZnRlcl9sZXZlbHMnXSwgJ2FmdGVyX21vdmVzJzogaW5kZXggKyAxLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICdtb3Zlcyc6IGluZGV4ICsgMSAtIGJvdW5kYXJ5fSkKICAgICAgICAgICAgICAgIGJvdW5kYXJ5ID0gaW5kZXggKyAxCiAgICAgICAgICAgIGxldmVsID0gc3RlcFsnYWZ0ZXJfbGV2ZWxzJ10KICAgICAgICAjIENhbm9uaWNhbCBKU09OIGRpc3Rpbmd1aXNoZXMgdHJ1ZSBmcm9tIDEsIHVubGlrZSBQeXRob24gZGljdCBlcXVhbGl0eS4KICAgICAgICBpZiAobGV2ZWwgIT0gMyBvciB0eXBlKHBbJ2xldmVsX2JvdW5kYXJpZXMnXSkgaXMgbm90IGxpc3Qgb3IKICAgICAgICAgICAgICAgIGFueSh0eXBlKGl0ZW0pIGlzIG5vdCBkaWN0IG9yIHNldChpdGVtKSAhPSB7J2xldmVsJywgJ2FmdGVyX21vdmVzJywgJ21vdmVzJ30gb3IKICAgICAgICAgICAgICAgICAgICBhbnkodHlwZSh2YWx1ZSkgaXMgbm90IGludCBmb3IgdmFsdWUgaW4gaXRlbS52YWx1ZXMoKSkKICAgICAgICAgICAgICAgICAgICBmb3IgaXRlbSBpbiBwWydsZXZlbF9ib3VuZGFyaWVzJ10pIG9yCiAgICAgICAgICAgICAgICBkaWdlc3QocFsnbGV2ZWxfYm91bmRhcmllcyddKSAhPSBkaWdlc3QoYm91bmRhcmllcykpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdvdmVyZml0IHByb2dyYW0gY29tcGxldGlvbiBib3VuZGFyaWVzIGRpZmZlcicpCiAgICAgICAgc2VsZi5fY3Vyc29yLCBzZWxmLl9wZW5kaW5nLCBzZWxmLl9mYWlsZWQgPSAwLCBOb25lLCBGYWxzZQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIGN1cnNvcihzZWxmKToKICAgICAgICByZXR1cm4gc2VsZi5fY3Vyc29yCgogICAgQHByb3BlcnR5CiAgICBkZWYgY29tcGxldGVkKHNlbGYpOgogICAgICAgIHJldHVybiBub3Qgc2VsZi5fZmFpbGVkIGFuZCBzZWxmLl9jdXJzb3IgPT0gbGVuKHNlbGYuX3Byb2dyYW1bJ3N0ZXBzJ10pCgogICAgZGVmIF9yZWZ1c2Uoc2VsZiwgbWVzc2FnZSk6CiAgICAgICAgc2VsZi5fZmFpbGVkID0gVHJ1ZQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IobWVzc2FnZSkKCiAgICBkZWYgc2VsZWN0KHNlbGYsIGdhbWVfaWQsIG9ic2VydmF0aW9uKToKICAgICAgICBpZiBzZWxmLl9mYWlsZWQ6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ292ZXJmaXQgcG9saWN5IHByZXZpb3VzbHkgZmFpbGVkJykKICAgICAgICBpZiBzZWxmLmNvbXBsZXRlZDoKICAgICAgICAgICAgcmFpc2UgU3RvcEl0ZXJhdGlvbignZmlyc3QgdGhyZWUgbGV2ZWxzIGNvbXBsZXRlJykKICAgICAgICBpZiBzZWxmLl9wZW5kaW5nIGlzIG5vdCBOb25lOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdwcmV2aW91cyBhY3Rpb24gYWNrbm93bGVkZ2VtZW50IGlzIHN0aWxsIHBlbmRpbmcnKQogICAgICAgIHRyeToKICAgICAgICAgICAgY3VycmVudCA9IF9zbmFwc2hvdChvYnNlcnZhdGlvbikKICAgICAgICAgICAgc3RlcCA9IHNlbGYuX3Byb2dyYW1bJ3N0ZXBzJ11bc2VsZi5fY3Vyc29yXQogICAgICAgICAgICBpZiAoZ2FtZV9pZCAhPSBHQU1FX0lEIG9yIGN1cnJlbnRbJ3N0YXRlJ10gIT0gJ05PVF9GSU5JU0hFRCcgb3IKICAgICAgICAgICAgICAgICAgICBjdXJyZW50WydsZXZlbHNfY29tcGxldGVkJ10gIT0gc3RlcFsnYmVmb3JlX2xldmVscyddIG9yCiAgICAgICAgICAgICAgICAgICAgZGlnZXN0KGN1cnJlbnQpICE9IHN0ZXBbJ2JlZm9yZV9zaGEyNTYnXSBvcgogICAgICAgICAgICAgICAgICAgIHN0ZXBbJ2FjdGlvbiddWydhY3Rpb25faWQnXSBub3QgaW4gY3VycmVudFsnbGVnYWxfYWN0aW9ucyddKToKICAgICAgICAgICAgICAgIHNlbGYuX3JlZnVzZSgnY3VycmVudCBvYnNlcnZhdGlvbiwgZ2FtZSBvciBhY3Rpb24gZGlmZmVycyBmcm9tIHRoZSBkZW1vbnN0cmF0aW9uJykKICAgICAgICBleGNlcHQgKFR5cGVFcnJvciwgS2V5RXJyb3IsIFZhbHVlRXJyb3IpOgogICAgICAgICAgICBzZWxmLl9mYWlsZWQgPSBUcnVlCiAgICAgICAgICAgIHJhaXNlCiAgICAgICAgc2VsZi5fcGVuZGluZyA9IGRlZXBjb3B5KHN0ZXBbJ2FjdGlvbiddKQogICAgICAgIHJldHVybiBkZWVwY29weShzZWxmLl9wZW5kaW5nKQoKICAgIGRlZiBhY2tub3dsZWRnZShzZWxmLCBhY3Rpb24sIGFmdGVyKToKICAgICAgICBpZiBzZWxmLl9mYWlsZWQ6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ292ZXJmaXQgcG9saWN5IHByZXZpb3VzbHkgZmFpbGVkJykKICAgICAgICBpZiBzZWxmLl9wZW5kaW5nIGlzIE5vbmU6CiAgICAgICAgICAgIHNlbGYuX3JlZnVzZSgnbm8gYWN0aW9uIGlzIGF3YWl0aW5nIGFja25vd2xlZGdlbWVudCcpCiAgICAgICAgdHJ5OgogICAgICAgICAgICBfYWN0aW9uKGFjdGlvbikKICAgICAgICAgICAgY3VycmVudCA9IF9zbmFwc2hvdChhZnRlcikKICAgICAgICAgICAgc3RlcCA9IHNlbGYuX3Byb2dyYW1bJ3N0ZXBzJ11bc2VsZi5fY3Vyc29yXQogICAgICAgICAgICBpZiAoYWN0aW9uICE9IHNlbGYuX3BlbmRpbmcgb3IgZGlnZXN0KGN1cnJlbnQpICE9IHN0ZXBbJ2FmdGVyX3NoYTI1NiddIG9yCiAgICAgICAgICAgICAgICAgICAgY3VycmVudFsnbGV2ZWxzX2NvbXBsZXRlZCddICE9IHN0ZXBbJ2FmdGVyX2xldmVscyddIG9yCiAgICAgICAgICAgICAgICAgICAgY3VycmVudFsnc3RhdGUnXSA9PSAnR0FNRV9PVkVSJyBvcgogICAgICAgICAgICAgICAgICAgIChjdXJyZW50WydzdGF0ZSddICE9ICdOT1RfRklOSVNIRUQnIGFuZCBjdXJyZW50WydsZXZlbHNfY29tcGxldGVkJ10gPCAzKSk6CiAgICAgICAgICAgICAgICBzZWxmLl9yZWZ1c2UoJ2Fja25vd2xlZGdlZCBhY3Rpb24gb3Igb2JzZXJ2YXRpb24gZGlmZmVycyBmcm9tIHRoZSBkZW1vbnN0cmF0aW9uJykKICAgICAgICBleGNlcHQgKFR5cGVFcnJvciwgS2V5RXJyb3IsIFZhbHVlRXJyb3IpOgogICAgICAgICAgICBzZWxmLl9mYWlsZWQgPSBUcnVlCiAgICAgICAgICAgIHJhaXNlCiAgICAgICAgc2VsZi5fY3Vyc29yICs9IDEKICAgICAgICBzZWxmLl9wZW5kaW5nID0gTm9uZQo=', 'research/ls20_overfit_v1/policy_seed0.json': '...', 'tests/__init__.py': 'IiIiUHJvamVjdCB0ZXN0IHN1aXRlLiIiIgo=', 'tests/test_cpu_competition_packaging_v1.py': 'IiIiQ1BVIGRlcGxveW1lbnQgbGlmZWN5Y2xlIGNvbnRyYWN0cyB1c2luZyBmYWtlIGluc3RhbGxzIGFuZCBhIGZha2Ugd2F0Y2hkb2cuCgpObyBwYWNrYWdlIGluc3RhbGxzLCBtb2RlbCBpbmZlcmVuY2UsIGdhbWUgZGlzcGF0Y2hlcyBvciBuZXR3b3JrIHJlcXVlc3RzIG9jY3VyLgpUaGUgb25seSByZWFsIHRpbWVyIHRlc3QgaW50ZXJydXB0cyBhIG1vY2tlZCByZXF1ZXN0cyBtZXRob2Qgc2xlZXBpbmcgbG9jYWxseS4KIiIiCmZyb20gY29udGV4dGxpYiBpbXBvcnQgRXhpdFN0YWNrCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgc2lnbmFsCmltcG9ydCB0ZW1wZmlsZQppbXBvcnQgdGltZQpmcm9tIHR5cGVzIGltcG9ydCBTaW1wbGVOYW1lc3BhY2UKaW1wb3J0IHVuaXR0ZXN0CmZyb20gdW5pdHRlc3QubW9jayBpbXBvcnQgcGF0Y2gKCmltcG9ydCByZXF1ZXN0cwoKZnJvbSByZXNlYXJjaC5jcHVfY29tcGV0aXRpb25fdjEgaW1wb3J0IHN1cGVydmlzb3IKZnJvbSByZXNlYXJjaC5jcHVfY29tcGV0aXRpb25fdjEucnVudGltZSBpbXBvcnQgRGVhZGxpbmVTZXNzaW9uCmZyb20gcmVzZWFyY2guaW5pdGlhbF9nZW9tZXRyeV9hc3Npc3RhbmNlX3YxIGltcG9ydCBsaXZlX3Jlc291cmNlcwoKCmNsYXNzIExpZmVjeWNsZUZpeHR1cmUodW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIHNldFVwKHNlbGYpOgogICAgICAgIHRlbXBvcmFyeSA9IHRlbXBmaWxlLlRlbXBvcmFyeURpcmVjdG9yeSgpCiAgICAgICAgc2VsZi5hZGRDbGVhbnVwKHRlbXBvcmFyeS5jbGVhbnVwKQogICAgICAgIHNlbGYuYmFzZSA9IFBhdGgodGVtcG9yYXJ5Lm5hbWUpCiAgICAgICAgc2VsZi5yb290ID0gc2VsZi5iYXNlIC8gJ3NvdXJjZScKICAgICAgICBjb250cmFjdHMgPSBzZWxmLnJvb3QgLyAncmVzZWFyY2gvY3B1X2NvbXBldGl0aW9uX3YxJwogICAgICAgIGNvbnRyYWN0cy5ta2RpcihwYXJlbnRzPVRydWUpCiAgICAgICAgKGNvbnRyYWN0cyAvICd3aGVlbHMuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyh7fSksIGVuY29kaW5nPSd1dGYtOCcpCiAgICAgICAgc2VsZi53aGVlbHMgPSBzZWxmLmJhc2UgLyAnd2hlZWxzJwogICAgICAgIHNlbGYud2hlZWxzLm1rZGlyKCkKICAgICAgICBzZWxmLnJ1bnRpbWVfdGVtcCA9IHNlbGYuYmFzZSAvICdydW50aW1lLXRlbXBvcmFyeScKICAgICAgICBzZWxmLnJ1bnRpbWVfdGVtcC5ta2RpcigpCiAgICAgICAgc2VsZi5jbG9jayA9IDEwMC4wCiAgICAgICAgc2VsZi5jb21tYW5kcyA9IFtdCiAgICAgICAgc2VsZi50cmFuc2l0aW9ucyA9IFtdCiAgICAgICAgc2VsZi53YXRjaCA9IE5vbmUKICAgICAgICBzZWxmLmZhaWxfY29tbWFuZCA9IE5vbmUKICAgICAgICBzZWxmLndvcmtlcl9maW5pc2ggPSBOb25lCgogICAgZGVmIGludm9rZShzZWxmLCBjYWxsYmFjaywgKiwgc2NvcmVkPVRydWUpOgogICAgICAgIGZpeHR1cmUgPSBzZWxmCgogICAgICAgIGNsYXNzIFdhdGNoZG9nOgogICAgICAgICAgICBkZWYgX19pbml0X18oc2VsZiwgZGVhZGxpbmUpOgogICAgICAgICAgICAgICAgc2VsZi5kZWFkbGluZSA9IGRlYWRsaW5lCiAgICAgICAgICAgICAgICBzZWxmLmFjdGl2ZSA9IEZhbHNlCiAgICAgICAgICAgICAgICBmaXh0dXJlLndhdGNoID0gc2VsZgoKICAgICAgICAgICAgZGVmIHN0YXJ0KHNlbGYpOgogICAgICAgICAgICAgICAgc2VsZi5hY3RpdmUgPSBUcnVlCiAgICAgICAgICAgICAgICBmaXh0dXJlLnRyYW5zaXRpb25zLmFwcGVuZCgnd2F0Y2gtc3RhcnQnKQoKICAgICAgICAgICAgZGVmIGNsb3NlKHNlbGYpOgogICAgICAgICAgICAgICAgZml4dHVyZS5hc3NlcnRUcnVlKHNlbGYuYWN0aXZlKQogICAgICAgICAgICAgICAgc2VsZi5hY3RpdmUgPSBGYWxzZQogICAgICAgICAgICAgICAgZml4dHVyZS50cmFuc2l0aW9ucy5hcHBlbmQoJ3dhdGNoLWNsb3NlJykKCiAgICAgICAgY2xhc3MgUHJvY2Vzc2VzOgogICAgICAgICAgICBkZWYgX19pbml0X18oc2VsZiwgd2F0Y2hlcik6CiAgICAgICAgICAgICAgICBmaXh0dXJlLmFzc2VydElzKHdhdGNoZXIsIGZpeHR1cmUud2F0Y2gpCgogICAgICAgICAgICBkZWYgY29tbWFuZChzZWxmLCBhcmd2LCBsb2csIGRlYWRsaW5lLCBlbnYsIGxhYmVsKToKICAgICAgICAgICAgICAgIGZpeHR1cmUuYXNzZXJ0VHJ1ZShmaXh0dXJlLndhdGNoLmFjdGl2ZSkKICAgICAgICAgICAgICAgIGluZGV4ID0gbGVuKGZpeHR1cmUuY29tbWFuZHMpCiAgICAgICAgICAgICAgICBmaXh0dXJlLmNvbW1hbmRzLmFwcGVuZChkaWN0KGFyZ3Y9YXJndiwgZGVhZGxpbmU9ZGVhZGxpbmUsIGxhYmVsPWxhYmVsKSkKICAgICAgICAgICAgICAgIGlmIGZpeHR1cmUuZmFpbF9jb21tYW5kID09IGluZGV4OgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignc3ludGhldGljIGluc3RhbGxhdGlvbiBmYWlsdXJlJykKICAgICAgICAgICAgICAgIGlmIGxhYmVsID09ICdTaW5nbGUtc2NvcmVjYXJkIENQVSBjb21wZXRpdGlvbiBhZ2VudCcgYW5kIGZpeHR1cmUud29ya2VyX2ZpbmlzaCBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgICAgICBmaXh0dXJlLmNsb2NrID0gZml4dHVyZS53b3JrZXJfZmluaXNoCiAgICAgICAgICAgICAgICByZXR1cm4gU2ltcGxlTmFtZXNwYWNlKHN0ZG91dD0nc3ludGhldGljIHdvcmtlciBvdXRwdXQnKQoKICAgICAgICAgICAgZGVmIHN0b3Aoc2VsZiwgZGVhZGxpbmUpOgogICAgICAgICAgICAgICAgZml4dHVyZS5hc3NlcnRUcnVlKGZpeHR1cmUud2F0Y2guYWN0aXZlKQogICAgICAgICAgICAgICAgZml4dHVyZS5hc3NlcnRFcXVhbChkZWFkbGluZSwgZml4dHVyZS53YXRjaC5kZWFkbGluZSkKICAgICAgICAgICAgICAgIGZpeHR1cmUudHJhbnNpdGlvbnMuYXBwZW5kKCdwcm9jZXNzLXN0b3AnKQogICAgICAgICAgICAgICAgcmV0dXJuIGRpY3QoZ3JvdXBzX2Fic2VudD1UcnVlLCBlcnJvcj1Ob25lLCBpbnRlcnJ1cHRlZD1GYWxzZSkKCiAgICAgICAgd2l0aCBFeGl0U3RhY2soKSBhcyBzdGFjazoKICAgICAgICAgICAgc3RhY2suZW50ZXJfY29udGV4dChwYXRjaC5vYmplY3QobGl2ZV9yZXNvdXJjZXMsICdIYXJkRGVhZGxpbmUnLCBXYXRjaGRvZykpCiAgICAgICAgICAgIHN0YWNrLmVudGVyX2NvbnRleHQocGF0Y2gub2JqZWN0KGxpdmVfcmVzb3VyY2VzLCAnUHJvY2Vzc2VzJywgUHJvY2Vzc2VzKSkKICAgICAgICAgICAgc3RhY2suZW50ZXJfY29udGV4dChwYXRjaC5vYmplY3QobGl2ZV9yZXNvdXJjZXMsICdlbnZpcm9ubWVudCcsIHJldHVybl92YWx1ZT17fSkpCiAgICAgICAgICAgIHN0YWNrLmVudGVyX2NvbnRleHQocGF0Y2gub2JqZWN0KHN1cGVydmlzb3Iuc3lzLCAndmVyc2lvbl9pbmZvJywgKDMsIDEyLCAwKSkpCiAgICAgICAgICAgIHN0YWNrLmVudGVyX2NvbnRleHQocGF0Y2gub2JqZWN0KHN1cGVydmlzb3IudGltZSwgJ21vbm90b25pYycsIGxhbWJkYTogc2VsZi5jbG9jaykpCiAgICAgICAgICAgIHN0YWNrLmVudGVyX2NvbnRleHQocGF0Y2gub2JqZWN0KHN1cGVydmlzb3IudGVtcGZpbGUsICdta2R0ZW1wJywgcmV0dXJuX3ZhbHVlPXN0cihzZWxmLnJ1bnRpbWVfdGVtcCkpKQogICAgICAgICAgICBzdXBlcnZpc29yLmVudHJ5KHNlbGYucm9vdCwgc2VsZi53aGVlbHMsIHNjb3JlZD1zY29yZWQsIGZpbmFsaXplPWNhbGxiYWNrKQoKICAgIGRlZiB0ZXN0X3dvcmtlcl9rZWVwc19jbGVhbnVwX3Jlc2VydmVfYW5kX2NhbGxiYWNrX3J1bnNfaW5zaWRlX3dhdGNoZG9nKHNlbGYpOgogICAgICAgIGRlZiBjYWxsYmFjayhjb21wbGV0ZWQpOgogICAgICAgICAgICBzZWxmLmFzc2VydFRydWUoY29tcGxldGVkKQogICAgICAgICAgICBzZWxmLmFzc2VydFRydWUoc2VsZi53YXRjaC5hY3RpdmUpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RmFsc2Uoc2VsZi5ydW50aW1lX3RlbXAuZXhpc3RzKCkpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc2VsZi50cmFuc2l0aW9uc1stMV0sICdwcm9jZXNzLXN0b3AnKQogICAgICAgICAgICBzZWxmLnRyYW5zaXRpb25zLmFwcGVuZCgnYXJ0aWZhY3QtZmluYWxpemUnKQoKICAgICAgICBzZWxmLmludm9rZShjYWxsYmFjaykKICAgICAgICB3b3JrZXIgPSBzZWxmLmNvbW1hbmRzWy0xXQogICAgICAgIHdvcmtfYXJnID0gd29ya2VyWydhcmd2J10uaW5kZXgoJy0td29yay1zZWNvbmRzJykgKyAxCiAgICAgICAgZmluYWxfYXJnID0gd29ya2VyWydhcmd2J10uaW5kZXgoJy0tZmluYWwtc2Vjb25kcycpICsgMQogICAgICAgIGFkbWlzc2lvbl9idWRnZXQgPSBpbnQod29ya2VyWydhcmd2J11bd29ya19hcmddKQogICAgICAgIHdvcmtlcl9maW5hbF9idWRnZXQgPSBpbnQod29ya2VyWydhcmd2J11bZmluYWxfYXJnXSkKICAgICAgICAjIEFkbWlzc2lvbiBtdXN0IGNsb3NlIHdoaWxlIHRoZSB3b3JrZXIgY2FuIHN0aWxsIGNsb3NlIGl0cyBzY29yZWNhcmQuCiAgICAgICAgc2VsZi5hc3NlcnRHcmVhdGVyKHdvcmtlcl9maW5hbF9idWRnZXQsIGFkbWlzc2lvbl9idWRnZXQpCiAgICAgICAgc2VsZi5hc3NlcnRHcmVhdGVyKHdvcmtlclsnZGVhZGxpbmUnXSwgMTAwLjAgKyBhZG1pc3Npb25fYnVkZ2V0KQogICAgICAgIHNlbGYuYXNzZXJ0TGVzcyh3b3JrZXJbJ2RlYWRsaW5lJ10sIHNlbGYud2F0Y2guZGVhZGxpbmUpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChzZWxmLnRyYW5zaXRpb25zLAogICAgICAgICAgICAgICAgICAgICAgICAgWyd3YXRjaC1zdGFydCcsICdwcm9jZXNzLXN0b3AnLCAnYXJ0aWZhY3QtZmluYWxpemUnLCAnd2F0Y2gtY2xvc2UnXSkKICAgICAgICBzZWxmLmFzc2VydEZhbHNlKHNlbGYud2F0Y2guYWN0aXZlKQoKICAgIGRlZiB0ZXN0X3dvcmtfcmV0dXJuaW5nX2FmdGVyX2FkbWlzc2lvbl9jYW5fc3RpbGxfZmluYWxpemVfaW5fcmVzZXJ2ZShzZWxmKToKICAgICAgICBzZWxmLndvcmtlcl9maW5pc2ggPSAzMTMwLjAgICMgQWRtaXNzaW9uIGNsb3NlZCBhdCAzMTAwOyBoYXJkIGRlYWRsaW5lIGlzIDM0MDAuCiAgICAgICAgY2FsbHMgPSBbXQoKICAgICAgICBkZWYgY2FsbGJhY2soY29tcGxldGVkKToKICAgICAgICAgICAgY2FsbHMuYXBwZW5kKGNvbXBsZXRlZCkKICAgICAgICAgICAgc2VsZi5hc3NlcnRUcnVlKHNlbGYud2F0Y2guYWN0aXZlKQogICAgICAgICAgICBzZWxmLmFzc2VydEZhbHNlKHNlbGYucnVudGltZV90ZW1wLmV4aXN0cygpKQogICAgICAgICAgICBzZWxmLmFzc2VydExlc3Moc2VsZi5jbG9jaywgc2VsZi53YXRjaC5kZWFkbGluZSkKCiAgICAgICAgc2VsZi5pbnZva2UoY2FsbGJhY2spCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChzZWxmLmNsb2NrLCBzZWxmLndvcmtlcl9maW5pc2gpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChjYWxscywgW1RydWVdKQogICAgICAgIHNlbGYuYXNzZXJ0RmFsc2Uoc2VsZi53YXRjaC5hY3RpdmUpCgogICAgZGVmIHRlc3RfbGF0ZV9hcnRpZmFjdF9maW5hbGl6YXRpb25fZmFpbHNfd2l0aF9lbWVyZ2VuY3lfd2F0Y2hkb2dfYWN0aXZlKHNlbGYpOgogICAgICAgIGRlZiBjYWxsYmFjayhjb21wbGV0ZWQpOgogICAgICAgICAgICBzZWxmLmFzc2VydFRydWUoY29tcGxldGVkKQogICAgICAgICAgICBzZWxmLmFzc2VydFRydWUoc2VsZi53YXRjaC5hY3RpdmUpCiAgICAgICAgICAgIHNlbGYuY2xvY2sgPSBzZWxmLndhdGNoLmRlYWRsaW5lICsgMC4xCgogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXNSZWdleChUaW1lb3V0RXJyb3IsICdmaW5hbGl6YXRpb24gZGVhZGxpbmUnKToKICAgICAgICAgICAgc2VsZi5pbnZva2UoY2FsbGJhY2spCiAgICAgICAgIyBBIGZhaWxlZCBvdmVycnVuIG11c3Qgbm90IGRpc2FybSB0aGUgZW1lcmdlbmN5IGRlYWRsaW5lLgogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShzZWxmLndhdGNoLmFjdGl2ZSkKICAgICAgICBzZWxmLmFzc2VydE5vdEluKCd3YXRjaC1jbG9zZScsIHNlbGYudHJhbnNpdGlvbnMpCgogICAgZGVmIHRlc3RfY2FsbGJhY2tfZmFpbHVyZV9wcm9wYWdhdGVzX2FuZF9jbG9zZXNfd2F0Y2hkb2dfYmVmb3JlX2RlYWRsaW5lKHNlbGYpOgogICAgICAgIGNsYXNzIEFydGlmYWN0RmFpbHVyZShFeGNlcHRpb24pOgogICAgICAgICAgICBwYXNzCgogICAgICAgIGRlZiBjYWxsYmFjayhjb21wbGV0ZWQpOgogICAgICAgICAgICBzZWxmLmFzc2VydFRydWUoY29tcGxldGVkKQogICAgICAgICAgICBzZWxmLmFzc2VydFRydWUoc2VsZi53YXRjaC5hY3RpdmUpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RmFsc2Uoc2VsZi5ydW50aW1lX3RlbXAuZXhpc3RzKCkpCiAgICAgICAgICAgIHJhaXNlIEFydGlmYWN0RmFpbHVyZSgnc3ludGhldGljIGFydGlmYWN0IGZhaWx1cmUnKQoKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKEFydGlmYWN0RmFpbHVyZSk6CiAgICAgICAgICAgIHNlbGYuaW52b2tlKGNhbGxiYWNrKQogICAgICAgIHNlbGYuYXNzZXJ0RmFsc2Uoc2VsZi53YXRjaC5hY3RpdmUpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChzZWxmLnRyYW5zaXRpb25zWy0xXSwgJ3dhdGNoLWNsb3NlJykKCiAgICBkZWYgdGVzdF9pbnN0YWxsX2ZhaWx1cmVfbmV2ZXJfZmluYWxpemVzX2FydGlmYWN0c19hc19zdWNjZXNzKHNlbGYpOgogICAgICAgIHNlbGYuZmFpbF9jb21tYW5kID0gMAogICAgICAgIGNvbXBsZXRlZF92YWx1ZXMgPSBbXQoKICAgICAgICBkZWYgY2FsbGJhY2soY29tcGxldGVkKToKICAgICAgICAgICAgY29tcGxldGVkX3ZhbHVlcy5hcHBlbmQoY29tcGxldGVkKQogICAgICAgICAgICBzZWxmLmFzc2VydFRydWUoc2VsZi53YXRjaC5hY3RpdmUpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RmFsc2Uoc2VsZi5ydW50aW1lX3RlbXAuZXhpc3RzKCkpCgogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXNSZWdleChSdW50aW1lRXJyb3IsICdpbnN0YWxsYXRpb24gZmFpbHVyZScpOgogICAgICAgICAgICBzZWxmLmludm9rZShjYWxsYmFjaykKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGNvbXBsZXRlZF92YWx1ZXMsIFtGYWxzZV0pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsZW4oc2VsZi5jb21tYW5kcyksIDEpCiAgICAgICAgc2VsZi5hc3NlcnRGYWxzZShzZWxmLndhdGNoLmFjdGl2ZSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHNlbGYudHJhbnNpdGlvbnMsIFsnd2F0Y2gtc3RhcnQnLCAncHJvY2Vzcy1zdG9wJywgJ3dhdGNoLWNsb3NlJ10pCgoKQHVuaXR0ZXN0LnNraXBVbmxlc3MoaGFzYXR0cihzaWduYWwsICdzZXRpdGltZXInKSwgJ3JlcXVpcmVzIFBPU0lYIGludGVydmFsIHRpbWVycycpCmNsYXNzIEFic29sdXRlUmVzcG9uc2VEZWFkbGluZSh1bml0dGVzdC5UZXN0Q2FzZSk6CiAgICBkZWYgdGVzdF9tb2NrZWRfc3RhbGxlZF9yZXNwb25zZV9pc19pbnRlcnJ1cHRlZF9hbmRfc2lnbmFsX3N0YXRlX3Jlc3RvcmVkKHNlbGYpOgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoc2lnbmFsLmdldGl0aW1lcihzaWduYWwuSVRJTUVSX1JFQUwpLCAoMC4wLCAwLjApKQogICAgICAgIHByZXZpb3VzID0gc2lnbmFsLmdldHNpZ25hbChzaWduYWwuU0lHQUxSTSkKCiAgICAgICAgZGVmIHN0YWxsZWRfcmVxdWVzdChfc2Vzc2lvbiwgX21ldGhvZCwgX3VybCwgKiprd2FyZ3MpOgogICAgICAgICAgICBzZWxmLmFzc2VydEZhbHNlKGt3YXJnc1snYWxsb3dfcmVkaXJlY3RzJ10pCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RmFsc2Uoa3dhcmdzWydzdHJlYW0nXSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRHcmVhdGVyKGt3YXJnc1sndGltZW91dCddLCAwKQogICAgICAgICAgICB0aW1lLnNsZWVwKDAuMTUpCiAgICAgICAgICAgIHNlbGYuZmFpbCgnYWJzb2x1dGUgYWxhcm0gZGlkIG5vdCBpbnRlcnJ1cHQgdGhlIG1vY2tlZCByZXNwb25zZScpCgogICAgICAgIHN0YXJ0ZWQgPSB0aW1lLm1vbm90b25pYygpCiAgICAgICAgc2Vzc2lvbiA9IERlYWRsaW5lU2Vzc2lvbihzdGFydGVkICsgMC4wMykKICAgICAgICBzZWxmLmFkZENsZWFudXAoc2Vzc2lvbi5jbG9zZSkKICAgICAgICB3aXRoIHBhdGNoLm9iamVjdChyZXF1ZXN0cy5TZXNzaW9uLCAncmVxdWVzdCcsIHN0YWxsZWRfcmVxdWVzdCk6CiAgICAgICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXNSZWdleChUaW1lb3V0RXJyb3IsICdhYnNvbHV0ZSBIVFRQIHJlc3BvbnNlIGRlYWRsaW5lJyk6CiAgICAgICAgICAgICAgICBzZXNzaW9uLnJlcXVlc3QoJ0dFVCcsICdodHRwczovL3VudXNlZC5pbnZhbGlkL25vLW5ldHdvcmsnKQogICAgICAgIHNlbGYuYXNzZXJ0TGVzcyh0aW1lLm1vbm90b25pYygpIC0gc3RhcnRlZCwgMC4xMikKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHNpZ25hbC5nZXRpdGltZXIoc2lnbmFsLklUSU1FUl9SRUFMKSwgKDAuMCwgMC4wKSkKICAgICAgICBzZWxmLmFzc2VydElzKHNpZ25hbC5nZXRzaWduYWwoc2lnbmFsLlNJR0FMUk0pLCBwcmV2aW91cykKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgdW5pdHRlc3QubWFpbigpCg==', 'tests/test_cpu_competition_v1.py': '...', 'tests/test_ls20_overfit_v1.py': 'IiIiU3ludGhldGljIGRlbW9uc3RyYXRpb24taW1pdGF0aW9uIGNvbnRyYWN0czsgbm8gZW5naW5lLCBtb2RlbCBvciBwcml2YXRlIGxvZy4KClRoZSBmaXh0dXJlIGhhcyB0aHJlZSB2aXNpYmxlIHRyYW5zaXRpb25zIGFuZCB0aHJlZSBlbmdpbmUtY291bnRlciBpbmNyZW1lbnRzLgpQaXhlbHMsIGxlZ2FsIGNvbnRyb2xzIGFuZCBjb3VudGVycyBhbGwgbWF0dGVyOiBhIHZhbGlkIGFjdGlvbiBzZXF1ZW5jZSBhbG9uZQptdXN0IG5ldmVyIGF1dGhvcml6ZSBkaXNwYXRjaCBhZnRlciBvYnNlcnZhdGlvbiBkcmlmdC4KIiIiCmZyb20gY29weSBpbXBvcnQgZGVlcGNvcHkKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCB0ZW1wZmlsZQppbXBvcnQgdW5pdHRlc3QKCmZyb20gcmVzZWFyY2guaW5pdGlhbF9nZW9tZXRyeV9hc3Npc3RhbmNlX3YxLmdlb21ldHJ5IGltcG9ydCBkaWdlc3QKZnJvbSByZXNlYXJjaC5sczIwX292ZXJmaXRfdjEucG9saWN5IGltcG9ydCBPdmVyZml0UG9saWN5LCBjb21waWxlX2RlbW8KCgpHQU1FID0gJ2xzMjAtOTYwNzYyN2InCgoKZGVmIHJhd19vYnNlcnZhdGlvbihzdGVwPTApOgogICAgZ3JpZCA9IFtbM10gKiA2NCBmb3IgXyBpbiByYW5nZSg2NCldCiAgICBncmlkWzBdWzBdID0gc3RlcAogICAgIyBJbnRlcmZhY2UgcGl4ZWxzIGFyZSBwYXJ0IG9mIHRoZSBvYnNlcnZhdGlvbiBnYXRlLCB0b28uCiAgICBncmlkWzYzXVs2M10gPSAxMSAtIHN0ZXAKICAgIHJldHVybiBkaWN0KGdhbWVfaWQ9R0FNRSwgc3RhdGU9J05PVF9GSU5JU0hFRCcsIGxldmVsc19jb21wbGV0ZWQ9c3RlcCwKICAgICAgICAgICAgICAgIHdpbl9sZXZlbHM9NywgYXZhaWxhYmxlX2FjdGlvbnM9WzEsIDIsIDMsIDRdLCBmcmFtZXM9W2dyaWRdLAogICAgICAgICAgICAgICAgY2Fub25pY2FsX2hhc2g9Zid7c3RlcDowNjR4fScpCgoKZGVmIG5hdGl2ZShyYXcpOgogICAgIyBJbmRlcGVuZGVudCBmaXh0dXJlIHByb2plY3Rpb24sIHJhdGhlciB0aGFuIHRoZSBwcm9kdWN0aW9uIG5vcm1hbGl6ZXIuCiAgICByZXR1cm4gZGljdChncmlkPWRlZXBjb3B5KHJhd1snZnJhbWVzJ11bLTFdKSwKICAgICAgICAgICAgICAgIGxlZ2FsX2FjdGlvbnM9c29ydGVkKHJhd1snYXZhaWxhYmxlX2FjdGlvbnMnXSksCiAgICAgICAgICAgICAgICBzdGF0ZT1yYXdbJ3N0YXRlJ10sIGxldmVsc19jb21wbGV0ZWQ9cmF3WydsZXZlbHNfY29tcGxldGVkJ10pCgoKZGVmIGRlbW9fcm93cygpOgogICAgaW5pdGlhbCA9IHJhd19vYnNlcnZhdGlvbigpCiAgICByb3dzID0gW2RpY3QodHlwZT0nc2Vzc2lvbl9zdGFydCcsIHNjaGVtYT0ncHVibGljX2dhbWVfbWFudWFsX3YxJywKICAgICAgICAgICAgICAgICBzZXNzaW9uPSdzeW50aGV0aWMtb3duZXItZGVtbycsIGFjdG9yPSdodW1hbl9tYW51YWwnLCBnYW1lX2lkPUdBTUUsCiAgICAgICAgICAgICAgICAgc2VlZD0wLCB1dGM9JzIwMjYtMTEtMDJUMTI6MDA6MDArMDA6MDAnLAogICAgICAgICAgICAgICAgIHNvdXJjZT0nb2ZmbGluZV9wdWJsaWNfZGV2ZWxvcG1lbnRfZW5naW5lJywKICAgICAgICAgICAgICAgICBleHBvc3VyZT0nc3ludGhldGljIGZpeHR1cmUsIG5vdCBhIHJlYWwgZGVtb25zdHJhdGlvbicsCiAgICAgICAgICAgICAgICAgc2RrX3ZlcnNpb25zPXsnYXJjLWFnaSc6ICcwLjkuOScsICdhcmNlbmdpbmUnOiAnMC45LjMnfSwKICAgICAgICAgICAgICAgICBpbml0aWFsX29ic2VydmF0aW9uPWluaXRpYWwpXQogICAgZm9yIHN0ZXAsIGFjdGlvbl9pZCBpbiBlbnVtZXJhdGUoKDMsIDEsIDQpLCBzdGFydD0xKToKICAgICAgICByb3dzLmV4dGVuZChbCiAgICAgICAgICAgIGRpY3QodHlwZT0nYWN0aW9uX2VudGVyZWQnLCBzZXF1ZW5jZT1zdGVwLCB1dGM9ZicyMDI2LTExLTAyVDEyOjAwOntzdGVwOjAyZH0rMDA6MDAnLAogICAgICAgICAgICAgICAgIHJlcXVlc3RfaWQ9ZidmaXh0dXJlLXtzdGVwfScsCiAgICAgICAgICAgICAgICAgYWN0aW9uPXsnYWN0aW9uX2lkJzogYWN0aW9uX2lkLCAnYWN0aW9uX2RhdGEnOiB7fX0sCiAgICAgICAgICAgICAgICAgYmVmb3JlPXJhd19vYnNlcnZhdGlvbihzdGVwIC0gMSkpLAogICAgICAgICAgICBkaWN0KHR5cGU9J2FjdGlvbl9hY2tub3dsZWRnZWQnLCBzZXF1ZW5jZT1zdGVwLCB1dGM9ZicyMDI2LTExLTAyVDEyOjAwOntzdGVwOjAyZH0uMDEwMDAwKzAwOjAwJywKICAgICAgICAgICAgICAgICByZXF1ZXN0X2lkPWYnZml4dHVyZS17c3RlcH0nLCBzZWNvbmRzPTAuMDEsCiAgICAgICAgICAgICAgICAgYWZ0ZXI9cmF3X29ic2VydmF0aW9uKHN0ZXApKSwKICAgICAgICBdKQogICAgcmV0dXJuIHJvd3MKCgpjbGFzcyBGaXh0dXJlKHVuaXR0ZXN0LlRlc3RDYXNlKToKICAgIGRlZiBzZXRVcChzZWxmKToKICAgICAgICBzZWxmLnRlbXBvcmFyeSA9IHRlbXBmaWxlLlRlbXBvcmFyeURpcmVjdG9yeSgpCiAgICAgICAgc2VsZi5hZGRDbGVhbnVwKHNlbGYudGVtcG9yYXJ5LmNsZWFudXApCiAgICAgICAgc2VsZi5wYXRoID0gUGF0aChzZWxmLnRlbXBvcmFyeS5uYW1lKSAvICdzeW50aGV0aWMubWFudWFsLmpzb25sJwoKICAgIGRlZiBjb21waWxlKHNlbGYsIHJvd3M9Tm9uZSwgKiwgZXhwZWN0ZWRfc2hhMjU2PU5vbmUpOgogICAgICAgIHJvd3MgPSBkZW1vX3Jvd3MoKSBpZiByb3dzIGlzIE5vbmUgZWxzZSByb3dzCiAgICAgICAgcGF5bG9hZCA9ICcnLmpvaW4oanNvbi5kdW1wcyhyb3csIHNvcnRfa2V5cz1UcnVlKSArICdcbicgZm9yIHJvdyBpbiByb3dzKS5lbmNvZGUoKQogICAgICAgIHNlbGYucGF0aC53cml0ZV9ieXRlcyhwYXlsb2FkKQogICAgICAgIHNoYSA9IGhhc2hsaWIuc2hhMjU2KHBheWxvYWQpLmhleGRpZ2VzdCgpCiAgICAgICAgcmV0dXJuIGNvbXBpbGVfZGVtbyhzZWxmLnBhdGgsIGV4cGVjdGVkX3NoYTI1NiBvciBzaGEpCgoKY2xhc3MgQ29tcGlsZXIoRml4dHVyZSk6CiAgICBkZWYgdGVzdF90aHJlZV9sZXZlbF9jb3VudGVyX2NvbXBsZXRpb25fYW5kX2V4YWN0X29ic2VydmF0aW9uX2dhdGVzKHNlbGYpOgogICAgICAgIHByb2dyYW0gPSBzZWxmLmNvbXBpbGUoKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocHJvZ3JhbVsnZ2FtZV9pZCddLCBHQU1FKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocHJvZ3JhbVsnc2VlZCddLCAwKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocHJvZ3JhbVsnZ29hbF9sZXZlbHMnXSwgMykKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHByb2dyYW1bJ3NvdXJjZV9kZW1vX3NoYTI1NiddLCBoYXNobGliLnNoYTI1NihzZWxmLnBhdGgucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHByb2dyYW1bJ2luaXRpYWxfb2JzZXJ2YXRpb24nXSwgbmF0aXZlKHJhd19vYnNlcnZhdGlvbigpKSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKFtyb3dbJ2luZGV4J10gZm9yIHJvdyBpbiBwcm9ncmFtWydzdGVwcyddXSwgWzAsIDEsIDJdKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoW3Jvd1snYWN0aW9uJ10gZm9yIHJvdyBpbiBwcm9ncmFtWydzdGVwcyddXSwKICAgICAgICAgICAgICAgICAgICAgICAgIFt7J2FjdGlvbl9pZCc6IG4sICdhY3Rpb25fZGF0YSc6IHt9fSBmb3IgbiBpbiAoMywgMSwgNCldKQogICAgICAgIGZvciBzdGVwLCByb3cgaW4gZW51bWVyYXRlKHByb2dyYW1bJ3N0ZXBzJ10pOgogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHJvd1snYmVmb3JlX3NoYTI1NiddLCBkaWdlc3QobmF0aXZlKHJhd19vYnNlcnZhdGlvbihzdGVwKSkpKQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHJvd1snYWZ0ZXJfc2hhMjU2J10sIGRpZ2VzdChuYXRpdmUocmF3X29ic2VydmF0aW9uKHN0ZXAgKyAxKSkpKQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKChyb3dbJ2JlZm9yZV9sZXZlbHMnXSwgcm93WydhZnRlcl9sZXZlbHMnXSksIChzdGVwLCBzdGVwICsgMSkpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChwcm9ncmFtWydsZXZlbF9ib3VuZGFyaWVzJ10sCiAgICAgICAgICAgICAgICAgICAgICAgICBbeydsZXZlbCc6IG4sICdhZnRlcl9tb3Zlcyc6IG4sICdtb3Zlcyc6IDF9IGZvciBuIGluICgxLCAyLCAzKV0pCgogICAgZGVmIHRlc3Rfc3RvcHNfY29tcGlsaW5nX2F0X2dvYWxfYW5kX2RvZXNfbm90X2ltcG9ydF9sYXRlcl9yZXNldChzZWxmKToKICAgICAgICByb3dzID0gZGVtb19yb3dzKCkKICAgICAgICByb3dzLmV4dGVuZChbCiAgICAgICAgICAgIGRpY3QodHlwZT0nYWN0aW9uX2VudGVyZWQnLCBzZXF1ZW5jZT00LCByZXF1ZXN0X2lkPSdsYXRlci1yZXNldCcsIHV0Yz0nMjAyNi0xMS0wMlQxMjowMDowNCswMDowMCcsCiAgICAgICAgICAgICAgICAgYWN0aW9uPXsnYWN0aW9uX2lkJzogMCwgJ2FjdGlvbl9kYXRhJzoge319LCBiZWZvcmU9cmF3X29ic2VydmF0aW9uKDMpKSwKICAgICAgICAgICAgZGljdCh0eXBlPSdhY3Rpb25fYWNrbm93bGVkZ2VkJywgc2VxdWVuY2U9NCwgcmVxdWVzdF9pZD0nbGF0ZXItcmVzZXQnLCB1dGM9JzIwMjYtMTEtMDJUMTI6MDA6MDQuMDEwMDAwKzAwOjAwJywKICAgICAgICAgICAgICAgICBzZWNvbmRzPTAuMDEsIGFmdGVyPXJhd19vYnNlcnZhdGlvbigpKSwKICAgICAgICBdKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwobGVuKHNlbGYuY29tcGlsZShyb3dzKVsnc3RlcHMnXSksIDMpCgogICAgZGVmIHRlc3Rfc291cmNlX2J5dGVzX211c3RfbWF0Y2hfdHJ1c3RlZF9oYXNoKHNlbGYpOgogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgIHNlbGYuY29tcGlsZShleHBlY3RlZF9zaGEyNTY9J2YnICogNjQpCgogICAgZGVmIHRlc3RfZGVtb19pZGVudGl0eV9hbmRfaW5pdGlhbF9jb3VudGVyX2FyZV9maXhlZChzZWxmKToKICAgICAgICBtdXRhdGlvbnMgPSBbCiAgICAgICAgICAgIGxhbWJkYSByOiByWzBdLnVwZGF0ZShhY3Rvcj0nYWdlbnQnKSwKICAgICAgICAgICAgbGFtYmRhIHI6IHJbMF0udXBkYXRlKGdhbWVfaWQ9J2thNTktb3RoZXInKSwKICAgICAgICAgICAgbGFtYmRhIHI6IHJbMF0udXBkYXRlKHNlZWQ9MSksCiAgICAgICAgICAgIGxhbWJkYSByOiByWzBdLnVwZGF0ZShzZWVkPVRydWUpLAogICAgICAgICAgICBsYW1iZGEgcjogclswXVsnaW5pdGlhbF9vYnNlcnZhdGlvbiddLnVwZGF0ZShsZXZlbHNfY29tcGxldGVkPTEpLAogICAgICAgICAgICBsYW1iZGEgcjogclswXVsnaW5pdGlhbF9vYnNlcnZhdGlvbiddLnVwZGF0ZShnYW1lX2lkPSdsczIwLW90aGVyJyksCiAgICAgICAgXQogICAgICAgIGZvciBtdXRhdGUgaW4gbXV0YXRpb25zOgogICAgICAgICAgICByb3dzID0gZGVtb19yb3dzKCkKICAgICAgICAgICAgbXV0YXRlKHJvd3MpCiAgICAgICAgICAgIHdpdGggc2VsZi5zdWJUZXN0KG11dGF0aW9uPW11dGF0aW9ucy5pbmRleChtdXRhdGUpKSwgc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgICAgICBzZWxmLmNvbXBpbGUocm93cykKCiAgICBkZWYgdGVzdF91bmFja25vd2xlZGdlZF9hY3Rpb25fYW5kX21pc3NpbmdfZ29hbF9yZWZ1c2Uoc2VsZik6CiAgICAgICAgZm9yIHJvd3MgaW4gKGRlbW9fcm93cygpWzotMV0sIGRlbW9fcm93cygpWzotMl0sIGRlbW9fcm93cygpWzoxXSk6CiAgICAgICAgICAgIHdpdGggc2VsZi5zdWJUZXN0KGxlbmd0aD1sZW4ocm93cykpLCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgICAgIHNlbGYuY29tcGlsZShyb3dzKQoKICAgIGRlZiB0ZXN0X3BhaXJpbmdfc2VxdWVuY2VfYW5kX2JlZm9yZV9jaGFpbl9hcmVfaW5kZXBlbmRlbnRseV9jaGVja2VkKHNlbGYpOgogICAgICAgIG11dGF0aW9ucyA9IFsKICAgICAgICAgICAgbGFtYmRhIHI6IHJbMl0udXBkYXRlKHJlcXVlc3RfaWQ9J2RpZmZlcmVudC1yZXF1ZXN0JyksCiAgICAgICAgICAgIGxhbWJkYSByOiByWzJdLnVwZGF0ZShzZXF1ZW5jZT0yKSwKICAgICAgICAgICAgbGFtYmRhIHI6IHJbMV0udXBkYXRlKHNlcXVlbmNlPVRydWUpLAogICAgICAgICAgICBsYW1iZGEgcjogclsyXS51cGRhdGUodHlwZT0nYWN0aW9uX2VudGVyZWQnKSwKICAgICAgICAgICAgbGFtYmRhIHI6IHJbM11bJ2JlZm9yZSddWydmcmFtZXMnXVstMV1bOF0uX19zZXRpdGVtX18oOCwgNyksCiAgICAgICAgXQogICAgICAgIGZvciBtdXRhdGUgaW4gbXV0YXRpb25zOgogICAgICAgICAgICByb3dzID0gZGVtb19yb3dzKCkKICAgICAgICAgICAgbXV0YXRlKHJvd3MpCiAgICAgICAgICAgIHdpdGggc2VsZi5zdWJUZXN0KG11dGF0aW9uPW11dGF0aW9ucy5pbmRleChtdXRhdGUpKSwgc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgICAgICBzZWxmLmNvbXBpbGUocm93cykKCiAgICBkZWYgdGVzdF9yZXNldF9jbGlja19pbGxlZ2FsX2FuZF9ub25pbnRlZ2VyX2FjdGlvbnNfcmVmdXNlKHNlbGYpOgogICAgICAgIGZvciBhY3Rpb24gaW4gKHsnYWN0aW9uX2lkJzogbiwgJ2FjdGlvbl9kYXRhJzoge319IGZvciBuIGluICgwLCA2LCA3LCBUcnVlLCAxLjApKToKICAgICAgICAgICAgcm93cyA9IGRlbW9fcm93cygpCiAgICAgICAgICAgIHJvd3NbMV1bJ2FjdGlvbiddID0gYWN0aW9uCiAgICAgICAgICAgIHdpdGggc2VsZi5zdWJUZXN0KGFjdGlvbj1hY3Rpb24pLCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgICAgIHNlbGYuY29tcGlsZShyb3dzKQogICAgICAgIHJvd3MgPSBkZW1vX3Jvd3MoKQogICAgICAgIHJvd3NbMV1bJ2FjdGlvbiddWydhY3Rpb25fZGF0YSddID0geyd4JzogMn0KICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOgogICAgICAgICAgICBzZWxmLmNvbXBpbGUocm93cykKCiAgICBkZWYgdGVzdF9sZWdhbF9jb250cm9sc19tdXN0X2F1dGhvcml6ZV9yZWNvcmRlZF9hY3Rpb24oc2VsZik6CiAgICAgICAgcm93cyA9IGRlbW9fcm93cygpCiAgICAgICAgcm93c1swXVsnaW5pdGlhbF9vYnNlcnZhdGlvbiddWydhdmFpbGFibGVfYWN0aW9ucyddID0gWzEsIDIsIDRdCiAgICAgICAgcm93c1sxXVsnYmVmb3JlJ10gPSBkZWVwY29weShyb3dzWzBdWydpbml0aWFsX29ic2VydmF0aW9uJ10pCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgc2VsZi5jb21waWxlKHJvd3MpCgogICAgZGVmIHRlc3RfY291bnRlcl9qdW1wX29yX3Rlcm1pbmFsX2JlZm9yZV9nb2FsX2Nhbm5vdF9mYWtlX2NvbXBsZXRpb24oc2VsZik6CiAgICAgICAgZm9yIGZpZWxkLCB2YWx1ZSBpbiAoKCdsZXZlbHNfY29tcGxldGVkJywgMyksICgnbGV2ZWxzX2NvbXBsZXRlZCcsIFRydWUpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICgnc3RhdGUnLCAnV0lOJyksICgnc3RhdGUnLCAnR0FNRV9PVkVSJykpOgogICAgICAgICAgICByb3dzID0gZGVtb19yb3dzKCkKICAgICAgICAgICAgcm93c1syXVsnYWZ0ZXInXVtmaWVsZF0gPSB2YWx1ZQogICAgICAgICAgICByb3dzWzNdWydiZWZvcmUnXSA9IGRlZXBjb3B5KHJvd3NbMl1bJ2FmdGVyJ10pCiAgICAgICAgICAgIHdpdGggc2VsZi5zdWJUZXN0KGZpZWxkPWZpZWxkLCB2YWx1ZT12YWx1ZSksIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOgogICAgICAgICAgICAgICAgc2VsZi5jb21waWxlKHJvd3MpCgogICAgZGVmIHRlc3RfZHVwbGljYXRlX2pzb25fZmllbGRzX2FuZF9ub25maW5pdGVfZHVyYXRpb25fcmVmdXNlKHNlbGYpOgogICAgICAgICMgQXV0aGVudGljYXRlIHRoZSBjaGFuZ2VkIGJ5dGVzIHNvIHJlamVjdGlvbiB0ZXN0cyBwYXJzaW5nLCBub3QgU0hBIG1pc21hdGNoLgogICAgICAgIHNlbGYuY29tcGlsZSgpCiAgICAgICAgdGV4dCA9IHNlbGYucGF0aC5yZWFkX3RleHQoKQogICAgICAgIGZvciBjaGFuZ2VkIGluICh0ZXh0LnJlcGxhY2UoJyJzZWVkIjogMCcsICcic2VlZCI6IDAsICJzZWVkIjogMCcsIDEpLAogICAgICAgICAgICAgICAgICAgICAgICB0ZXh0LnJlcGxhY2UoJyJzZWNvbmRzIjogMC4wMScsICcic2Vjb25kcyI6IE5hTicsIDEpKToKICAgICAgICAgICAgc2VsZi5wYXRoLndyaXRlX2J5dGVzKGNoYW5nZWQuZW5jb2RlKCkpCiAgICAgICAgICAgIHNoYSA9IGhhc2hsaWIuc2hhMjU2KGNoYW5nZWQuZW5jb2RlKCkpLmhleGRpZ2VzdCgpCiAgICAgICAgICAgIHdpdGggc2VsZi5zdWJUZXN0KG5vbmZpbml0ZT0nTmFOJyBpbiBjaGFuZ2VkKSwgc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgICAgICBjb21waWxlX2RlbW8oc2VsZi5wYXRoLCBzaGEpCgoKY2xhc3MgUG9saWN5KEZpeHR1cmUpOgogICAgZGVmIHRlc3RfdGFtcGVyZWRfcHJvZ3JhbV9oYXNoX2NoYWluX2FuZF9jb21wbGV0aW9uX2JvdW5kYXJpZXNfcmVmdXNlKHNlbGYpOgogICAgICAgIG11dGF0aW9ucyA9IFsKICAgICAgICAgICAgbGFtYmRhIHA6IHBbJ3N0ZXBzJ11bMF0udXBkYXRlKGJlZm9yZV9zaGEyNTY9J2EnICogNjQpLAogICAgICAgICAgICBsYW1iZGEgcDogcFsnc3RlcHMnXVsxXS51cGRhdGUoYmVmb3JlX3NoYTI1Nj0nYicgKiA2NCksCiAgICAgICAgICAgIGxhbWJkYSBwOiBwWydzdGVwcyddWzBdLnVwZGF0ZShhZnRlcl9sZXZlbHM9MyksCiAgICAgICAgICAgIGxhbWJkYSBwOiBwWydzdGVwcyddWy0xXS51cGRhdGUoYWZ0ZXJfbGV2ZWxzPTIpLAogICAgICAgICAgICBsYW1iZGEgcDogcFsnbGV2ZWxfYm91bmRhcmllcyddWzBdLnVwZGF0ZShsZXZlbD1UcnVlKSwKICAgICAgICAgICAgbGFtYmRhIHA6IHBbJ2xldmVsX2JvdW5kYXJpZXMnXVsxXS51cGRhdGUoYWZ0ZXJfbW92ZXM9MyksCiAgICAgICAgICAgIGxhbWJkYSBwOiBwLnVwZGF0ZShzZWVkPVRydWUpLAogICAgICAgIF0KICAgICAgICBwcm9ncmFtID0gc2VsZi5jb21waWxlKCkKICAgICAgICBmb3IgbXV0YXRlIGluIG11dGF0aW9uczoKICAgICAgICAgICAgY2hhbmdlZCA9IGRlZXBjb3B5KHByb2dyYW0pCiAgICAgICAgICAgIG11dGF0ZShjaGFuZ2VkKQogICAgICAgICAgICB3aXRoIHNlbGYuc3ViVGVzdChjYXNlPW11dGF0aW9ucy5pbmRleChtdXRhdGUpKSwgc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgICAgICBPdmVyZml0UG9saWN5KGNoYW5nZWQpCgogICAgZGVmIHRlc3Rfc2VsZWN0X2RvZXNfbm90X2FkdmFuY2VfYW5kX2Fja25vd2xlZGdlbWVudF9pc19yZXF1aXJlZChzZWxmKToKICAgICAgICBwb2xpY3kgPSBPdmVyZml0UG9saWN5KHNlbGYuY29tcGlsZSgpKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocG9saWN5LmN1cnNvciwgMCkKICAgICAgICBzZWxmLmFzc2VydEZhbHNlKHBvbGljeS5jb21wbGV0ZWQpCiAgICAgICAgYWN0aW9uID0gcG9saWN5LnNlbGVjdChHQU1FLCBuYXRpdmUocmF3X29ic2VydmF0aW9uKCkpKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoYWN0aW9uLCB7J2FjdGlvbl9pZCc6IDMsICdhY3Rpb25fZGF0YSc6IHt9fSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHBvbGljeS5jdXJzb3IsIDApCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgcG9saWN5LnNlbGVjdChHQU1FLCBuYXRpdmUocmF3X29ic2VydmF0aW9uKCkpKQogICAgICAgICMgQSBkdXBsaWNhdGUgc2VsZWN0IGlzIHJlZnVzZWQgd2l0aG91dCBlcmFzaW5nIHRoZSB2YWxpZCBwZW5kaW5nIGFjdGlvbi4KICAgICAgICBwb2xpY3kuYWNrbm93bGVkZ2UoYWN0aW9uLCBuYXRpdmUocmF3X29ic2VydmF0aW9uKDEpKSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHBvbGljeS5jdXJzb3IsIDEpCgogICAgZGVmIHRlc3Rfc3VjY2Vzc19yZXF1aXJlc190aHJlZV92ZXJpZmllZF9jb3VudGVyX2luY3JlbWVudHMoc2VsZik6CiAgICAgICAgcG9saWN5ID0gT3ZlcmZpdFBvbGljeShzZWxmLmNvbXBpbGUoKSkKICAgICAgICBmb3Igc3RlcCwgZXhwZWN0ZWQgaW4gZW51bWVyYXRlKCgzLCAxLCA0KSk6CiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RmFsc2UocG9saWN5LmNvbXBsZXRlZCkKICAgICAgICAgICAgYWN0aW9uID0gcG9saWN5LnNlbGVjdChHQU1FLCBuYXRpdmUocmF3X29ic2VydmF0aW9uKHN0ZXApKSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChhY3Rpb25bJ2FjdGlvbl9pZCddLCBleHBlY3RlZCkKICAgICAgICAgICAgcG9saWN5LmFja25vd2xlZGdlKGFjdGlvbiwgbmF0aXZlKHJhd19vYnNlcnZhdGlvbihzdGVwICsgMSkpKQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShwb2xpY3kuY29tcGxldGVkKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocG9saWN5LmN1cnNvciwgMykKICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFN0b3BJdGVyYXRpb24pOgogICAgICAgICAgICBwb2xpY3kuc2VsZWN0KEdBTUUsIG5hdGl2ZShyYXdfb2JzZXJ2YXRpb24oMykpKQoKICAgIGRlZiB0ZXN0X3dyb25nX2dhbWVfb3JfYW55X29ic2VydmF0aW9uX2RyaWZ0X3Blcm1hbmVudGx5X3F1YXJhbnRpbmVzKHNlbGYpOgogICAgICAgIGNoYW5nZXMgPSBbCiAgICAgICAgICAgIGxhbWJkYSBvOiBvWydncmlkJ11bNF0uX19zZXRpdGVtX18oNSwgOCksCiAgICAgICAgICAgIGxhbWJkYSBvOiBvWydncmlkJ11bNjNdLl9fc2V0aXRlbV9fKDYzLCA3KSwKICAgICAgICAgICAgbGFtYmRhIG86IG8udXBkYXRlKGxlZ2FsX2FjdGlvbnM9WzEsIDIsIDRdKSwKICAgICAgICAgICAgbGFtYmRhIG86IG8udXBkYXRlKGxldmVsc19jb21wbGV0ZWQ9MSksCiAgICAgICAgICAgIGxhbWJkYSBvOiBvLnVwZGF0ZShsZXZlbHNfY29tcGxldGVkPUZhbHNlKSwKICAgICAgICAgICAgbGFtYmRhIG86IG8udXBkYXRlKHN0YXRlPSdHQU1FX09WRVInKSwKICAgICAgICBdCiAgICAgICAgY2FzZXMgPSBbKEdBTUUsIG11dGF0ZSkgZm9yIG11dGF0ZSBpbiBjaGFuZ2VzXSArIFsoJ2xzMjAtb3RoZXInLCBsYW1iZGEgbzogTm9uZSldCiAgICAgICAgZm9yIGdhbWUsIG11dGF0ZSBpbiBjYXNlczoKICAgICAgICAgICAgcG9saWN5ID0gT3ZlcmZpdFBvbGljeShzZWxmLmNvbXBpbGUoKSkKICAgICAgICAgICAgb2JzID0gbmF0aXZlKHJhd19vYnNlcnZhdGlvbigpKQogICAgICAgICAgICBtdXRhdGUob2JzKQogICAgICAgICAgICB3aXRoIHNlbGYuc3ViVGVzdChnYW1lPWdhbWUsIGNhc2U9Y2FzZXMuaW5kZXgoKGdhbWUsIG11dGF0ZSkpKToKICAgICAgICAgICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgICAgICAgICAgcG9saWN5LnNlbGVjdChnYW1lLCBvYnMpCiAgICAgICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHBvbGljeS5jdXJzb3IsIDApCiAgICAgICAgICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOgogICAgICAgICAgICAgICAgICAgIHBvbGljeS5zZWxlY3QoR0FNRSwgbmF0aXZlKHJhd19vYnNlcnZhdGlvbigpKSkKCiAgICBkZWYgdGVzdF93cm9uZ19hY2tfYWN0aW9uX3BpeGVsX29yX2NvdW50ZXJfbmV2ZXJfYWR2YW5jZXNfYW5kX3F1YXJhbnRpbmVzKHNlbGYpOgogICAgICAgIGNoYW5nZXMgPSBbCiAgICAgICAgICAgIGxhbWJkYSBhLCBvOiBhLnVwZGF0ZShhY3Rpb25faWQ9NCksCiAgICAgICAgICAgIGxhbWJkYSBhLCBvOiBhLnVwZGF0ZShhY3Rpb25faWQ9VHJ1ZSksCiAgICAgICAgICAgIGxhbWJkYSBhLCBvOiBhLnVwZGF0ZShhY3Rpb25fZGF0YT17J3gnOiAxfSksCiAgICAgICAgICAgIGxhbWJkYSBhLCBvOiBvWydncmlkJ11bNF0uX19zZXRpdGVtX18oNSwgOCksCiAgICAgICAgICAgIGxhbWJkYSBhLCBvOiBvLnVwZGF0ZShsZXZlbHNfY29tcGxldGVkPTIpLAogICAgICAgICAgICBsYW1iZGEgYSwgbzogby51cGRhdGUobGV2ZWxzX2NvbXBsZXRlZD1UcnVlKSwKICAgICAgICAgICAgbGFtYmRhIGEsIG86IG8udXBkYXRlKGxlZ2FsX2FjdGlvbnM9WzEsIDIsIDNdKSwKICAgICAgICBdCiAgICAgICAgZm9yIG11dGF0ZSBpbiBjaGFuZ2VzOgogICAgICAgICAgICBwb2xpY3kgPSBPdmVyZml0UG9saWN5KHNlbGYuY29tcGlsZSgpKQogICAgICAgICAgICBhY3Rpb24gPSBwb2xpY3kuc2VsZWN0KEdBTUUsIG5hdGl2ZShyYXdfb2JzZXJ2YXRpb24oKSkpCiAgICAgICAgICAgIGFmdGVyID0gbmF0aXZlKHJhd19vYnNlcnZhdGlvbigxKSkKICAgICAgICAgICAgbXV0YXRlKGFjdGlvbiwgYWZ0ZXIpCiAgICAgICAgICAgIHdpdGggc2VsZi5zdWJUZXN0KGNhc2U9Y2hhbmdlcy5pbmRleChtdXRhdGUpKSwgc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgICAgICBwb2xpY3kuYWNrbm93bGVkZ2UoYWN0aW9uLCBhZnRlcikKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChwb2xpY3kuY3Vyc29yLCAwKQogICAgICAgICAgICBzZWxmLmFzc2VydEZhbHNlKHBvbGljeS5jb21wbGV0ZWQpCiAgICAgICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgICAgICBwb2xpY3kuc2VsZWN0KEdBTUUsIG5hdGl2ZShyYXdfb2JzZXJ2YXRpb24oKSkpCgogICAgZGVmIHRlc3RfdW5yZXF1ZXN0ZWRfYW5kX2R1cGxpY2F0ZV9hY2tub3dsZWRnZW1lbnRzX3JlZnVzZShzZWxmKToKICAgICAgICBhY3Rpb24gPSB7J2FjdGlvbl9pZCc6IDMsICdhY3Rpb25fZGF0YSc6IHt9fQogICAgICAgIHBvbGljeSA9IE92ZXJmaXRQb2xpY3koc2VsZi5jb21waWxlKCkpCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgcG9saWN5LmFja25vd2xlZGdlKGFjdGlvbiwgbmF0aXZlKHJhd19vYnNlcnZhdGlvbigxKSkpCiAgICAgICAgcG9saWN5ID0gT3ZlcmZpdFBvbGljeShzZWxmLmNvbXBpbGUoKSkKICAgICAgICBwb2xpY3kuYWNrbm93bGVkZ2UocG9saWN5LnNlbGVjdChHQU1FLCBuYXRpdmUocmF3X29ic2VydmF0aW9uKCkpKSwgbmF0aXZlKHJhd19vYnNlcnZhdGlvbigxKSkpCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgcG9saWN5LmFja25vd2xlZGdlKGFjdGlvbiwgbmF0aXZlKHJhd19vYnNlcnZhdGlvbigxKSkpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChwb2xpY3kuY3Vyc29yLCAxKQoKICAgIGRlZiB0ZXN0X3JldHVybmVkX2FjdGlvbl9hbmRfY29uc3RydWN0b3JfaW5wdXRfZG9fbm90X211dGF0ZV9vd25lZF9wcm9ncmFtKHNlbGYpOgogICAgICAgIHByb2dyYW0gPSBzZWxmLmNvbXBpbGUoKQogICAgICAgIHBvbGljeSA9IE92ZXJmaXRQb2xpY3kocHJvZ3JhbSkKICAgICAgICBwcm9ncmFtWydzdGVwcyddWzBdWydhY3Rpb24nXVsnYWN0aW9uX2lkJ10gPSA0CiAgICAgICAgYWN0aW9uID0gcG9saWN5LnNlbGVjdChHQU1FLCBuYXRpdmUocmF3X29ic2VydmF0aW9uKCkpKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoYWN0aW9uWydhY3Rpb25faWQnXSwgMykKICAgICAgICBhY3Rpb25bJ2FjdGlvbl9pZCddID0gMgogICAgICAgIHBvbGljeS5hY2tub3dsZWRnZSh7J2FjdGlvbl9pZCc6IDMsICdhY3Rpb25fZGF0YSc6IHt9fSwgbmF0aXZlKHJhd19vYnNlcnZhdGlvbigxKSkpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChwb2xpY3kuY3Vyc29yLCAxKQoKCmlmIF9fbmFtZV9fID09ICdfX21haW5fXyc6CiAgICB1bml0dGVzdC5tYWluKCkK'}
    pins = {'agent/__init__.py': 'b3eb0af7a0a461dde9fcc9cadf4987ccf282db7936d87fa49114d3fe2488eab0', 'agent/action.py': 'f39c95a7b096830416bd1ea85d1d26ed7e8d52328291894863f66bfea4a4c23a', 'agent/action_journal.py': '720203276ae1a452e74b5f2f61ff3a29b1ac3eeec85ecd5f35290d148fa30073', 'agent/competition_loop.py': '08bef472a29b01a0698cb8469683291c006fff6639cff04735e4857609f42284', 'agent/controller.py': '1382b598c3c633223dc901af560b0161837dea60c2293816f1edc535eea042b0', 'agent/diagnostics.py': '14bd8686d57c9ba4091d93ceef3ba964238aa79576b123c0baadbf860c449d85', 'agent/evidence.py': '01d98b290240341597b55d9f44d93b30e9d0f1b139d7e6759fb6d2ade9d054fd', 'agent/framework_adapter.py': '3ac7848aecb2cadcab80f08d762a55114093151cee9d8c8e3ab824c6372218ef', 'agent/state.py': 'c038905633455f77fb99582cac092809c29e2ee5de06d3666b53bd3c9bc754d7', 'agent/watchdog.py': '180fa253d50e1c5848bf26bfe9d15514ebc748b9b7ed072da81ec178e8b47f31', 'certification/direct_publisher_smoke_v1/__init__.py': '08c32bf5bc6167ad1202120f90a00043271310c34896d33188c78e2049eb2665', 'certification/direct_publisher_smoke_v1/install.py': '568ef619fce6fc8ed03eda07e7a240f079e017379e1494f4a6b95672cad0019b', 'certification/direct_publisher_smoke_v1/preflight.py': '369a39a09c4f5f97b0d92613c3ecc6aa281779af4721a4154fe98576e4fc3b0d', 'certification/direct_publisher_smoke_v1/server.py': 'e3cbdf0949f21fc2edb6f8c781a55aaeacf3410aa4f5794e3ebe420f2bcee888', 'certification/phase4_multimodal_preflight_v3/__init__.py': 'ba9d210e8c6b4c0475ff24e6ff948dcc1d82f0002a2907450eb239362991ebfb', 'certification/phase4_multimodal_preflight_v3/images.py': '7121f88fc1b1ffec953c769c43614f4cf311ad725a1a754f97545ccc49099f27', 'research/cpu_competition_v1/__init__.py': 'e8fb2bf90c2de9ffc4b2bdceb727318910b788749444f6845537a17ed464b50a', 'research/cpu_competition_v1/requirements.lock': '9cbd701f16f3e3441692d9328f57b1f3c15c168a7e227431223cf9a3ddb1eb7d', 'research/cpu_competition_v1/runtime.py': '05e6339c846d43d4ed4542cce0cf88b2057b22ebc33ab11c4ac45b788a4e9c75', 'research/cpu_competition_v1/supervisor.py': '64304b29296b08a8230eca48c6903ccdb2899cecd51747a5317cdab83d76243d', 'research/cpu_competition_v1/wheels.json': '2479b23c71acae91ca5a99ae46e90ef3c31e0723f24ee505f9c6da5c27d2dbfe', 'research/initial_geometry_assistance_v1/__init__.py': 'ca54f4f9af2fefe1e6859bc1215a9dd15f5e542f43b2dea1aafc37e04aedc737', 'research/initial_geometry_assistance_v1/geometry.py': 'c86bcad5806f4c2fb5230922d6dfc97ba93aadd7de88a581da78122c340c0a1f', 'research/initial_geometry_assistance_v1/live_io.py': '6eb99fd31f0ce53544b7c217e271d1fb56eecb73684e27175d59fe76952ed855', 'research/initial_geometry_assistance_v1/live_resources.py': '0c527da213913ebe00c6348e4120c556899bf810414f0f854e49cf2acaf7dea7', 'research/initial_geometry_assistance_v1/policy.py': '717cb1d9358713ce66bc8f6c4a6da448df7c7da5d53408539f29aa88498e0b4a', 'research/ls20_overfit_v1/__init__.py': '1df46a915569d6f7dcd061ca904671ce44febe8c2d082824d9475b8740393b80', 'research/ls20_overfit_v1/policy.py': '070e673aa0db373813eb401c6a960c2411553e724b044d97714560076c7a67fa', 'research/ls20_overfit_v1/policy_seed0.json': 'd70c8ff61040d49405f95ef5d56e17f1634ac51695c97e01b2f7f5a92c35b2cd', 'tests/__init__.py': '08b8e5387013e869a19974637957373870abb44c916683d143d1209df831dd61', 'tests/test_cpu_competition_packaging_v1.py': '9db075f48be9cfff496bbaf57ea5019fdf0fb5f0144fc4f2667af9f623ba7af8', 'tests/test_cpu_competition_v1.py': '699dbe09de7e666b9fae88b2e21d5c9dbfe7c4f5c0bb7cce80483f0baf11513e', 'tests/test_ls20_overfit_v1.py': '2c7dbcd4a8b9e92a5c5d1d25e9d4443f0dad117abb9de5795c2728948305e14b'}
    for name, text in payload.items():
        data = base64.b64decode(text, validate=True)
        if hashlib.sha256(data).hexdigest() != pins[name]:
            raise ValueError('embedded CPU source changed')
        target = source/name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
    sys.path.insert(0,str(source))
    from research.cpu_competition_v1.supervisor import entry
    candidates = [pathlib.Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'),
                  pathlib.Path('/kaggle/input/arc-prize-2026-arc-agi-3/arc_agi_3_wheels')]
    available = [p for p in candidates if p.is_dir()]
    if len(available) != 1:
        raise RuntimeError('exact competition CPU wheel mount missing or ambiguous')
    def finish(completed):
        try:
            if completed:
                if scored:
                    if not submission.is_file() or submission.stat().st_size == 0:
                        raise RuntimeError('gateway did not create submission.parquet')
                else:
                    import pandas as pd
                    pd.DataFrame([['1_0','1',True,1]],columns=['row_id','game_id','end_of_game','score']).to_parquet(submission,index=False)
                    print('Validation only: no game calls; placeholder is not a leaderboard score.',flush=True)
        finally:
            shutil.rmtree(source)
    entry(source,available[0],scored=scored,finalize=finish)
finally:
    if source.exists():
        shutil.rmtree(source)
